# ANSC 4040 — Clean XGBRanker Pipeline

This notebook keeps only the code needed for the current **trajectory-aware candidate generation + XGBoost learning-to-rank** approach.

Pipeline:

1. Load cleaned modelling data.
2. Keep the outer chronological 60/20/20 split.
3. Split the outer training period into history / reranker-train / reranker-dev.
4. Build structural candidate profiles and sampled historical reference rows.
5. Build candidate-level KNN / trajectory / prototype features.
6. Add within-query rank features.
7. Train `XGBRanker` using query groups (`qid`).
8. Evaluate on the same 500-query development sample.

Removed from the old notebook: Extra Trees evidence cells, old candidate hyperparameter searches, HGB experiments/tuning, duplicate KNN production code, and old Stage 1/Stage 2 hybrid-score searches.

**Current fixed candidate settings:** start tolerance = 21 days, max candidates = 120, DIM tolerance = 30 days, DIM bin width = 7 days.

In [2]:
# ============================================================
# 1. Imports and configuration
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler

import xgboost as xgb
from xgboost import XGBRanker

RANDOM_STATE = 42

MODEL_DATA_PATH = Path(
    r"/Library/ansci-4040-fall-2026/jl4937_6040-project-1/dataset/Dataset_for_model_buliding_2.csv"
)

MODEL_COLUMNS = [
    "AnimalId",
    "LactationNumber",
    "DaysInMilk",
    "ReproductionStatus",
    "EventDate",
    "Avgmilkflow",
    "Flow30_60Session",
    "YieldFirst2Min_Session",
    "YieldSession",
    "DurationSession_sec",
    "milking",
]

# Fixed settings selected in the earlier experiments
DIM_BIN_WIDTH = 7

RERANKER_CANDIDATE_PARAMS = {
    "start_tolerance_days": 21,
    "max_candidates": 120,
    "projected_rank_penalty_days": 7,
    "dim_tolerance_days": 30,
}

REFERENCE_ROWS_PER_GROUP = 250
TRAIN_QUERY_ROWS = 2000
DEV_QUERY_ROWS = 500
DEV_RANDOM_STATE = 123

# Optional cache: avoids rebuilding candidate tables after a kernel restart.
USE_CANDIDATE_CACHE = True
CACHE_DIR = Path("xgboost_cache")
CACHE_DIR.mkdir(exist_ok=True)

TRAIN_CACHE_PATH = CACHE_DIR / "train_candidates_2000_tol21_max120_dim30_bin7.pkl"
DEV_CACHE_PATH = CACHE_DIR / "dev_candidates_500_tol21_max120_dim30_bin7.pkl"

print("XGBoost version:", xgb.__version__)
print("Candidate parameters:", RERANKER_CANDIDATE_PARAMS)
print("DIM_BIN_WIDTH:", DIM_BIN_WIDTH)

XGBoost version: 3.4.1
Candidate parameters: {'start_tolerance_days': 21, 'max_candidates': 120, 'projected_rank_penalty_days': 7, 'dim_tolerance_days': 30}
DIM_BIN_WIDTH: 7


In [3]:
# ============================================================
# 2. Load data and create chronological splits
# ============================================================

if not MODEL_DATA_PATH.exists():
    raise FileNotFoundError(f"Model data file not found: {MODEL_DATA_PATH}")

data = pd.read_csv(MODEL_DATA_PATH, low_memory=False)
data.columns = data.columns.astype(str).str.strip()

missing_columns = [c for c in MODEL_COLUMNS if c not in data.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

data = data[MODEL_COLUMNS].copy()

data["AnimalId"] = data["AnimalId"].astype("string")
data["EventDate"] = pd.to_datetime(data["EventDate"], errors="coerce")

for col in ["LactationNumber", "DaysInMilk", "DurationSession_sec"]:
    data[col] = pd.to_numeric(data[col], errors="coerce", downcast="integer")

for col in [
    "Avgmilkflow",
    "Flow30_60Session",
    "YieldFirst2Min_Session",
    "YieldSession",
]:
    data[col] = pd.to_numeric(data[col], errors="coerce", downcast="float")

data["ReproductionStatus"] = data["ReproductionStatus"].astype("category")

# ---------- Outer chronological 60/20/20 split ----------
labelled_data = (
    data.loc[data["AnimalId"].notna() & data["EventDate"].notna()]
    .sort_values("EventDate")
    .reset_index(drop=True)
)

n = len(labelled_data)
train_cut_date = labelled_data.iloc[int(n * 0.60)]["EventDate"]
validation_cut_date = labelled_data.iloc[int(n * 0.80)]["EventDate"]

data_train = labelled_data.loc[
    labelled_data["EventDate"] < train_cut_date
].copy()

data_validation = labelled_data.loc[
    (labelled_data["EventDate"] >= train_cut_date)
    & (labelled_data["EventDate"] < validation_cut_date)
].copy()

data_test = labelled_data.loc[
    labelled_data["EventDate"] >= validation_cut_date
].copy()

print(f"Outer train:      {len(data_train):,}")
print(f"Outer validation: {len(data_validation):,}")
print(f"Outer test:       {len(data_test):,}")

# ---------- Internal split inside OUTER TRAIN only ----------
reranker_base = data_train.sort_values("EventDate").reset_index(drop=True)
n_reranker = len(reranker_base)

history_cut_date = reranker_base.iloc[int(n_reranker * 0.70)]["EventDate"]
dev_cut_date = reranker_base.iloc[int(n_reranker * 0.85)]["EventDate"]

reranker_history = reranker_base.loc[
    reranker_base["EventDate"] < history_cut_date
].copy()

reranker_train_queries = reranker_base.loc[
    (reranker_base["EventDate"] >= history_cut_date)
    & (reranker_base["EventDate"] < dev_cut_date)
].copy()

reranker_dev_queries = reranker_base.loc[
    reranker_base["EventDate"] >= dev_cut_date
].copy()

print("\nInternal reranker split:")
print(
    "History:",
    f"{len(reranker_history):,}",
    reranker_history["EventDate"].min(),
    "to",
    reranker_history["EventDate"].max(),
)
print(
    "Train queries:",
    f"{len(reranker_train_queries):,}",
    reranker_train_queries["EventDate"].min(),
    "to",
    reranker_train_queries["EventDate"].max(),
)
print(
    "Dev queries:",
    f"{len(reranker_dev_queries):,}",
    reranker_dev_queries["EventDate"].min(),
    "to",
    reranker_dev_queries["EventDate"].max(),
)

Outer train:      3,985,088
Outer validation: 1,327,822
Outer test:       1,332,262

Internal reranker split:
History: 2,787,167 2019-06-14 00:00:00 to 2020-07-05 00:00:00
Train queries: 592,716 2020-07-06 00:00:00 to 2020-09-20 00:00:00
Dev queries: 605,205 2020-09-21 00:00:00 to 2020-12-13 00:00:00


In [4]:
# ============================================================
# 3. Helper functions
# ============================================================

BASE_NUMERIC_FEATURES = [
    "DaysInMilk",
    "Avgmilkflow",
    "Flow30_60Session",
    "YieldFirst2Min_Session",
    "YieldSession",
    "DurationSession_sec",
]


def event_day_from_origin(frame, origin_date):
    return (
        pd.to_datetime(frame["EventDate"], errors="coerce") - origin_date
    ).dt.days.astype("float32")


def lactation_start_day(frame, origin_date):
    event_date = pd.to_datetime(frame["EventDate"], errors="coerce")
    lactation_start = event_date - pd.to_timedelta(frame["DaysInMilk"], unit="D")
    return (lactation_start - origin_date).dt.days.astype("float32")


def build_structural_profiles(
    labelled_frame,
    class_to_number,
    origin_date,
    include_projection=True,
):
    temp = labelled_frame[
        ["AnimalId", "LactationNumber", "EventDate", "DaysInMilk"]
    ].copy()

    temp["EventDay"] = event_day_from_origin(temp, origin_date)
    temp["LactationStartDay"] = lactation_start_day(temp, origin_date)

    temp = temp.dropna(
        subset=[
            "AnimalId",
            "LactationNumber",
            "DaysInMilk",
            "EventDay",
            "LactationStartDay",
        ]
    )

    observed = (
        temp.groupby(["AnimalId", "LactationNumber"], observed=True)
        .agg(
            LactationStartDay=("LactationStartDay", "median"),
            LastEventDay=("EventDay", "max"),
            LastDaysInMilk=("DaysInMilk", "max"),
        )
        .reset_index()
    )

    observed["ClassId"] = (
        observed["AnimalId"].astype(object).map(class_to_number)
    )
    observed = observed.loc[observed["ClassId"].notna()].copy()

    observed["ClassId"] = observed["ClassId"].astype(np.int32)
    observed["LactationNumber"] = (
        observed["LactationNumber"].round().astype(np.int16)
    )

    for col in ["LactationStartDay", "LastEventDay", "LastDaysInMilk"]:
        observed[col] = observed[col].round().astype(np.int32)

    observed["IsProjected"] = False

    interval_rows = []
    for class_id, group in observed.groupby("ClassId", sort=False):
        group = group.sort_values("LactationNumber")
        lactations = group["LactationNumber"].to_numpy()
        starts = group["LactationStartDay"].to_numpy()

        for i in range(1, len(group)):
            if lactations[i] - lactations[i - 1] == 1:
                interval = starts[i] - starts[i - 1]
                if 250 <= interval <= 550:
                    interval_rows.append(
                        {"ClassId": int(class_id), "IntervalDays": int(interval)}
                    )

    if interval_rows:
        interval_df = pd.DataFrame(interval_rows)
        global_interval = int(round(interval_df["IntervalDays"].median()))
        animal_interval = (
            interval_df.groupby("ClassId")["IntervalDays"].median().to_dict()
        )
    else:
        global_interval = 365
        animal_interval = {}

    if not include_projection:
        return observed, global_interval

    projected_rows = []

    for class_id, group in observed.groupby("ClassId", sort=False):
        last = group.sort_values("LactationNumber").iloc[-1]
        last_lactation = int(last["LactationNumber"])

        if last_lactation >= 10:
            continue

        interval = int(
            round(animal_interval.get(int(class_id), global_interval))
        )

        projected_rows.append(
            {
                "AnimalId": last["AnimalId"],
                "LactationNumber": last_lactation + 1,
                "LactationStartDay": int(last["LactationStartDay"] + interval),
                "LastEventDay": int(last["LastEventDay"]),
                "LastDaysInMilk": int(last["LastDaysInMilk"]),
                "ClassId": int(class_id),
                "IsProjected": True,
            }
        )

    if projected_rows:
        profiles = pd.concat(
            [observed, pd.DataFrame(projected_rows)],
            ignore_index=True,
        )
    else:
        profiles = observed.copy()

    return profiles, global_interval


def make_profile_lookup(profiles):
    lookup = {}

    for lactation, group in profiles.groupby("LactationNumber", sort=False):
        lookup[int(lactation)] = {
            "class_ids": group["ClassId"].to_numpy(dtype=np.int32),
            "start_days": group["LactationStartDay"].to_numpy(dtype=np.float32),
            "last_event_days": group["LastEventDay"].to_numpy(dtype=np.float32),
            "last_dim": group["LastDaysInMilk"].to_numpy(dtype=np.float32),
            "projected": group["IsProjected"].to_numpy(dtype=bool),
        }

    return lookup


def get_candidate_profiles(
    profile_lookup,
    lactation_number,
    query_start_day,
    start_tolerance_days,
    max_candidates,
    projected_rank_penalty_days,
):
    if pd.isna(lactation_number) or pd.isna(query_start_day):
        return None

    lactation_number = int(round(float(lactation_number)))
    profile = profile_lookup.get(lactation_number)

    if profile is None:
        return None

    start_difference = np.abs(
        profile["start_days"] - float(query_start_day)
    )

    rank_difference = (
        start_difference
        + profile["projected"].astype(np.float32)
        * projected_rank_penalty_days
    )

    inside = start_difference <= start_tolerance_days

    if inside.any():
        positions = np.flatnonzero(inside)
    else:
        positions = np.arange(len(profile["class_ids"]))

    positions = positions[
        np.argsort(rank_difference[positions])
    ][:max_candidates]

    return {
        key: value[positions]
        for key, value in profile.items()
    } | {
        "start_difference": start_difference[positions]
    }


def create_distance_features(raw_frame):
    X = raw_frame[
        BASE_NUMERIC_FEATURES + ["ReproductionStatus"]
    ].copy()

    duration_min = (
        X["DurationSession_sec"].astype(np.float32) / 60.0
    )

    X["YieldPerMinute"] = (
        X["YieldSession"] / duration_min.replace(0, np.nan)
    )
    X["First2MinFraction"] = (
        X["YieldFirst2Min_Session"]
        / X["YieldSession"].replace(0, np.nan)
    )
    X["FlowShapeRatio"] = (
        X["Flow30_60Session"]
        / X["Avgmilkflow"].replace(0, np.nan)
    )

    return X


def fit_distance_preprocessor(raw_frame):
    features = create_distance_features(raw_frame)

    numeric_columns = [
        c for c in features.columns if c != "ReproductionStatus"
    ]

    numeric = features[numeric_columns].copy()
    numeric_fill = numeric.median(numeric_only=True)
    numeric = numeric.fillna(numeric_fill)

    scaler = StandardScaler()
    numeric_scaled = scaler.fit_transform(numeric).astype(np.float32)

    reproduction = pd.get_dummies(
        features["ReproductionStatus"],
        prefix="ReproductionStatus",
        dtype=np.float32,
    )

    reproduction_columns = list(reproduction.columns)

    X = np.column_stack(
        [
            numeric_scaled,
            reproduction.to_numpy(dtype=np.float32),
        ]
    ).astype(np.float32)

    feature_names = numeric_columns + reproduction_columns

    return (
        X,
        numeric_columns,
        numeric_fill,
        scaler,
        reproduction_columns,
        feature_names,
    )


def transform_distance_features(
    raw_frame,
    numeric_columns,
    numeric_fill,
    scaler,
    reproduction_columns,
):
    features = create_distance_features(raw_frame)

    numeric = (
        features[numeric_columns]
        .copy()
        .fillna(numeric_fill)
    )

    numeric_scaled = scaler.transform(numeric).astype(np.float32)

    reproduction = (
        pd.get_dummies(
            features["ReproductionStatus"],
            prefix="ReproductionStatus",
            dtype=np.float32,
        )
        .reindex(columns=reproduction_columns, fill_value=0)
    )

    return np.column_stack(
        [
            numeric_scaled,
            reproduction.to_numpy(dtype=np.float32),
        ]
    ).astype(np.float32)


def sample_reference_rows(
    frame,
    max_rows_per_group,
    recent_fraction=0.80,
    random_state=42,
):
    selected_indices = []

    grouped = frame.groupby(
        ["AnimalId", "LactationNumber"],
        observed=True,
        sort=False,
    )

    for _, group in grouped:
        group = group.sort_values("EventDate")

        if len(group) <= max_rows_per_group:
            selected_indices.extend(group.index.tolist())
            continue

        n_recent = max(
            1,
            int(round(max_rows_per_group * recent_fraction)),
        )

        recent = group.tail(n_recent)
        remaining_n = max_rows_per_group - len(recent)

        if remaining_n > 0:
            historical_pool = group.loc[
                ~group.index.isin(recent.index)
            ]

            historical = historical_pool.sample(
                n=min(remaining_n, len(historical_pool)),
                replace=False,
                random_state=random_state,
            )

            selected = pd.concat([recent, historical])
        else:
            selected = recent

        selected_indices.extend(selected.index.tolist())

    return pd.Index(selected_indices)


def build_reference_index(class_ids, lactations):
    exact = {}
    class_lactations = {}

    for row_position, (class_id, lactation) in enumerate(
        zip(class_ids, lactations)
    ):
        class_id = int(class_id)
        lactation = int(lactation)

        exact.setdefault((class_id, lactation), []).append(row_position)
        class_lactations.setdefault(class_id, set()).add(lactation)

    exact = {
        key: np.asarray(positions, dtype=np.int32)
        for key, positions in exact.items()
    }

    class_lactations = {
        class_id: np.asarray(sorted(values), dtype=np.int16)
        for class_id, values in class_lactations.items()
    }

    return exact, class_lactations


def choose_reference_lactation(
    class_id,
    query_lactation,
    class_lactations,
):
    available = class_lactations.get(int(class_id))

    if available is None or len(available) == 0:
        return None, 99

    query_lactation = int(round(float(query_lactation)))

    if query_lactation in available:
        return query_lactation, 0

    previous = available[available <= query_lactation]

    if len(previous) > 0:
        selected = int(previous.max())
    else:
        selected = int(
            available[
                np.argmin(np.abs(available - query_lactation))
            ]
        )

    return selected, abs(query_lactation - selected)


print("Helper functions loaded.")

Helper functions loaded.


In [5]:
# ============================================================
# 4. Build history-only candidate/reference system
# ============================================================

reranker_class_values = pd.Index(
    reranker_history["AnimalId"].astype(object).unique()
)

reranker_class_to_number = {
    animal_id: class_number
    for class_number, animal_id in enumerate(reranker_class_values)
}

reranker_number_to_class = {
    class_number: animal_id
    for class_number, animal_id in enumerate(reranker_class_values)
}

reranker_origin_date = reranker_history["EventDate"].min()

(
    reranker_profiles,
    reranker_global_lactation_interval,
) = build_structural_profiles(
    reranker_history,
    reranker_class_to_number,
    reranker_origin_date,
    include_projection=True,
)

reranker_profile_lookup = make_profile_lookup(reranker_profiles)

reranker_reference_indices = sample_reference_rows(
    reranker_history,
    max_rows_per_group=REFERENCE_ROWS_PER_GROUP,
    recent_fraction=0.80,
    random_state=RANDOM_STATE,
)

reranker_reference_raw = (
    reranker_history.loc[reranker_reference_indices].copy()
)

(
    X_reranker_reference,
    reranker_numeric_columns,
    reranker_numeric_fill,
    reranker_scaler,
    reranker_reproduction_columns,
    reranker_feature_names,
) = fit_distance_preprocessor(reranker_reference_raw)

reranker_reference_class_ids = (
    reranker_reference_raw["AnimalId"]
    .astype(object)
    .map(reranker_class_to_number)
    .astype(np.int32)
    .to_numpy()
)

reranker_reference_lactations = (
    reranker_reference_raw["LactationNumber"]
    .round()
    .astype(np.int16)
    .to_numpy()
)

reranker_reference_dim = (
    reranker_reference_raw["DaysInMilk"]
    .astype(np.float32)
    .to_numpy()
)

reranker_reference_event_day = (
    event_day_from_origin(
        reranker_reference_raw,
        reranker_origin_date,
    ).to_numpy()
)

(
    reranker_reference_exact_index,
    reranker_reference_class_lactations,
) = build_reference_index(
    reranker_reference_class_ids,
    reranker_reference_lactations,
)

print("History AnimalId classes:", f"{len(reranker_class_values):,}")
print("Structural profiles:", f"{len(reranker_profiles):,}")
print("Reference rows:", f"{len(reranker_reference_raw):,}")
print("Reference matrix:", X_reranker_reference.shape)
print("Distance features:", len(reranker_feature_names))

History AnimalId classes: 6,667
Structural profiles: 16,396
Reference rows: 1,783,366
Reference matrix: (1783366, 13)
Distance features: 13


In [6]:
# ============================================================
# 5. Build cow and lactation/DIM prototypes
# ============================================================

reranker_cow_prototype = {}

for class_id in np.unique(reranker_reference_class_ids):
    positions = np.flatnonzero(
        reranker_reference_class_ids == class_id
    )

    reranker_cow_prototype[int(class_id)] = np.median(
        X_reranker_reference[positions],
        axis=0,
    ).astype(np.float32)


reranker_reference_dim_bin = np.floor(
    reranker_reference_dim / DIM_BIN_WIDTH
).astype(np.int16)

reranker_lactation_prototype = {}

for (class_id, lactation_number), positions in (
    reranker_reference_exact_index.items()
):
    bins_here = reranker_reference_dim_bin[positions]
    prototype_by_bin = {}

    for dim_bin in np.unique(bins_here):
        bin_positions = positions[bins_here == dim_bin]

        prototype_by_bin[int(dim_bin)] = np.median(
            X_reranker_reference[bin_positions],
            axis=0,
        ).astype(np.float32)

    reranker_lactation_prototype[
        (int(class_id), int(lactation_number))
    ] = prototype_by_bin


def get_nearest_lactation_prototype(
    prototype_table,
    class_id,
    lactation_number,
    query_dim,
):
    prototype_dict = prototype_table.get(
        (int(class_id), int(lactation_number))
    )

    if not prototype_dict:
        return None

    query_bin = int(np.floor(float(query_dim) / DIM_BIN_WIDTH))
    available_bins = np.asarray(
        list(prototype_dict.keys()),
        dtype=np.int16,
    )

    nearest_bin = int(
        available_bins[
            np.argmin(np.abs(available_bins - query_bin))
        ]
    )

    return prototype_dict[nearest_bin]


print("Whole-cow prototypes:", f"{len(reranker_cow_prototype):,}")
print(
    "AnimalId-lactation prototype groups:",
    f"{len(reranker_lactation_prototype):,}",
)

Whole-cow prototypes: 6,667
AnimalId-lactation prototype groups: 9,731


In [7]:
# ============================================================
# 6. Candidate-level feature construction
# ============================================================

def _nearest_mean(distances, k):
    k = min(k, len(distances))
    return float(
        np.mean(
            np.partition(distances, k - 1)[:k]
        )
    )


def build_candidate_features_for_one_query(
    query_frame,
    query_id,
):
    if len(query_frame) != 1:
        raise ValueError("query_frame must contain exactly one row.")

    query_vector = transform_distance_features(
        query_frame,
        reranker_numeric_columns,
        reranker_numeric_fill,
        reranker_scaler,
        reranker_reproduction_columns,
    )[0]

    query_event_day = float(
        event_day_from_origin(
            query_frame,
            reranker_origin_date,
        ).iloc[0]
    )

    query_start_day = float(
        lactation_start_day(
            query_frame,
            reranker_origin_date,
        ).iloc[0]
    )

    query_dim = float(query_frame["DaysInMilk"].iloc[0])
    query_lactation = int(
        round(float(query_frame["LactationNumber"].iloc[0]))
    )

    true_animal_id = query_frame["AnimalId"].astype(object).iloc[0]
    true_class_id = reranker_class_to_number.get(
        true_animal_id,
        np.nan,
    )

    candidate = get_candidate_profiles(
        reranker_profile_lookup,
        query_lactation,
        query_start_day,
        start_tolerance_days=RERANKER_CANDIDATE_PARAMS[
            "start_tolerance_days"
        ],
        max_candidates=RERANKER_CANDIDATE_PARAMS[
            "max_candidates"
        ],
        projected_rank_penalty_days=RERANKER_CANDIDATE_PARAMS[
            "projected_rank_penalty_days"
        ],
    )

    if candidate is None:
        return pd.DataFrame()

    candidate_count = len(candidate["class_ids"])

    numeric_feature_positions = {
        name: position
        for position, name in enumerate(reranker_feature_names)
        if not name.startswith("ReproductionStatus_")
    }

    reproduction_positions = [
        position
        for position, name in enumerate(reranker_feature_names)
        if name.startswith("ReproductionStatus_")
    ]

    rows = []

    for candidate_position in range(candidate_count):
        class_id = int(candidate["class_ids"][candidate_position])
        start_difference = float(
            candidate["start_difference"][candidate_position]
        )
        is_projected = int(
            candidate["projected"][candidate_position]
        )
        last_event_day = float(
            candidate["last_event_days"][candidate_position]
        )

        (
            selected_lactation,
            lactation_gap,
        ) = choose_reference_lactation(
            class_id,
            query_lactation,
            reranker_reference_class_lactations,
        )

        if selected_lactation is None:
            continue

        reference_positions = reranker_reference_exact_index.get(
            (class_id, selected_lactation)
        )

        if reference_positions is None or len(reference_positions) == 0:
            continue

        dim_difference = np.abs(
            reranker_reference_dim[reference_positions] - query_dim
        )

        closest_dim_difference = float(np.min(dim_difference))

        inside_dim = (
            dim_difference
            <= RERANKER_CANDIDATE_PARAMS["dim_tolerance_days"]
        )

        if inside_dim.any():
            usable_positions = reference_positions[inside_dim]
        else:
            nearest_order = np.argsort(dim_difference)
            usable_positions = reference_positions[
                nearest_order[: min(10, len(nearest_order))]
            ]

        differences = (
            X_reranker_reference[usable_positions] - query_vector
        )

        manhattan_distances = np.sum(
            np.abs(differences),
            axis=1,
        )

        euclidean_distances = np.sqrt(
            np.sum(differences ** 2, axis=1)
        )

        local_min_manhattan = float(np.min(manhattan_distances))
        local_mean3_manhattan = _nearest_mean(
            manhattan_distances, 3
        )
        local_mean5_manhattan = _nearest_mean(
            manhattan_distances, 5
        )
        local_min_euclidean = float(np.min(euclidean_distances))
        local_mean3_euclidean = _nearest_mean(
            euclidean_distances, 3
        )

        lactation_prototype = get_nearest_lactation_prototype(
            reranker_lactation_prototype,
            class_id,
            selected_lactation,
            query_dim,
        )

        residual_features = {}

        if lactation_prototype is not None:
            prototype_difference = (
                query_vector - lactation_prototype
            )

            lactation_manhattan = float(
                np.sum(np.abs(prototype_difference))
            )

            lactation_euclidean = float(
                np.sqrt(np.sum(prototype_difference ** 2))
            )

            for feature_name, position in (
                numeric_feature_positions.items()
            ):
                residual_features[
                    f"Residual_{feature_name}"
                ] = float(
                    abs(prototype_difference[position])
                )

            if reproduction_positions:
                reproduction_distance = float(
                    np.sum(
                        np.abs(
                            prototype_difference[
                                reproduction_positions
                            ]
                        )
                    )
                )
            else:
                reproduction_distance = 0.0

        else:
            lactation_manhattan = np.nan
            lactation_euclidean = np.nan
            reproduction_distance = np.nan

            for feature_name in numeric_feature_positions:
                residual_features[
                    f"Residual_{feature_name}"
                ] = np.nan

        cow_prototype = reranker_cow_prototype.get(class_id)

        if cow_prototype is not None:
            cow_difference = query_vector - cow_prototype

            cow_manhattan = float(
                np.sum(np.abs(cow_difference))
            )

            cow_euclidean = float(
                np.sqrt(np.sum(cow_difference ** 2))
            )
        else:
            cow_manhattan = np.nan
            cow_euclidean = np.nan

        recency_days = max(
            0.0,
            query_event_day - last_event_day,
        )

        row = {
            "QueryId": query_id,
            "CandidateClassId": class_id,
            "TrueClassId": true_class_id,
            "IsCorrectCandidate": (
                int(class_id == true_class_id)
                if not pd.isna(true_class_id)
                else 0
            ),
            "CandidateCount": candidate_count,
            "CandidateStartRank": candidate_position + 1,
            "StartDifference": start_difference,
            "IsProjected": is_projected,
            "RecencyDays": recency_days,
            "LogRecencyDays": np.log1p(recency_days),
            "LactationGap": lactation_gap,
            "QueryDaysInMilk": query_dim,
            "QueryLactationNumber": query_lactation,
            "ClosestDIMDifference": closest_dim_difference,
            "ReferenceRowCount": len(reference_positions),
            "LocalMinManhattan": local_min_manhattan,
            "LocalMean3Manhattan": local_mean3_manhattan,
            "LocalMean5Manhattan": local_mean5_manhattan,
            "LocalMinEuclidean": local_min_euclidean,
            "LocalMean3Euclidean": local_mean3_euclidean,
            "LactationPrototypeManhattan": lactation_manhattan,
            "LactationPrototypeEuclidean": lactation_euclidean,
            "CowPrototypeManhattan": cow_manhattan,
            "CowPrototypeEuclidean": cow_euclidean,
            "ReproductionDistance": reproduction_distance,
        }

        row.update(residual_features)
        rows.append(row)

    return pd.DataFrame(rows)


RANKING_COLUMNS = [
    "StartDifference",
    "RecencyDays",
    "ClosestDIMDifference",
    "LocalMinManhattan",
    "LocalMean3Manhattan",
    "LocalMean5Manhattan",
    "LocalMinEuclidean",
    "LocalMean3Euclidean",
    "LactationPrototypeManhattan",
    "LactationPrototypeEuclidean",
    "CowPrototypeManhattan",
    "CowPrototypeEuclidean",
    "ReproductionDistance",
    "Residual_DaysInMilk",
    "Residual_Avgmilkflow",
    "Residual_Flow30_60Session",
    "Residual_YieldFirst2Min_Session",
    "Residual_YieldSession",
    "Residual_DurationSession_sec",
    "Residual_YieldPerMinute",
    "Residual_First2MinFraction",
    "Residual_FlowShapeRatio",
]


def add_candidate_rank_features(candidate_pairs):
    result = candidate_pairs.copy()

    for column in RANKING_COLUMNS:
        if column not in result.columns:
            continue

        result[f"{column}_Rank"] = (
            result.groupby("QueryId")[column]
            .rank(
                method="average",
                ascending=True,
                pct=True,
                na_option="bottom",
            )
            .astype(np.float32)
        )

    return result


def build_candidate_dataset(query_frame, prefix, progress_every=100):
    tables = []

    for counter, query_index in enumerate(
        query_frame.index,
        start=1,
    ):
        one_query = query_frame.loc[[query_index]].copy()

        candidate_table = build_candidate_features_for_one_query(
            one_query,
            query_id=f"{prefix}_{query_index}",
        )

        if len(candidate_table) > 0:
            tables.append(candidate_table)

        if (
            counter % progress_every == 0
            or counter == len(query_frame)
        ):
            print(
                f"{prefix}: processed "
                f"{counter} / {len(query_frame)} queries"
            )

    if not tables:
        return pd.DataFrame()

    return add_candidate_rank_features(
        pd.concat(tables, ignore_index=True)
    )


print("Candidate-feature functions loaded.")

Candidate-feature functions loaded.


In [8]:
# ============================================================
# 7. Build / load the 2,000-query train set and 500-query dev set
# ============================================================

reranker_train_seen = reranker_train_queries.loc[
    reranker_train_queries["AnimalId"]
    .astype(object)
    .isin(reranker_class_values)
].copy()

larger_train_queries = reranker_train_seen.sample(
    n=min(TRAIN_QUERY_ROWS, len(reranker_train_seen)),
    random_state=RANDOM_STATE,
).copy()

pilot_dev_queries = reranker_dev_queries.sample(
    n=min(DEV_QUERY_ROWS, len(reranker_dev_queries)),
    random_state=DEV_RANDOM_STATE,
).copy()


if (
    USE_CANDIDATE_CACHE
    and TRAIN_CACHE_PATH.exists()
    and DEV_CACHE_PATH.exists()
):
    print("Loading cached candidate tables...")
    train_candidate_pairs = pd.read_pickle(TRAIN_CACHE_PATH)
    dev_candidate_pairs = pd.read_pickle(DEV_CACHE_PATH)

else:
    print("Building training candidate table...")
    train_candidate_pairs = build_candidate_dataset(
        larger_train_queries,
        prefix="larger_train",
        progress_every=100,
    )

    print("\nBuilding development candidate table...")
    dev_candidate_pairs = build_candidate_dataset(
        pilot_dev_queries,
        prefix="dev_query",
        progress_every=50,
    )

    if USE_CANDIDATE_CACHE:
        train_candidate_pairs.to_pickle(TRAIN_CACHE_PATH)
        dev_candidate_pairs.to_pickle(DEV_CACHE_PATH)
        print("\nCandidate tables saved to:", CACHE_DIR.resolve())


train_presence = (
    train_candidate_pairs.groupby("QueryId")[
        "IsCorrectCandidate"
    ].max()
)

dev_presence = (
    dev_candidate_pairs.groupby("QueryId")[
        "IsCorrectCandidate"
    ].max()
)

print("\nTraining candidate table:", train_candidate_pairs.shape)
print("Training candidate coverage:", f"{train_presence.mean():.2%}")
print("Development candidate table:", dev_candidate_pairs.shape)

# Coverage across all sampled development queries, including any query
# that generated zero candidate rows.
dev_query_ids = [
    f"dev_query_{idx}"
    for idx in pilot_dev_queries.index
]

dev_candidate_coverage = (
    pd.Series(dev_query_ids)
    .map(dev_presence)
    .fillna(0)
    .astype(bool)
    .mean()
)

print(
    "Development candidate coverage ceiling:",
    f"{dev_candidate_coverage:.2%}",
)

Loading cached candidate tables...

Training candidate table: (225735, 56)
Training candidate coverage: 93.20%
Development candidate table: (55380, 56)
Development candidate coverage ceiling: 67.40%


In [9]:
# ============================================================
# 8. Prepare XGBRanker data
# ============================================================

# Training groups with no correct candidate cannot teach the ranker
# which candidate should be ranked first, so remove them.
train_query_has_positive = (
    train_candidate_pairs.groupby("QueryId")[
        "IsCorrectCandidate"
    ].transform("max")
)

xgb_train_data = (
    train_candidate_pairs.loc[
        train_query_has_positive == 1
    ]
    .sort_values(["QueryId", "CandidateStartRank"])
    .reset_index(drop=True)
)

NON_FEATURE_COLUMNS = [
    "QueryId",
    "CandidateClassId",
    "TrueClassId",
    "IsCorrectCandidate",
]

RERANKER_FEATURES = [
    column
    for column in xgb_train_data.columns
    if column not in NON_FEATURE_COLUMNS
]

X_xgb_train = (
    xgb_train_data[RERANKER_FEATURES]
    .replace([np.inf, -np.inf], np.nan)
    .astype(np.float32)
)

y_xgb_train = (
    xgb_train_data["IsCorrectCandidate"]
    .astype(np.float32)
    .to_numpy()
)

qid_xgb_train = (
    pd.factorize(
        xgb_train_data["QueryId"],
        sort=True,
    )[0]
    .astype(np.uint32)
)

# Development candidate matrix.
xgb_dev_data = (
    dev_candidate_pairs
    .sort_values(["QueryId", "CandidateStartRank"])
    .reset_index(drop=True)
)

X_xgb_dev = (
    xgb_dev_data[RERANKER_FEATURES]
    .replace([np.inf, -np.inf], np.nan)
    .astype(np.float32)
)

print("Training rows:", f"{len(xgb_train_data):,}")
print("Training groups:", xgb_train_data["QueryId"].nunique())
print("Number of features:", len(RERANKER_FEATURES))
print("X_train shape:", X_xgb_train.shape)
print("X_dev shape:", X_xgb_dev.shape)
print("Positive rows:", int(y_xgb_train.sum()))
print(
    "qid sorted:",
    bool(np.all(qid_xgb_train[:-1] <= qid_xgb_train[1:])),
)

group_check = (
    xgb_train_data.groupby("QueryId")[
        "IsCorrectCandidate"
    ].sum()
)

print("Groups with exactly one positive:", int((group_check == 1).sum()))
print("Groups with zero positives:", int((group_check == 0).sum()))
print("Groups with >1 positive:", int((group_check > 1).sum()))

Training rows: 210,303
Training groups: 1864
Number of features: 52
X_train shape: (210303, 52)
X_dev shape: (55380, 52)
Positive rows: 1864
qid sorted: True
Groups with exactly one positive: 1864
Groups with zero positives: 0
Groups with >1 positive: 0


In [10]:
# ============================================================
# 9. Train the first XGBRanker baseline
# ============================================================

xgb_ranker = XGBRanker(
    objective="rank:pairwise",
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=1,
    subsample=0.90,
    colsample_bytree=0.90,
    reg_lambda=1.0,
    reg_alpha=0.0,
    tree_method="hist",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

xgb_ranker.fit(
    X_xgb_train,
    y_xgb_train,
    qid=qid_xgb_train,
    verbose=False,
)

print("XGBRanker trained successfully.")

XGBRanker trained successfully.


In [11]:
# ============================================================
# 10. Evaluate on the SAME 500 development queries
# ============================================================

dev_scores = xgb_ranker.predict(X_xgb_dev)

dev_scored = xgb_dev_data[
    [
        "QueryId",
        "CandidateClassId",
        "TrueClassId",
        "IsCorrectCandidate",
    ]
].copy()

dev_scored["RankScore"] = dev_scores

best_candidates = (
    dev_scored
    .sort_values(
        ["QueryId", "RankScore"],
        ascending=[True, False],
    )
    .groupby("QueryId", as_index=False)
    .first()
)

best_candidates["PredictionCorrect"] = (
    best_candidates["CandidateClassId"]
    == best_candidates["TrueClassId"]
)

evaluation_table = pilot_dev_queries[
    ["AnimalId"]
].copy()

evaluation_table["QueryId"] = [
    f"dev_query_{idx}"
    for idx in evaluation_table.index
]

evaluation_table["AnimalSeenInHistory"] = (
    evaluation_table["AnimalId"]
    .astype(object)
    .isin(reranker_class_values)
)

correct_candidate_by_query = (
    dev_candidate_pairs.groupby("QueryId")[
        "IsCorrectCandidate"
    ].max()
)

evaluation_table["CorrectCandidatePresent"] = (
    evaluation_table["QueryId"]
    .map(correct_candidate_by_query)
    .fillna(0)
    .astype(bool)
)

evaluation_table = evaluation_table.merge(
    best_candidates[
        [
            "QueryId",
            "PredictionCorrect",
            "RankScore",
        ]
    ],
    on="QueryId",
    how="left",
)

evaluation_table["PredictionCorrect"] = (
    evaluation_table["PredictionCorrect"]
    .fillna(False)
    .astype(bool)
)

overall_accuracy = evaluation_table[
    "PredictionCorrect"
].mean()

seen_accuracy = evaluation_table.loc[
    evaluation_table["AnimalSeenInHistory"],
    "PredictionCorrect",
].mean()

conditional_accuracy = evaluation_table.loc[
    evaluation_table["CorrectCandidatePresent"],
    "PredictionCorrect",
].mean()

candidate_coverage = evaluation_table[
    "CorrectCandidatePresent"
].mean()

print("=" * 60)
print("XGBRanker overall accuracy:", f"{overall_accuracy:.2%}")
print("XGBRanker seen-AnimalId accuracy:", f"{seen_accuracy:.2%}")
print(
    "XGBRanker conditional ranking accuracy:",
    f"{conditional_accuracy:.2%}",
)
print(
    "Candidate coverage ceiling:",
    f"{candidate_coverage:.2%}",
)
print(
    "Correct predictions:",
    int(evaluation_table["PredictionCorrect"].sum()),
    "/",
    len(evaluation_table),
)

print("\nReference HGB result from earlier experiments:")
print("Overall accuracy:             28.40%")
print("Conditional ranking accuracy: 42.14%")

XGBRanker overall accuracy: 25.60%
XGBRanker seen-AnimalId accuracy: 29.56%
XGBRanker conditional ranking accuracy: 37.98%
Candidate coverage ceiling: 67.40%
Correct predictions: 128 / 500

Reference HGB result from earlier experiments:
Overall accuracy:             28.40%
Conditional ranking accuracy: 42.14%


In [12]:
# ============================================================
# Compare ranking objectives
# Keep EVERYTHING else the same
# ============================================================

OBJECTIVES = [
    "rank:pairwise",
    "rank:ndcg",
]

objective_results = []

for objective_value in OBJECTIVES:

    print("\n" + "=" * 60)
    print("Objective:", objective_value)

    model = XGBRanker(
        objective=objective_value,

        n_estimators=400,
        learning_rate=0.05,
        max_depth=6,
        min_child_weight=1,

        subsample=0.90,
        colsample_bytree=0.90,

        reg_lambda=1.0,
        reg_alpha=0.0,

        tree_method="hist",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )

    model.fit(
        X_xgb_train,
        y_xgb_train,
        qid=qid_xgb_train,
        verbose=False,
    )


    # --------------------------------------------------------
    # Predict
    # --------------------------------------------------------

    scores = model.predict(
        X_xgb_dev
    )

    scored = xgb_dev_data[
        [
            "QueryId",
            "CandidateClassId",
            "TrueClassId",
            "IsCorrectCandidate",
        ]
    ].copy()

    scored["RankScore"] = scores


    # --------------------------------------------------------
    # Best candidate per query
    # --------------------------------------------------------

    best = (
        scored
        .sort_values(
            ["QueryId", "RankScore"],
            ascending=[True, False],
        )
        .groupby(
            "QueryId",
            as_index=False
        )
        .first()
    )

    best["PredictionCorrect"] = (
        best["CandidateClassId"]
        ==
        best["TrueClassId"]
    )


    # --------------------------------------------------------
    # Evaluation
    # --------------------------------------------------------

    eval_table = pilot_dev_queries[
        ["AnimalId"]
    ].copy()

    eval_table["QueryId"] = [
        f"dev_query_{idx}"
        for idx in eval_table.index
    ]

    eval_table["AnimalSeenInHistory"] = (
        eval_table["AnimalId"]
        .astype(object)
        .isin(reranker_class_values)
    )

    eval_table[
        "CorrectCandidatePresent"
    ] = (
        eval_table["QueryId"]
        .map(correct_candidate_by_query)
        .fillna(0)
        .astype(bool)
    )

    eval_table = (
        eval_table
        .merge(
            best[
                [
                    "QueryId",
                    "PredictionCorrect",
                ]
            ],
            on="QueryId",
            how="left",
        )
    )

    eval_table["PredictionCorrect"] = (
        eval_table["PredictionCorrect"]
        .fillna(False)
        .astype(bool)
    )


    overall = (
        eval_table[
            "PredictionCorrect"
        ].mean()
    )

    seen = (
        eval_table.loc[
            eval_table[
                "AnimalSeenInHistory"
            ],
            "PredictionCorrect"
        ].mean()
    )

    conditional = (
        eval_table.loc[
            eval_table[
                "CorrectCandidatePresent"
            ],
            "PredictionCorrect"
        ].mean()
    )


    objective_results.append({
        "objective": objective_value,
        "overall_accuracy": overall * 100,
        "seen_accuracy": seen * 100,
        "conditional_accuracy": conditional * 100,
    })


objective_results_df = (
    pd.DataFrame(
        objective_results
    )
)

display(
    objective_results_df
)


Objective: rank:pairwise

Objective: rank:ndcg


,objective,overall_accuracy,seen_accuracy,conditional_accuracy
0,rank:pairwise,25.6,29.561201,37.982196
1,rank:ndcg,26.0,30.023095,38.575668


In [13]:
# ============================================================
# NEXT STEP
# Test whether cow-prototype fallback can recover
# missing correct candidates
# ============================================================

# Convert cow prototypes into one matrix
prototype_class_ids = np.array(
    sorted(reranker_cow_prototype.keys()),
    dtype=np.int32
)

prototype_matrix = np.vstack([
    reranker_cow_prototype[class_id]
    for class_id in prototype_class_ids
]).astype(np.float32)


TOP_K_VALUES = [
    20,
    50,
    100,
    200,
]

prototype_recovery_results = []


for top_k in TOP_K_VALUES:

    recovered_seen = 0
    recovered_all = 0

    total_seen_queries = 0
    total_queries = len(pilot_dev_queries)

    original_correct = 0
    combined_correct = 0

    for query_index in pilot_dev_queries.index:

        query_frame = (
            pilot_dev_queries
            .loc[[query_index]]
            .copy()
        )

        query_id = f"dev_query_{query_index}"

        true_animal_id = (
            query_frame["AnimalId"]
            .astype(object)
            .iloc[0]
        )

        true_class_id = (
            reranker_class_to_number
            .get(true_animal_id, None)
        )

        # ----------------------------------------
        # Original structural candidate status
        # ----------------------------------------

        original_present = bool(
            correct_candidate_by_query
            .get(query_id, False)
        )

        if original_present:
            original_correct += 1
            combined_correct += 1

        # Cannot recover completely unseen AnimalIds
        if true_class_id is None:
            continue

        total_seen_queries += 1

        if original_present:
            continue


        # ----------------------------------------
        # Transform query to SAME feature space
        # ----------------------------------------

        query_vector = (
            transform_distance_features(
                query_frame,
                reranker_numeric_columns,
                reranker_numeric_fill,
                reranker_scaler,
                reranker_reproduction_columns,
            )[0]
        )


        # ----------------------------------------
        # Whole-cow prototype Manhattan distance
        # ----------------------------------------

        distances = np.sum(
            np.abs(
                prototype_matrix
                - query_vector
            ),
            axis=1,
        )

        nearest_positions = np.argpartition(
            distances,
            min(top_k, len(distances)) - 1
        )[:top_k]

        fallback_class_ids = (
            prototype_class_ids[
                nearest_positions
            ]
        )


        # ----------------------------------------
        # Did prototype fallback recover true cow?
        # ----------------------------------------

        if true_class_id in fallback_class_ids:

            recovered_seen += 1
            recovered_all += 1
            combined_correct += 1


    original_coverage = (
        original_correct
        / total_queries
    )

    combined_coverage = (
        combined_correct
        / total_queries
    )

    seen_recovery_rate = (
        recovered_seen
        /
        max(
            1,
            total_seen_queries
            - original_correct
        )
    )


    prototype_recovery_results.append({
        "prototype_top_k":
            top_k,

        "original_coverage":
            original_coverage * 100,

        "combined_coverage":
            combined_coverage * 100,

        "coverage_gain_pp":
            (
                combined_coverage
                - original_coverage
            ) * 100,

        "missing_seen_queries_recovered":
            recovered_seen,

        "seen_missing_recovery_rate":
            seen_recovery_rate * 100,
    })


prototype_recovery_results_df = (
    pd.DataFrame(
        prototype_recovery_results
    )
)

display(
    prototype_recovery_results_df
)

,prototype_top_k,original_coverage,combined_coverage,coverage_gain_pp,missing_seen_queries_recovered,seen_missing_recovery_rate
0,20,67.4,67.4,0.0,0,0.000000
1,50,67.4,67.4,0.0,0,0.000000
2,100,67.4,67.6,0.2,1,1.041667
3,200,67.4,67.6,0.2,1,1.041667


In [14]:
# ============================================================
# Diagnose WHY seen AnimalIds are missing from candidate set
# ============================================================

diagnostic_rows = []

current_tolerance = RERANKER_CANDIDATE_PARAMS[
    "start_tolerance_days"
]
current_max_candidates = RERANKER_CANDIDATE_PARAMS[
    "max_candidates"
]
projection_penalty = RERANKER_CANDIDATE_PARAMS[
    "projected_rank_penalty_days"
]


# Only inspect:
# AnimalId exists in history
# BUT correct candidate was not generated
missing_seen_eval = evaluation_table.loc[
    evaluation_table["AnimalSeenInHistory"]
    &
    (~evaluation_table["CorrectCandidatePresent"])
].copy()


for _, eval_row in missing_seen_eval.iterrows():

    query_id = eval_row["QueryId"]

    # Recover original dataframe index
    query_index = int(
        query_id.replace("dev_query_", "")
    )

    query_frame = (
        pilot_dev_queries
        .loc[[query_index]]
        .copy()
    )

    true_animal_id = (
        query_frame["AnimalId"]
        .astype(object)
        .iloc[0]
    )

    true_class_id = (
        reranker_class_to_number
        .get(true_animal_id)
    )

    query_lactation = int(
        round(
            float(
                query_frame[
                    "LactationNumber"
                ].iloc[0]
            )
        )
    )

    query_start_day = float(
        lactation_start_day(
            query_frame,
            reranker_origin_date,
        ).iloc[0]
    )


    # --------------------------------------------------------
    # Get ALL structural profiles for this lactation number
    # --------------------------------------------------------

    profile = reranker_profile_lookup.get(
        query_lactation
    )

    if profile is None:

        diagnostic_rows.append({
            "QueryId": query_id,
            "Reason":
                "No profiles for this lactation number",
            "TrueStartDifference": np.nan,
            "TrueStructuralRank": np.nan,
            "ProfilesInLactation": 0,
            "ProfilesInsideTolerance": 0,
        })

        continue


    class_ids = profile["class_ids"]

    true_positions = np.flatnonzero(
        class_ids == true_class_id
    )


    # --------------------------------------------------------
    # True cow has NO structural profile for this lactation
    # --------------------------------------------------------

    if len(true_positions) == 0:

        diagnostic_rows.append({
            "QueryId": query_id,
            "Reason":
                "True cow has no profile for query lactation",
            "TrueStartDifference": np.nan,
            "TrueStructuralRank": np.nan,
            "ProfilesInLactation": len(class_ids),
            "ProfilesInsideTolerance": np.nan,
        })

        continue


    # --------------------------------------------------------
    # Calculate structural ranking
    # --------------------------------------------------------

    start_difference = np.abs(
        profile["start_days"]
        -
        query_start_day
    )

    rank_difference = (
        start_difference
        +
        profile["projected"].astype(
            np.float32
        )
        * projection_penalty
    )

    inside = (
        start_difference
        <= current_tolerance
    )


    # Candidate generator behaviour
    if inside.any():
        candidate_pool = np.flatnonzero(
            inside
        )
    else:
        candidate_pool = np.arange(
            len(class_ids)
        )


    sorted_pool = candidate_pool[
        np.argsort(
            rank_difference[
                candidate_pool
            ]
        )
    ]

    selected_positions = (
        sorted_pool[
            :current_max_candidates
        ]
    )


    true_position = int(
        true_positions[
            np.argmin(
                rank_difference[
                    true_positions
                ]
            )
        ]
    )

    true_start_difference = float(
        start_difference[
            true_position
        ]
    )


    # Rank among the pool actually considered
    pool_match = np.flatnonzero(
        sorted_pool
        ==
        true_position
    )

    if len(pool_match) > 0:
        true_rank = int(
            pool_match[0] + 1
        )
    else:
        true_rank = np.nan


    # --------------------------------------------------------
    # Determine failure reason
    # --------------------------------------------------------

    if true_position in selected_positions:

        reason = (
            "Selected structurally but lost "
            "during feature construction"
        )

    elif (
        inside.any()
        and
        not inside[
            true_position
        ]
    ):

        reason = (
            "True cow outside start-date tolerance"
        )

    elif (
        not pd.isna(true_rank)
        and
        true_rank > current_max_candidates
    ):

        reason = (
            "True cow ranked below max_candidates"
        )

    else:

        reason = "Other structural reason"


    diagnostic_rows.append({
        "QueryId":
            query_id,

        "Reason":
            reason,

        "TrueStartDifference":
            true_start_difference,

        "TrueStructuralRank":
            true_rank,

        "ProfilesInLactation":
            len(class_ids),

        "ProfilesInsideTolerance":
            int(
                inside.sum()
            ),
    })


# ============================================================
# Results
# ============================================================

candidate_failure_diagnostics = (
    pd.DataFrame(
        diagnostic_rows
    )
)

print(
    "Seen queries missing correct candidate:",
    len(candidate_failure_diagnostics)
)

print("\nFailure reasons:")

failure_summary = (
    candidate_failure_diagnostics[
        "Reason"
    ]
    .value_counts()
    .rename_axis("Reason")
    .reset_index(
        name="QueryCount"
    )
)

failure_summary[
    "Percentage"
] = (
    failure_summary[
        "QueryCount"
    ]
    /
    len(
        candidate_failure_diagnostics
    )
    * 100
)

display(
    failure_summary
)

print("\nStart-difference / rank diagnostics:")

display(
    candidate_failure_diagnostics[
        [
            "TrueStartDifference",
            "TrueStructuralRank",
            "ProfilesInsideTolerance",
        ]
    ]
    .describe()
)

Seen queries missing correct candidate: 96

Failure reasons:


,Reason,QueryCount,Percentage
0,True cow outside start-date tolerance,89,92.708333
1,True cow ranked below max_candidates,7,7.291667



Start-difference / rank diagnostics:


,TrueStartDifference,TrueStructuralRank,ProfilesInsideTolerance
count,96.000000,7.000000,96.000000
mean,62.614583,148.000000,148.302083
std,50.756046,19.148542,58.368903
min,14.000000,128.000000,29.000000
25%,27.750000,134.500000,94.500000
50%,41.500000,149.000000,167.000000
75%,79.000000,152.500000,190.500000
max,251.000000,185.000000,225.000000


In [16]:
# ============================================================
# NEXT STEP
# Re-test start-date tolerance
#
# Keep max_candidates = 120
# Only change start tolerance
# ============================================================

START_TOLERANCE_VALUES = [
    21,
    30,
    45,
    60,
    90,
    120,
    180,
]

MAX_CANDIDATES = 120

tolerance_results = []


for tolerance_days in START_TOLERANCE_VALUES:

    total_queries = len(pilot_dev_queries)

    seen_queries = 0

    correct_all = 0
    correct_seen = 0

    candidate_counts = []


    for query_index in pilot_dev_queries.index:

        query_frame = (
            pilot_dev_queries
            .loc[[query_index]]
            .copy()
        )

        true_animal_id = (
            query_frame["AnimalId"]
            .astype(object)
            .iloc[0]
        )

        true_class_id = (
            reranker_class_to_number
            .get(
                true_animal_id,
                None
            )
        )


        # ----------------------------------------------------
        # Completely unseen AnimalId cannot be recovered
        # ----------------------------------------------------

        if true_class_id is None:
            continue

        seen_queries += 1


        query_lactation = int(
            round(
                float(
                    query_frame[
                        "LactationNumber"
                    ].iloc[0]
                )
            )
        )

        query_start_day = float(
            lactation_start_day(
                query_frame,
                reranker_origin_date,
            ).iloc[0]
        )


        profile = (
            reranker_profile_lookup
            .get(
                query_lactation
            )
        )


        if profile is None:
            candidate_counts.append(0)
            continue


        class_ids = (
            profile["class_ids"]
        )


        # ----------------------------------------------------
        # Structural distance
        # ----------------------------------------------------

        start_difference = np.abs(
            profile["start_days"]
            -
            query_start_day
        )

        rank_difference = (
            start_difference
            +
            profile["projected"].astype(
                np.float32
            )
            *
            RERANKER_CANDIDATE_PARAMS[
                "projected_rank_penalty_days"
            ]
        )


        # ----------------------------------------------------
        # Apply tested start tolerance
        # ----------------------------------------------------

        inside = (
            start_difference
            <= tolerance_days
        )


        if inside.any():

            candidate_pool = np.flatnonzero(
                inside
            )

        else:

            candidate_pool = np.arange(
                len(class_ids)
            )


        # ----------------------------------------------------
        # Rank candidates
        # ----------------------------------------------------

        sorted_pool = (
            candidate_pool[
                np.argsort(
                    rank_difference[
                        candidate_pool
                    ]
                )
            ]
        )


        selected_positions = (
            sorted_pool[
                :MAX_CANDIDATES
            ]
        )


        selected_class_ids = (
            class_ids[
                selected_positions
            ]
        )


        candidate_counts.append(
            len(
                selected_class_ids
            )
        )


        # ----------------------------------------------------
        # Coverage
        # ----------------------------------------------------

        if true_class_id in selected_class_ids:

            correct_seen += 1
            correct_all += 1


    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    overall_coverage = (
        correct_all
        /
        total_queries
    )

    seen_coverage = (
        correct_seen
        /
        seen_queries
    )

    tolerance_results.append({

        "start_tolerance_days":
            tolerance_days,

        "overall_coverage":
            overall_coverage * 100,

        "seen_coverage":
            seen_coverage * 100,

        "correct_queries":
            correct_all,

        "mean_candidate_count":
            np.mean(
                candidate_counts
            ),

        "median_candidate_count":
            np.median(
                candidate_counts
            ),

        "max_candidate_count":
            np.max(
                candidate_counts
            ),
    })


# ============================================================
# Results
# ============================================================

tolerance_results_df = (
    pd.DataFrame(
        tolerance_results
    )
)

tolerance_results_df[
    "coverage_gain_pp"
] = (
    tolerance_results_df[
        "overall_coverage"
    ]
    -
    tolerance_results_df.loc[
        tolerance_results_df[
            "start_tolerance_days"
        ]
        == 21,
        "overall_coverage"
    ].iloc[0]
)

display(
    tolerance_results_df
)

,start_tolerance_days,overall_coverage,seen_coverage,correct_queries,mean_candidate_count,median_candidate_count,max_candidate_count,coverage_gain_pp
0,21,67.4,77.829099,337,110.958430,120.0,120,0.0
1,30,67.8,78.290993,339,115.482679,120.0,120,0.4
2,45,68.4,78.983834,342,118.533487,120.0,120,1.0
3,60,68.6,79.214781,343,119.374134,120.0,120,1.2
4,90,68.4,78.983834,342,119.819861,120.0,120,1.0
5,120,68.6,79.214781,343,119.847575,120.0,120,1.2
6,180,68.4,78.983834,342,119.912240,120.0,120,1.0


In [17]:
# ============================================================
# NEXT STEP
# Test GLOBAL structural ranking
#
# No hard start-date tolerance.
# Rank ALL profiles for the same lactation,
# then keep the top K candidates.
# ============================================================

GLOBAL_TOP_K_VALUES = [
    80,
    120,
    160,
    240,
    400,
]

global_rank_results = []


for top_k in GLOBAL_TOP_K_VALUES:

    total_queries = len(pilot_dev_queries)

    seen_queries = 0
    correct_all = 0
    correct_seen = 0

    candidate_counts = []


    for query_index in pilot_dev_queries.index:

        query_frame = (
            pilot_dev_queries
            .loc[[query_index]]
            .copy()
        )

        true_animal_id = (
            query_frame["AnimalId"]
            .astype(object)
            .iloc[0]
        )

        true_class_id = (
            reranker_class_to_number
            .get(
                true_animal_id,
                None
            )
        )

        # Completely unseen AnimalId
        if true_class_id is None:
            continue

        seen_queries += 1


        # ----------------------------------------------------
        # Query lactation + estimated lactation start
        # ----------------------------------------------------

        query_lactation = int(
            round(
                float(
                    query_frame[
                        "LactationNumber"
                    ].iloc[0]
                )
            )
        )

        query_start_day = float(
            lactation_start_day(
                query_frame,
                reranker_origin_date,
            ).iloc[0]
        )


        profile = (
            reranker_profile_lookup
            .get(
                query_lactation
            )
        )

        if profile is None:
            candidate_counts.append(0)
            continue


        class_ids = profile["class_ids"]


        # ----------------------------------------------------
        # GLOBAL structural score
        #
        # IMPORTANT:
        # No hard tolerance filtering here.
        # ----------------------------------------------------

        start_difference = np.abs(
            profile["start_days"]
            -
            query_start_day
        )

        structural_score = (
            start_difference
            +
            profile["projected"].astype(
                np.float32
            )
            *
            RERANKER_CANDIDATE_PARAMS[
                "projected_rank_penalty_days"
            ]
        )


        # ----------------------------------------------------
        # Rank ALL structural profiles
        # ----------------------------------------------------

        sorted_positions = np.argsort(
            structural_score
        )

        selected_positions = (
            sorted_positions[
                :min(
                    top_k,
                    len(sorted_positions)
                )
            ]
        )

        selected_class_ids = (
            class_ids[
                selected_positions
            ]
        )

        # Multiple structural profiles can theoretically
        # map to the same cow, so evaluate unique cows.
        selected_unique_classes = (
            np.unique(
                selected_class_ids
            )
        )

        candidate_counts.append(
            len(
                selected_unique_classes
            )
        )


        # ----------------------------------------------------
        # Coverage
        # ----------------------------------------------------

        if true_class_id in selected_unique_classes:

            correct_seen += 1
            correct_all += 1


    overall_coverage = (
        correct_all
        /
        total_queries
    )

    seen_coverage = (
        correct_seen
        /
        seen_queries
    )


    global_rank_results.append({

        "global_top_k":
            top_k,

        "overall_coverage":
            overall_coverage * 100,

        "seen_coverage":
            seen_coverage * 100,

        "correct_queries":
            correct_all,

        "mean_unique_candidates":
            np.mean(
                candidate_counts
            ),

        "median_unique_candidates":
            np.median(
                candidate_counts
            ),
    })


# ============================================================
# Results
# ============================================================

global_rank_results_df = (
    pd.DataFrame(
        global_rank_results
    )
)

global_rank_results_df[
    "gain_vs_current_pp"
] = (
    global_rank_results_df[
        "overall_coverage"
    ]
    -
    67.4
)

display(
    global_rank_results_df
)

,global_top_k,overall_coverage,seen_coverage,correct_queries,mean_unique_candidates,median_unique_candidates,gain_vs_current_pp
0,80,65.4,75.519630,327,80.000000,80.0,-2.0
1,120,68.2,78.752887,341,120.000000,120.0,0.8
2,160,70.4,81.293303,352,160.000000,160.0,3.0
3,240,73.8,85.219400,369,239.946882,240.0,6.4
4,400,81.0,93.533487,405,399.577367,400.0,13.6


In [18]:
# ============================================================
# NEXT STEP
# Hybrid candidate retrieval
#
# Original hard-tolerance candidates
# +
# Global structural Top-K candidates
# ============================================================

GLOBAL_ADD_VALUES = [
    40,
    80,
    120,
    160,
    240,
    320,
    400,
]

hybrid_candidate_results = []


for global_top_k in GLOBAL_ADD_VALUES:

    total_queries = len(pilot_dev_queries)

    seen_queries = 0
    correct_all = 0
    correct_seen = 0

    candidate_counts = []


    for query_index in pilot_dev_queries.index:

        query_frame = (
            pilot_dev_queries
            .loc[[query_index]]
            .copy()
        )

        true_animal_id = (
            query_frame["AnimalId"]
            .astype(object)
            .iloc[0]
        )

        true_class_id = (
            reranker_class_to_number.get(
                true_animal_id,
                None
            )
        )


        # ----------------------------------------------------
        # Unseen cow cannot be retrieved
        # ----------------------------------------------------

        if true_class_id is None:
            continue

        seen_queries += 1


        query_lactation = int(
            round(
                float(
                    query_frame[
                        "LactationNumber"
                    ].iloc[0]
                )
            )
        )

        query_start_day = float(
            lactation_start_day(
                query_frame,
                reranker_origin_date,
            ).iloc[0]
        )


        # ====================================================
        # PART 1
        # Original hard-tolerance candidates
        # ====================================================

        original_candidate = get_candidate_profiles(
            reranker_profile_lookup,
            query_lactation,
            query_start_day,

            start_tolerance_days=
                RERANKER_CANDIDATE_PARAMS[
                    "start_tolerance_days"
                ],

            max_candidates=
                RERANKER_CANDIDATE_PARAMS[
                    "max_candidates"
                ],

            projected_rank_penalty_days=
                RERANKER_CANDIDATE_PARAMS[
                    "projected_rank_penalty_days"
                ],
        )


        if original_candidate is None:

            original_class_ids = np.array(
                [],
                dtype=np.int32
            )

        else:

            original_class_ids = (
                original_candidate[
                    "class_ids"
                ]
            )


        # ====================================================
        # PART 2
        # Global structural candidates
        # ====================================================

        profile = (
            reranker_profile_lookup.get(
                query_lactation
            )
        )


        if profile is None:

            global_class_ids = np.array(
                [],
                dtype=np.int32
            )

        else:

            start_difference = np.abs(
                profile["start_days"]
                -
                query_start_day
            )

            structural_score = (
                start_difference
                +
                profile["projected"].astype(
                    np.float32
                )
                *
                RERANKER_CANDIDATE_PARAMS[
                    "projected_rank_penalty_days"
                ]
            )

            sorted_positions = np.argsort(
                structural_score
            )

            selected_positions = (
                sorted_positions[
                    :min(
                        global_top_k,
                        len(sorted_positions)
                    )
                ]
            )

            global_class_ids = (
                profile[
                    "class_ids"
                ][selected_positions]
            )


        # ====================================================
        # PART 3
        # Union + remove duplicates
        # ====================================================

        combined_class_ids = np.unique(
            np.concatenate(
                [
                    original_class_ids,
                    global_class_ids,
                ]
            )
        )


        candidate_counts.append(
            len(
                combined_class_ids
            )
        )


        # ====================================================
        # Coverage
        # ====================================================

        if true_class_id in combined_class_ids:

            correct_seen += 1
            correct_all += 1


    overall_coverage = (
        correct_all
        /
        total_queries
    )

    seen_coverage = (
        correct_seen
        /
        seen_queries
    )


    hybrid_candidate_results.append({

        "global_added_top_k":
            global_top_k,

        "overall_coverage":
            overall_coverage * 100,

        "seen_coverage":
            seen_coverage * 100,

        "correct_queries":
            correct_all,

        "mean_unique_candidates":
            np.mean(
                candidate_counts
            ),

        "median_unique_candidates":
            np.median(
                candidate_counts
            ),

        "coverage_gain_vs_current_pp":
            (
                overall_coverage * 100
                - 67.4
            ),
    })


# ============================================================
# Results
# ============================================================

hybrid_candidate_results_df = (
    pd.DataFrame(
        hybrid_candidate_results
    )
)

display(
    hybrid_candidate_results_df
)

,global_added_top_k,overall_coverage,seen_coverage,correct_queries,mean_unique_candidates,median_unique_candidates,coverage_gain_vs_current_pp
0,40,67.4,77.829099,337,111.244804,120.0,0.0
1,80,67.8,78.290993,339,113.907621,120.0,0.4
2,120,68.6,79.214781,343,121.766744,122.0,1.2
3,160,70.4,81.293303,352,160.000000,160.0,3.0
4,240,73.8,85.219400,369,239.946882,240.0,6.4
5,320,78.8,90.993072,394,319.762125,320.0,11.4
6,400,81.0,93.533487,405,399.577367,400.0,13.6


In [19]:
# ============================================================
# NEXT STEP
# Switch candidate generator to GLOBAL TOP 320
#
# First rebuild ONLY the 500-query development candidate table.
# Do not rebuild the 2,000-query training table yet.
# ============================================================

GLOBAL_MAX_CANDIDATES = 320

RERANKER_CANDIDATE_PARAMS[
    "max_candidates"
] = GLOBAL_MAX_CANDIDATES


# ============================================================
# Redefine get_candidate_profiles()
#
# Same function signature as before,
# but NO hard start-date tolerance.
# ============================================================

def get_candidate_profiles(
    profile_lookup,
    lactation_number,
    query_start_day,
    start_tolerance_days,
    max_candidates,
    projected_rank_penalty_days,
):
    """
    Global structural candidate generator.

    For the query lactation:
        1. Consider ALL structural profiles.
        2. Rank by lactation-start difference
           + projection penalty.
        3. Keep the best max_candidates.

    start_tolerance_days is intentionally ignored.
    It remains in the function signature so the existing
    feature-building code does not need to be changed.
    """

    if (
        pd.isna(lactation_number)
        or pd.isna(query_start_day)
    ):
        return None

    lactation_number = int(
        round(
            float(lactation_number)
        )
    )

    profile = profile_lookup.get(
        lactation_number
    )

    if profile is None:
        return None


    # --------------------------------------------------------
    # Structural distance
    # --------------------------------------------------------

    start_difference = np.abs(
        profile["start_days"]
        -
        float(query_start_day)
    )


    structural_score = (
        start_difference
        +
        profile["projected"].astype(
            np.float32
        )
        *
        projected_rank_penalty_days
    )


    # --------------------------------------------------------
    # GLOBAL ranking
    # No tolerance filtering
    # --------------------------------------------------------

    positions = np.argsort(
        structural_score
    )

    positions = positions[
        :min(
            max_candidates,
            len(positions)
        )
    ]


    return {
        key: value[positions]
        for key, value in profile.items()
    } | {
        "start_difference":
            start_difference[positions]
    }


print(
    "Candidate generator updated:"
)

print(
    "GLOBAL Top",
    RERANKER_CANDIDATE_PARAMS[
        "max_candidates"
    ]
)

Candidate generator updated:
GLOBAL Top 320


In [20]:
# ============================================================
# Rebuild ONLY the same 500 development queries
# using GLOBAL TOP 320
# ============================================================

GLOBAL320_DEV_CACHE_PATH = (
    CACHE_DIR
    /
    "dev_candidates_500_global320_dim30_bin7.pkl"
)


if GLOBAL320_DEV_CACHE_PATH.exists():

    print(
        "Loading cached GLOBAL-320 dev candidates..."
    )

    global320_dev_candidate_pairs = (
        pd.read_pickle(
            GLOBAL320_DEV_CACHE_PATH
        )
    )

else:

    print(
        "Building GLOBAL-320 development candidates..."
    )

    global320_dev_candidate_pairs = (
        build_candidate_dataset(
            pilot_dev_queries,
            prefix="dev_query",
            progress_every=50,
        )
    )

    global320_dev_candidate_pairs.to_pickle(
        GLOBAL320_DEV_CACHE_PATH
    )

    print(
        "Saved:",
        GLOBAL320_DEV_CACHE_PATH
    )


# ============================================================
# Actual candidate coverage after FULL feature construction
# ============================================================

global320_presence = (
    global320_dev_candidate_pairs
    .groupby("QueryId")[
        "IsCorrectCandidate"
    ]
    .max()
)


global320_query_ids = [
    f"dev_query_{idx}"
    for idx in pilot_dev_queries.index
]


global320_coverage = (
    pd.Series(
        global320_query_ids
    )
    .map(
        global320_presence
    )
    .fillna(0)
    .mean()
)


# ------------------------------------------------------------
# Seen-AnimalId coverage
# ------------------------------------------------------------

global320_query_info = (
    pilot_dev_queries[
        ["AnimalId"]
    ]
    .copy()
)

global320_query_info[
    "QueryId"
] = global320_query_ids

global320_query_info[
    "AnimalSeenInHistory"
] = (
    global320_query_info[
        "AnimalId"
    ]
    .astype(object)
    .isin(
        reranker_class_values
    )
)

global320_query_info[
    "CorrectCandidatePresent"
] = (
    global320_query_info[
        "QueryId"
    ]
    .map(
        global320_presence
    )
    .fillna(0)
    .astype(bool)
)


global320_seen_coverage = (
    global320_query_info.loc[
        global320_query_info[
            "AnimalSeenInHistory"
        ],
        "CorrectCandidatePresent",
    ]
    .mean()
)


# ------------------------------------------------------------
# Candidate counts
# ------------------------------------------------------------

global320_candidate_counts = (
    global320_dev_candidate_pairs
    .groupby("QueryId")
    .size()
)


print("=" * 60)

print(
    "Candidate rows:",
    f"{len(global320_dev_candidate_pairs):,}"
)

print(
    "Queries with candidate rows:",
    global320_dev_candidate_pairs[
        "QueryId"
    ].nunique()
)

print(
    "Mean candidates/query:",
    f"{global320_candidate_counts.mean():.2f}"
)

print(
    "Overall candidate coverage:",
    f"{global320_coverage:.2%}"
)

print(
    "Seen-AnimalId candidate coverage:",
    f"{global320_seen_coverage:.2%}"
)

print(
    "Correct queries:",
    int(
        global320_query_info[
            "CorrectCandidatePresent"
        ].sum()
    ),
    "/",
    len(
        global320_query_info
    )
)

Building GLOBAL-320 development candidates...
dev_query: processed 50 / 500 queries
dev_query: processed 100 / 500 queries
dev_query: processed 150 / 500 queries
dev_query: processed 200 / 500 queries
dev_query: processed 250 / 500 queries
dev_query: processed 300 / 500 queries
dev_query: processed 350 / 500 queries
dev_query: processed 400 / 500 queries
dev_query: processed 450 / 500 queries
dev_query: processed 500 / 500 queries
Saved: xgboost_cache/dev_candidates_500_global320_dim30_bin7.pkl
Candidate rows: 159,897
Queries with candidate rows: 500
Mean candidates/query: 319.79
Overall candidate coverage: 78.80%
Seen-AnimalId candidate coverage: 90.99%
Correct queries: 394 / 500


In [21]:
# ============================================================
# NEXT STEP
# Rebuild the SAME 2,000 training queries
# using GLOBAL TOP 320
# ============================================================

GLOBAL320_TRAIN_CACHE_PATH = (
    CACHE_DIR
    /
    "train_candidates_2000_global320_dim30_bin7.pkl"
)


# ------------------------------------------------------------
# Build or load cached candidate table
# ------------------------------------------------------------

if GLOBAL320_TRAIN_CACHE_PATH.exists():

    print(
        "Loading cached GLOBAL-320 training candidates..."
    )

    global320_train_candidate_pairs = (
        pd.read_pickle(
            GLOBAL320_TRAIN_CACHE_PATH
        )
    )

else:

    print(
        "Building GLOBAL-320 training candidates..."
    )

    global320_train_candidate_pairs = (
        build_candidate_dataset(
            larger_train_queries,
            prefix="train_query",
            progress_every=100,
        )
    )

    global320_train_candidate_pairs.to_pickle(
        GLOBAL320_TRAIN_CACHE_PATH
    )

    print(
        "Saved:",
        GLOBAL320_TRAIN_CACHE_PATH
    )


# ============================================================
# Add within-query rank features
# ============================================================

global320_train_candidate_pairs_ranked = (
    add_candidate_rank_features(
        global320_train_candidate_pairs
    )
)


# ============================================================
# Candidate coverage diagnostics
# ============================================================

train_presence_global320 = (
    global320_train_candidate_pairs_ranked
    .groupby("QueryId")[
        "IsCorrectCandidate"
    ]
    .max()
)


total_training_queries = len(
    larger_train_queries
)

queries_with_correct_candidate = int(
    train_presence_global320.sum()
)

queries_without_correct_candidate = (
    total_training_queries
    -
    queries_with_correct_candidate
)


training_candidate_coverage = (
    queries_with_correct_candidate
    /
    total_training_queries
)


candidate_counts_global320 = (
    global320_train_candidate_pairs_ranked
    .groupby("QueryId")
    .size()
)


print("=" * 60)

print(
    "Candidate rows:",
    f"{len(global320_train_candidate_pairs_ranked):,}"
)

print(
    "Training QueryIds:",
    global320_train_candidate_pairs_ranked[
        "QueryId"
    ].nunique()
)

print(
    "Mean candidates/query:",
    f"{candidate_counts_global320.mean():.2f}"
)

print(
    "Training candidate coverage:",
    f"{training_candidate_coverage:.2%}"
)

print(
    "Queries with correct candidate:",
    queries_with_correct_candidate
)

print(
    "Queries without correct candidate:",
    queries_without_correct_candidate
)

print(
    "Table shape:",
    global320_train_candidate_pairs_ranked.shape
)

print(
    "Memory:",
    f"{global320_train_candidate_pairs_ranked.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

Building GLOBAL-320 training candidates...
train_query: processed 100 / 2000 queries
train_query: processed 200 / 2000 queries
train_query: processed 300 / 2000 queries
train_query: processed 400 / 2000 queries
train_query: processed 500 / 2000 queries
train_query: processed 600 / 2000 queries
train_query: processed 700 / 2000 queries
train_query: processed 800 / 2000 queries
train_query: processed 900 / 2000 queries
train_query: processed 1000 / 2000 queries
train_query: processed 1100 / 2000 queries
train_query: processed 1200 / 2000 queries
train_query: processed 1300 / 2000 queries
train_query: processed 1400 / 2000 queries
train_query: processed 1500 / 2000 queries
train_query: processed 1600 / 2000 queries
train_query: processed 1700 / 2000 queries
train_query: processed 1800 / 2000 queries
train_query: processed 1900 / 2000 queries
train_query: processed 2000 / 2000 queries
Saved: xgboost_cache/train_candidates_2000_global320_dim30_bin7.pkl
Candidate rows: 639,040
Training Query

In [22]:
# ============================================================
# NEXT STEP
# Prepare GLOBAL-320 data for HGB reranker
# ============================================================

# ------------------------------------------------------------
# 1. Add rank features to development table
# ------------------------------------------------------------

global320_dev_candidate_pairs_ranked = (
    add_candidate_rank_features(
        global320_dev_candidate_pairs
    )
)


# ------------------------------------------------------------
# 2. Keep only training queries
#    where the correct candidate is available
# ------------------------------------------------------------

train_has_positive_global320 = (
    global320_train_candidate_pairs_ranked
    .groupby("QueryId")["IsCorrectCandidate"]
    .transform("max")
)

global320_hgb_train = (
    global320_train_candidate_pairs_ranked.loc[
        train_has_positive_global320 == 1
    ]
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 3. Define features fresh
# ------------------------------------------------------------

NON_FEATURE_COLUMNS = [
    "QueryId",
    "CandidateClassId",
    "TrueClassId",
    "IsCorrectCandidate",
]

RERANKER_FEATURES_GLOBAL320 = [
    column
    for column
    in global320_hgb_train.columns
    if column not in NON_FEATURE_COLUMNS
]


# ------------------------------------------------------------
# 4. Training matrix
# ------------------------------------------------------------

X_hgb_global320_train = (
    global320_hgb_train[
        RERANKER_FEATURES_GLOBAL320
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .astype(np.float32)
)

y_hgb_global320_train = (
    global320_hgb_train[
        "IsCorrectCandidate"
    ]
    .astype(np.int8)
)


# ------------------------------------------------------------
# 5. Development matrix
# ------------------------------------------------------------

X_hgb_global320_dev = (
    global320_dev_candidate_pairs_ranked[
        RERANKER_FEATURES_GLOBAL320
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .astype(np.float32)
)


print(
    "Training rows:",
    f"{len(global320_hgb_train):,}"
)

print(
    "Training queries:",
    global320_hgb_train[
        "QueryId"
    ].nunique()
)

print(
    "Number of features:",
    len(
        RERANKER_FEATURES_GLOBAL320
    )
)

print(
    "Training matrix:",
    X_hgb_global320_train.shape
)

print(
    "Development matrix:",
    X_hgb_global320_dev.shape
)

print(
    "Positive rows:",
    int(
        y_hgb_global320_train.sum()
    )
)

Training rows: 616,000
Training queries: 1928
Number of features: 52
Training matrix: (616000, 52)
Development matrix: (159897, 52)
Positive rows: 1928


In [23]:
from sklearn.ensemble import HistGradientBoostingClassifier

# ============================================================
# Train HGB on GLOBAL TOP 320 candidates
# ============================================================

global320_hgb = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=400,
    max_leaf_nodes=7,
    min_samples_leaf=10,
    l2_regularization=1.0,

    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=20,

    random_state=RANDOM_STATE
)


# IMPORTANT:
# no sample_weight
global320_hgb.fit(
    X_hgb_global320_train,
    y_hgb_global320_train
)


print(
    "GLOBAL-320 HGB trained."
)

print(
    "Iterations used:",
    global320_hgb.n_iter_
)

GLOBAL-320 HGB trained.
Iterations used: 326


In [24]:
# ============================================================
# Evaluate GLOBAL-320 HGB
# ============================================================

global320_hgb_probabilities = (
    global320_hgb.predict_proba(
        X_hgb_global320_dev
    )[:, 1]
)


global320_hgb_scored = (
    global320_dev_candidate_pairs_ranked[
        [
            "QueryId",
            "CandidateClassId",
            "TrueClassId",
            "IsCorrectCandidate",
        ]
    ]
    .copy()
)

global320_hgb_scored[
    "CorrectProbability"
] = global320_hgb_probabilities


# ------------------------------------------------------------
# Best candidate per query
# ------------------------------------------------------------

global320_hgb_best = (
    global320_hgb_scored
    .sort_values(
        [
            "QueryId",
            "CorrectProbability",
        ],
        ascending=[
            True,
            False,
        ]
    )
    .groupby(
        "QueryId",
        as_index=False
    )
    .first()
)

global320_hgb_best[
    "PredictionCorrect"
] = (
    global320_hgb_best[
        "CandidateClassId"
    ]
    ==
    global320_hgb_best[
        "TrueClassId"
    ]
)


# ------------------------------------------------------------
# Evaluation table
# ------------------------------------------------------------

global320_hgb_eval = (
    global320_query_info[
        [
            "QueryId",
            "AnimalSeenInHistory",
            "CorrectCandidatePresent",
        ]
    ]
    .merge(
        global320_hgb_best[
            [
                "QueryId",
                "PredictionCorrect",
            ]
        ],
        on="QueryId",
        how="left",
    )
)

global320_hgb_eval[
    "PredictionCorrect"
] = (
    global320_hgb_eval[
        "PredictionCorrect"
    ]
    .fillna(False)
)


# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

global320_hgb_overall = (
    global320_hgb_eval[
        "PredictionCorrect"
    ].mean()
)

global320_hgb_seen = (
    global320_hgb_eval.loc[
        global320_hgb_eval[
            "AnimalSeenInHistory"
        ],
        "PredictionCorrect",
    ].mean()
)

global320_hgb_conditional = (
    global320_hgb_eval.loc[
        global320_hgb_eval[
            "CorrectCandidatePresent"
        ],
        "PredictionCorrect",
    ].mean()
)


print("=" * 60)

print(
    "GLOBAL-320 HGB overall accuracy:",
    f"{global320_hgb_overall:.2%}"
)

print(
    "GLOBAL-320 HGB seen-AnimalId accuracy:",
    f"{global320_hgb_seen:.2%}"
)

print(
    "GLOBAL-320 HGB conditional ranking accuracy:",
    f"{global320_hgb_conditional:.2%}"
)

print(
    "Candidate coverage ceiling:",
    f"{global320_hgb_eval['CorrectCandidatePresent'].mean():.2%}"
)

print(
    "Correct predictions:",
    int(
        global320_hgb_eval[
            "PredictionCorrect"
        ].sum()
    ),
    "/",
    len(
        global320_hgb_eval
    )
)

GLOBAL-320 HGB overall accuracy: 17.00%
GLOBAL-320 HGB seen-AnimalId accuracy: 19.63%
GLOBAL-320 HGB conditional ranking accuracy: 21.57%
Candidate coverage ceiling: 78.80%
Correct predictions: 85 / 500


In [25]:
# ============================================================
# NEXT STEP
# Check Top-K recall of the GLOBAL-320 HGB pre-ranker
#
# Question:
# Even if Top-1 accuracy is low,
# does HGB keep the true cow near the top?
# ============================================================

TOP_K_VALUES = [
    5,
    10,
    20,
    40,
    80,
    120,
    160,
]

topk_results = []


# ------------------------------------------------------------
# Rank candidates within each query by HGB probability
# ------------------------------------------------------------

global320_ranked_by_hgb = (
    global320_hgb_scored
    .sort_values(
        [
            "QueryId",
            "CorrectProbability",
        ],
        ascending=[
            True,
            False,
        ]
    )
    .copy()
)

global320_ranked_by_hgb[
    "HGBRank"
] = (
    global320_ranked_by_hgb
    .groupby("QueryId")
    .cumcount()
    + 1
)


# ============================================================
# Evaluate true-candidate recall at different Top-K values
# ============================================================

for top_k in TOP_K_VALUES:

    topk_candidates = (
        global320_ranked_by_hgb.loc[
            global320_ranked_by_hgb[
                "HGBRank"
            ]
            <= top_k
        ]
    )

    correct_present_topk = (
        topk_candidates
        .groupby("QueryId")[
            "IsCorrectCandidate"
        ]
        .max()
    )


    eval_topk = (
        global320_query_info[
            [
                "QueryId",
                "AnimalSeenInHistory",
                "CorrectCandidatePresent",
            ]
        ]
        .copy()
    )

    eval_topk[
        "CorrectInTopK"
    ] = (
        eval_topk[
            "QueryId"
        ]
        .map(
            correct_present_topk
        )
        .fillna(0)
        .astype(bool)
    )


    # --------------------------------------------------------
    # Overall recall
    # --------------------------------------------------------

    overall_topk_recall = (
        eval_topk[
            "CorrectInTopK"
        ].mean()
    )


    # --------------------------------------------------------
    # Among seen AnimalIds
    # --------------------------------------------------------

    seen_topk_recall = (
        eval_topk.loc[
            eval_topk[
                "AnimalSeenInHistory"
            ],
            "CorrectInTopK",
        ]
        .mean()
    )


    # --------------------------------------------------------
    # Among queries where correct candidate was actually
    # available in Global-320
    # --------------------------------------------------------

    conditional_topk_recall = (
        eval_topk.loc[
            eval_topk[
                "CorrectCandidatePresent"
            ],
            "CorrectInTopK",
        ]
        .mean()
    )


    topk_results.append({

        "top_k":
            top_k,

        "overall_recall":
            overall_topk_recall * 100,

        "seen_recall":
            seen_topk_recall * 100,

        "conditional_recall":
            conditional_topk_recall * 100,

        "correct_queries":
            int(
                eval_topk[
                    "CorrectInTopK"
                ].sum()
            ),
    })


topk_results_df = (
    pd.DataFrame(
        topk_results
    )
)

display(
    topk_results_df
)

,top_k,overall_recall,seen_recall,conditional_recall,correct_queries
0,5,38.6,44.572748,48.984772,193
1,10,47.8,55.196305,60.659898,239
2,20,61.0,70.438799,77.411168,305
3,40,67.2,77.598152,85.279188,336
4,80,71.6,82.678984,90.862944,358
5,120,75.2,86.836028,95.431472,376
6,160,77.0,88.914550,97.715736,385


In [26]:
# ============================================================
# Stage 2 - Step 1
#
# Create DISJOINT training queries for Stage 2.
#
# Stage 1 HGB was trained on:
#     larger_train_queries
#
# Stage 2 must use queries that Stage 1 has NEVER seen.
# ============================================================

STAGE2_QUERY_ROWS = 1000

# ------------------------------------------------------------
# All seen queries available in reranker training period
# ------------------------------------------------------------

stage2_pool = (
    reranker_train_seen
    .loc[
        ~reranker_train_seen.index.isin(
            larger_train_queries.index
        )
    ]
    .copy()
)


print(
    "Available disjoint queries:",
    f"{len(stage2_pool):,}"
)


# ------------------------------------------------------------
# Sample Stage-2 queries
# ------------------------------------------------------------

stage2_train_queries = (
    stage2_pool
    .sample(
        n=min(
            STAGE2_QUERY_ROWS,
            len(stage2_pool)
        ),
        random_state=2026,
    )
    .copy()
)


# ------------------------------------------------------------
# Safety check
# ------------------------------------------------------------

overlap_count = len(
    set(
        stage2_train_queries.index
    )
    &
    set(
        larger_train_queries.index
    )
)


print(
    "Stage 1 training queries:",
    len(larger_train_queries)
)

print(
    "Stage 2 training queries:",
    len(stage2_train_queries)
)

print(
    "Overlap between Stage 1 and Stage 2:",
    overlap_count
)

print(
    "Stage 2 date range:",
    stage2_train_queries[
        "EventDate"
    ].min(),
    "to",
    stage2_train_queries[
        "EventDate"
    ].max(),
)

Available disjoint queries: 563,349
Stage 1 training queries: 2000
Stage 2 training queries: 1000
Overlap between Stage 1 and Stage 2: 0
Stage 2 date range: 2020-07-06 00:00:00 to 2020-09-20 00:00:00


In [27]:
# ============================================================
# Stage 2 - Step 2
# Build GLOBAL-320 candidates for 1000 Stage-2 queries
# ============================================================

STAGE2_GLOBAL320_CACHE_PATH = (
    CACHE_DIR
    /
    "stage2_train_candidates_1000_global320.pkl"
)


if STAGE2_GLOBAL320_CACHE_PATH.exists():

    print(
        "Loading cached Stage-2 GLOBAL-320 candidates..."
    )

    stage2_global320_pairs = (
        pd.read_pickle(
            STAGE2_GLOBAL320_CACHE_PATH
        )
    )

else:

    print(
        "Building Stage-2 GLOBAL-320 candidates..."
    )

    stage2_global320_pairs = (
        build_candidate_dataset(
            stage2_train_queries,
            prefix="stage2_query",
            progress_every=50,
        )
    )

    stage2_global320_pairs.to_pickle(
        STAGE2_GLOBAL320_CACHE_PATH
    )

    print(
        "Saved:",
        STAGE2_GLOBAL320_CACHE_PATH
    )


# Add rank features computed across all ~320 candidates
stage2_global320_ranked = (
    add_candidate_rank_features(
        stage2_global320_pairs
    )
)


# ============================================================
# Coverage BEFORE Stage-1 filtering
# ============================================================

stage2_presence_320 = (
    stage2_global320_ranked
    .groupby("QueryId")[
        "IsCorrectCandidate"
    ]
    .max()
)

stage2_coverage_320 = (
    stage2_presence_320.sum()
    /
    len(stage2_train_queries)
)


print("=" * 60)

print(
    "Stage-2 candidate rows:",
    f"{len(stage2_global320_ranked):,}"
)

print(
    "Stage-2 queries:",
    stage2_global320_ranked[
        "QueryId"
    ].nunique()
)

print(
    "Mean candidates/query:",
    f"{stage2_global320_ranked.groupby('QueryId').size().mean():.2f}"
)

print(
    "GLOBAL-320 candidate coverage:",
    f"{stage2_coverage_320:.2%}"
)

print(
    "Queries with correct candidate:",
    int(
        stage2_presence_320.sum()
    )
)

Building Stage-2 GLOBAL-320 candidates...
stage2_query: processed 50 / 1000 queries
stage2_query: processed 100 / 1000 queries
stage2_query: processed 150 / 1000 queries
stage2_query: processed 200 / 1000 queries
stage2_query: processed 250 / 1000 queries
stage2_query: processed 300 / 1000 queries
stage2_query: processed 350 / 1000 queries
stage2_query: processed 400 / 1000 queries
stage2_query: processed 450 / 1000 queries
stage2_query: processed 500 / 1000 queries
stage2_query: processed 550 / 1000 queries
stage2_query: processed 600 / 1000 queries
stage2_query: processed 650 / 1000 queries
stage2_query: processed 700 / 1000 queries
stage2_query: processed 750 / 1000 queries
stage2_query: processed 800 / 1000 queries
stage2_query: processed 850 / 1000 queries
stage2_query: processed 900 / 1000 queries
stage2_query: processed 950 / 1000 queries
stage2_query: processed 1000 / 1000 queries
Saved: xgboost_cache/stage2_train_candidates_1000_global320.pkl
Stage-2 candidate rows: 319,520
St

In [28]:
# ============================================================
# Stage 2 - Step 3
# Stage-1 HGB pre-ranking
# Keep Top 120 candidates per query
# ============================================================

STAGE1_KEEP_TOP_K = 120


# ------------------------------------------------------------
# Prepare SAME features used by Stage-1 HGB
# ------------------------------------------------------------

X_stage2_stage1 = (
    stage2_global320_ranked[
        RERANKER_FEATURES_GLOBAL320
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .astype(np.float32)
)


# ------------------------------------------------------------
# Stage-1 HGB score
# ------------------------------------------------------------

stage2_stage1_probability = (
    global320_hgb.predict_proba(
        X_stage2_stage1
    )[:, 1]
)


stage2_scored_320 = (
    stage2_global320_ranked
    .copy()
)

stage2_scored_320[
    "Stage1Probability"
] = stage2_stage1_probability


# ------------------------------------------------------------
# Rank inside each query
# ------------------------------------------------------------

stage2_scored_320[
    "Stage1Rank"
] = (
    stage2_scored_320
    .groupby("QueryId")[
        "Stage1Probability"
    ]
    .rank(
        method="first",
        ascending=False,
    )
)


# ------------------------------------------------------------
# Keep only Top 120
# ------------------------------------------------------------

stage2_top120 = (
    stage2_scored_320.loc[
        stage2_scored_320[
            "Stage1Rank"
        ]
        <= STAGE1_KEEP_TOP_K
    ]
    .copy()
    .reset_index(drop=True)
)


# ============================================================
# IMPORTANT:
# Recompute within-query rank features
# now that each query only has Top 120 candidates.
# ============================================================

stage2_top120_ranked = (
    add_candidate_rank_features(
        stage2_top120
    )
)


# ============================================================
# Coverage AFTER Stage-1 filtering
# ============================================================

stage2_presence_top120 = (
    stage2_top120_ranked
    .groupby("QueryId")[
        "IsCorrectCandidate"
    ]
    .max()
)


stage2_top120_coverage = (
    stage2_presence_top120.sum()
    /
    len(stage2_train_queries)
)


conditional_retention = (
    stage2_presence_top120.sum()
    /
    max(
        1,
        stage2_presence_320.sum()
    )
)


print("=" * 60)

print(
    "Rows after Stage-1 Top-120:",
    f"{len(stage2_top120_ranked):,}"
)

print(
    "Queries:",
    stage2_top120_ranked[
        "QueryId"
    ].nunique()
)

print(
    "Mean candidates/query:",
    f"{stage2_top120_ranked.groupby('QueryId').size().mean():.2f}"
)

print(
    "Coverage before Stage-1 filtering:",
    f"{stage2_coverage_320:.2%}"
)

print(
    "Coverage after Stage-1 Top-120:",
    f"{stage2_top120_coverage:.2%}"
)

print(
    "Conditional retention:",
    f"{conditional_retention:.2%}"
)

print(
    "Correct-candidate queries retained:",
    int(
        stage2_presence_top120.sum()
    )
)

Rows after Stage-1 Top-120: 119,926
Queries: 1000
Mean candidates/query: 119.93
Coverage before Stage-1 filtering: 97.70%
Coverage after Stage-1 Top-120: 97.40%
Conditional retention: 99.69%
Correct-candidate queries retained: 974


In [15]:
# ============================================================
# 11. Inspect XGBRanker feature importance
# ============================================================

feature_importance = (
    pd.DataFrame(
        {
            "Feature": RERANKER_FEATURES,
            "Importance": xgb_ranker.feature_importances_,
        }
    )
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

display(feature_importance.head(20))

,Feature,Importance
0,StartDifference,0.486752
1,StartDifference_Rank,0.064569
2,LocalMean3Manhattan_Rank,0.042044
3,LocalMinManhattan_Rank,0.031061
4,CandidateStartRank,0.028158
5,LocalMean3Euclidean_Rank,0.023650
6,LocalMean5Manhattan_Rank,0.020383
7,LactationGap,0.017768
8,IsProjected,0.016886
9,QueryDaysInMilk,0.016363


In [30]:
# ============================================================
# Stage 2 - Step 4
# Prepare clean Top-120 training data
# ============================================================

# Start from Stage-1 filtered candidates
stage2_train_top120 = (
    stage2_scored_320.loc[
        stage2_scored_320["Stage1Rank"]
        <= 120
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# IMPORTANT:
# CandidateCount must now describe the Top-120 set,
# not the original Top-320 set
# ------------------------------------------------------------

stage2_train_top120["CandidateCount"] = (
    stage2_train_top120
    .groupby("QueryId")["QueryId"]
    .transform("size")
    .astype(np.int16)
)


# Stage-1 rank percentile
stage2_train_top120["Stage1RankPct"] = (
    stage2_train_top120["Stage1Rank"]
    /
    stage2_train_top120["CandidateCount"]
).astype(np.float32)


# ------------------------------------------------------------
# Recalculate within-query rank features
# based ONLY on Top-120 candidates
# ------------------------------------------------------------

stage2_train_top120_ranked = (
    add_candidate_rank_features(
        stage2_train_top120
    )
)


# ------------------------------------------------------------
# Keep only queries where true cow survived Stage 1
# ------------------------------------------------------------

stage2_has_positive = (
    stage2_train_top120_ranked
    .groupby("QueryId")["IsCorrectCandidate"]
    .transform("max")
)

stage2_model_train = (
    stage2_train_top120_ranked.loc[
        stage2_has_positive == 1
    ]
    .reset_index(drop=True)
)


# ============================================================
# Stage-2 feature list
# ============================================================

STAGE2_NON_FEATURE_COLUMNS = [
    "QueryId",
    "CandidateClassId",
    "TrueClassId",
    "IsCorrectCandidate",
]

STAGE2_FEATURES = [
    column
    for column in stage2_model_train.columns
    if column not in STAGE2_NON_FEATURE_COLUMNS
]


X_stage2_train = (
    stage2_model_train[
        STAGE2_FEATURES
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .astype(np.float32)
)

y_stage2_train = (
    stage2_model_train[
        "IsCorrectCandidate"
    ]
    .astype(np.int8)
)


print("=" * 60)

print(
    "Stage-2 training rows:",
    f"{len(stage2_model_train):,}"
)

print(
    "Stage-2 training queries:",
    stage2_model_train["QueryId"].nunique()
)

print(
    "Stage-2 features:",
    len(STAGE2_FEATURES)
)

print(
    "Positive rows:",
    int(y_stage2_train.sum())
)

print(
    "Negative rows:",
    int(
        len(y_stage2_train)
        -
        y_stage2_train.sum()
    )
)

print(
    "Stage1Probability included:",
    "Stage1Probability" in STAGE2_FEATURES
)

print(
    "Stage1Rank included:",
    "Stage1Rank" in STAGE2_FEATURES
)

print(
    "Stage1RankPct included:",
    "Stage1RankPct" in STAGE2_FEATURES
)

Stage-2 training rows: 116,806
Stage-2 training queries: 974
Stage-2 features: 55
Positive rows: 974
Negative rows: 115832
Stage1Probability included: True
Stage1Rank included: True
Stage1RankPct included: True


In [31]:
# ============================================================
# Stage 2 - Step 5
# Train second HGB reranker
# ============================================================

stage2_hgb = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=400,
    max_leaf_nodes=7,
    min_samples_leaf=10,
    l2_regularization=1.0,

    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=20,

    random_state=RANDOM_STATE,
)

# No sample weighting
stage2_hgb.fit(
    X_stage2_train,
    y_stage2_train
)

print(
    "Stage-2 HGB trained successfully."
)

print(
    "Iterations used:",
    stage2_hgb.n_iter_
)

Stage-2 HGB trained successfully.
Iterations used: 168


In [32]:
# ============================================================
# Stage 2 - Step 6
# Prepare DEV Top-120 candidates
# ============================================================

stage2_dev_320 = (
    global320_dev_candidate_pairs_ranked
    .copy()
)

# Stage-1 probabilities already calculated previously
stage2_dev_320[
    "Stage1Probability"
] = global320_hgb_probabilities


stage2_dev_320[
    "Stage1Rank"
] = (
    stage2_dev_320
    .groupby("QueryId")[
        "Stage1Probability"
    ]
    .rank(
        method="first",
        ascending=False,
    )
)


# Keep Stage-1 Top120
stage2_dev_top120 = (
    stage2_dev_320.loc[
        stage2_dev_320[
            "Stage1Rank"
        ]
        <= 120
    ]
    .copy()
    .reset_index(drop=True)
)


# Update CandidateCount
stage2_dev_top120[
    "CandidateCount"
] = (
    stage2_dev_top120
    .groupby("QueryId")["QueryId"]
    .transform("size")
    .astype(np.int16)
)


stage2_dev_top120[
    "Stage1RankPct"
] = (
    stage2_dev_top120[
        "Stage1Rank"
    ]
    /
    stage2_dev_top120[
        "CandidateCount"
    ]
).astype(np.float32)


# Recompute Top120 rank features
stage2_dev_top120_ranked = (
    add_candidate_rank_features(
        stage2_dev_top120
    )
)


# ============================================================
# Predict using Stage-2 model
# ============================================================

X_stage2_dev = (
    stage2_dev_top120_ranked[
        STAGE2_FEATURES
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .astype(np.float32)
)


stage2_probability = (
    stage2_hgb.predict_proba(
        X_stage2_dev
    )[:, 1]
)


stage2_dev_scored = (
    stage2_dev_top120_ranked[
        [
            "QueryId",
            "CandidateClassId",
            "TrueClassId",
            "IsCorrectCandidate",
        ]
    ]
    .copy()
)

stage2_dev_scored[
    "Stage2Probability"
] = stage2_probability


# ------------------------------------------------------------
# Final AnimalId prediction
# ------------------------------------------------------------

stage2_best = (
    stage2_dev_scored
    .sort_values(
        [
            "QueryId",
            "Stage2Probability",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .groupby(
        "QueryId",
        as_index=False,
    )
    .first()
)

stage2_best[
    "PredictionCorrect"
] = (
    stage2_best[
        "CandidateClassId"
    ]
    ==
    stage2_best[
        "TrueClassId"
    ]
)


# ============================================================
# Stage-1 Top120 candidate coverage
# ============================================================

stage2_dev_presence = (
    stage2_dev_top120_ranked
    .groupby("QueryId")[
        "IsCorrectCandidate"
    ]
    .max()
)


# ============================================================
# Evaluation
# ============================================================

stage2_eval = (
    global320_query_info[
        [
            "QueryId",
            "AnimalSeenInHistory",
        ]
    ]
    .copy()
)

stage2_eval[
    "CorrectCandidatePresent"
] = (
    stage2_eval[
        "QueryId"
    ]
    .map(
        stage2_dev_presence
    )
    .fillna(0)
    .astype(bool)
)


stage2_eval = (
    stage2_eval
    .merge(
        stage2_best[
            [
                "QueryId",
                "PredictionCorrect",
            ]
        ],
        on="QueryId",
        how="left",
    )
)

stage2_eval[
    "PredictionCorrect"
] = (
    stage2_eval[
        "PredictionCorrect"
    ]
    .fillna(False)
)


# ============================================================
# Metrics
# ============================================================

stage2_overall = (
    stage2_eval[
        "PredictionCorrect"
    ].mean()
)

stage2_seen = (
    stage2_eval.loc[
        stage2_eval[
            "AnimalSeenInHistory"
        ],
        "PredictionCorrect",
    ].mean()
)

stage2_conditional = (
    stage2_eval.loc[
        stage2_eval[
            "CorrectCandidatePresent"
        ],
        "PredictionCorrect",
    ].mean()
)

stage2_ceiling = (
    stage2_eval[
        "CorrectCandidatePresent"
    ].mean()
)


print("=" * 60)

print(
    "Two-stage HGB overall accuracy:",
    f"{stage2_overall:.2%}"
)

print(
    "Two-stage HGB seen-AnimalId accuracy:",
    f"{stage2_seen:.2%}"
)

print(
    "Two-stage HGB conditional ranking accuracy:",
    f"{stage2_conditional:.2%}"
)

print(
    "Stage-1 Top120 coverage ceiling:",
    f"{stage2_ceiling:.2%}"
)

print(
    "Correct predictions:",
    int(
        stage2_eval[
            "PredictionCorrect"
        ].sum()
    ),
    "/",
    len(stage2_eval)
)

Two-stage HGB overall accuracy: 18.20%
Two-stage HGB seen-AnimalId accuracy: 21.02%
Two-stage HGB conditional ranking accuracy: 24.20%
Stage-1 Top120 coverage ceiling: 75.20%
Correct predictions: 91 / 500


In [33]:
# ============================================================
# NEXT STEP
# Find the best candidate-set size
#
# Global structural candidates:
# Top 120 / 160 / 200 / 240 / 320
#
# SAME HGB settings for fair comparison
# ============================================================

CANDIDATE_K_VALUES = [
    120,
    160,
    200,
    240,
    320,
]

candidate_k_results = []


for candidate_k in CANDIDATE_K_VALUES:

    print("\n" + "=" * 70)
    print("Testing Global Top-K =", candidate_k)


    # ========================================================
    # 1. TRAIN subset
    # ========================================================

    train_k = (
        global320_train_candidate_pairs.loc[
            global320_train_candidate_pairs[
                "CandidateStartRank"
            ]
            <= candidate_k
        ]
        .copy()
        .reset_index(drop=True)
    )


    # Update CandidateCount
    train_k["CandidateCount"] = (
        train_k
        .groupby("QueryId")["QueryId"]
        .transform("size")
        .astype(np.int16)
    )


    # Recompute within-query rank features
    train_k_ranked = (
        add_candidate_rank_features(
            train_k
        )
    )


    # Keep only queries where true cow is available
    train_has_positive = (
        train_k_ranked
        .groupby("QueryId")[
            "IsCorrectCandidate"
        ]
        .transform("max")
    )

    train_k_model = (
        train_k_ranked.loc[
            train_has_positive == 1
        ]
        .reset_index(drop=True)
    )


    # ========================================================
    # 2. DEV subset
    # ========================================================

    dev_k = (
        global320_dev_candidate_pairs.loc[
            global320_dev_candidate_pairs[
                "CandidateStartRank"
            ]
            <= candidate_k
        ]
        .copy()
        .reset_index(drop=True)
    )


    dev_k["CandidateCount"] = (
        dev_k
        .groupby("QueryId")["QueryId"]
        .transform("size")
        .astype(np.int16)
    )


    dev_k_ranked = (
        add_candidate_rank_features(
            dev_k
        )
    )


    # ========================================================
    # 3. Features
    # ========================================================

    NON_FEATURE_COLUMNS_K = [
        "QueryId",
        "CandidateClassId",
        "TrueClassId",
        "IsCorrectCandidate",
    ]

    features_k = [
        column
        for column in train_k_model.columns
        if column not in NON_FEATURE_COLUMNS_K
    ]


    X_train_k = (
        train_k_model[
            features_k
        ]
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
        .astype(np.float32)
    )

    y_train_k = (
        train_k_model[
            "IsCorrectCandidate"
        ]
        .astype(np.int8)
    )


    X_dev_k = (
        dev_k_ranked[
            features_k
        ]
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
        .astype(np.float32)
    )


    # ========================================================
    # 4. Train SAME HGB
    # ========================================================

    model_k = HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_iter=400,
        max_leaf_nodes=7,
        min_samples_leaf=10,
        l2_regularization=1.0,

        early_stopping=True,
        validation_fraction=0.10,
        n_iter_no_change=20,

        random_state=RANDOM_STATE,
    )

    model_k.fit(
        X_train_k,
        y_train_k
    )


    # ========================================================
    # 5. Predict
    # ========================================================

    probabilities_k = (
        model_k.predict_proba(
            X_dev_k
        )[:, 1]
    )


    scored_k = (
        dev_k_ranked[
            [
                "QueryId",
                "CandidateClassId",
                "TrueClassId",
                "IsCorrectCandidate",
            ]
        ]
        .copy()
    )

    scored_k[
        "Probability"
    ] = probabilities_k


    best_k = (
        scored_k
        .sort_values(
            [
                "QueryId",
                "Probability",
            ],
            ascending=[
                True,
                False,
            ]
        )
        .groupby(
            "QueryId",
            as_index=False
        )
        .first()
    )

    best_k[
        "PredictionCorrect"
    ] = (
        best_k[
            "CandidateClassId"
        ]
        ==
        best_k[
            "TrueClassId"
        ]
    )


    # ========================================================
    # 6. Candidate coverage
    # ========================================================

    presence_k = (
        dev_k_ranked
        .groupby("QueryId")[
            "IsCorrectCandidate"
        ]
        .max()
    )


    eval_k = (
        global320_query_info[
            [
                "QueryId",
                "AnimalSeenInHistory",
            ]
        ]
        .copy()
    )


    eval_k[
        "CorrectCandidatePresent"
    ] = (
        eval_k[
            "QueryId"
        ]
        .map(
            presence_k
        )
        .fillna(0)
        .astype(bool)
    )


    eval_k = (
        eval_k
        .merge(
            best_k[
                [
                    "QueryId",
                    "PredictionCorrect",
                ]
            ],
            on="QueryId",
            how="left",
        )
    )


    eval_k[
        "PredictionCorrect"
    ] = (
        eval_k[
            "PredictionCorrect"
        ]
        .fillna(False)
    )


    # ========================================================
    # 7. Metrics
    # ========================================================

    coverage_k = (
        eval_k[
            "CorrectCandidatePresent"
        ].mean()
    )

    overall_k = (
        eval_k[
            "PredictionCorrect"
        ].mean()
    )

    conditional_k = (
        eval_k.loc[
            eval_k[
                "CorrectCandidatePresent"
            ],
            "PredictionCorrect",
        ]
        .mean()
    )

    seen_k = (
        eval_k.loc[
            eval_k[
                "AnimalSeenInHistory"
            ],
            "PredictionCorrect",
        ]
        .mean()
    )


    candidate_k_results.append({

        "candidate_top_k":
            candidate_k,

        "training_queries":
            train_k_model[
                "QueryId"
            ].nunique(),

        "iterations":
            model_k.n_iter_,

        "candidate_coverage":
            coverage_k * 100,

        "overall_accuracy":
            overall_k * 100,

        "seen_accuracy":
            seen_k * 100,

        "conditional_accuracy":
            conditional_k * 100,

        "correct_predictions":
            int(
                eval_k[
                    "PredictionCorrect"
                ].sum()
            ),
    })


# ============================================================
# Results
# ============================================================

candidate_k_results_df = (
    pd.DataFrame(
        candidate_k_results
    )
)

display(
    candidate_k_results_df
)


Testing Global Top-K = 120

Testing Global Top-K = 160

Testing Global Top-K = 200

Testing Global Top-K = 240

Testing Global Top-K = 320


,candidate_top_k,training_queries,iterations,candidate_coverage,overall_accuracy,seen_accuracy,conditional_accuracy,correct_predictions
0,120,1869,400,68.2,26.2,30.254042,38.416422,131
1,160,1881,361,70.4,27.0,31.177829,38.352273,135
2,200,1893,400,72.0,18.6,21.478060,25.833333,93
3,240,1902,307,73.8,18.8,21.709007,25.474255,94
4,320,1928,326,78.8,17.0,19.630485,21.573604,85


In [34]:
# ============================================================
# NEXT STEP
# Global Top160 + tolerance-band features
#
# Do NOT remove candidates.
# Turn the old hard tolerance into model features.
# ============================================================

TEST_K = 160


# ============================================================
# 1. Build Top160 TRAIN table
# ============================================================

train160 = (
    global320_train_candidate_pairs.loc[
        global320_train_candidate_pairs[
            "CandidateStartRank"
        ]
        <= TEST_K
    ]
    .copy()
    .reset_index(drop=True)
)


# ============================================================
# 2. Build Top160 DEV table
# ============================================================

dev160 = (
    global320_dev_candidate_pairs.loc[
        global320_dev_candidate_pairs[
            "CandidateStartRank"
        ]
        <= TEST_K
    ]
    .copy()
    .reset_index(drop=True)
)


# ============================================================
# 3. Check StartDifference exists
# ============================================================

assert "StartDifference" in train160.columns, (
    "StartDifference column not found."
)

assert "StartDifference" in dev160.columns, (
    "StartDifference column not found."
)


# ============================================================
# 4. Add tolerance-band features
# ============================================================

def add_tolerance_features(df):

    df = df.copy()

    df["Within7Days"] = (
        df["StartDifference"] <= 7
    ).astype(np.int8)

    df["Within14Days"] = (
        df["StartDifference"] <= 14
    ).astype(np.int8)

    df["Within21Days"] = (
        df["StartDifference"] <= 21
    ).astype(np.int8)

    df["Within30Days"] = (
        df["StartDifference"] <= 30
    ).astype(np.int8)

    df["Within45Days"] = (
        df["StartDifference"] <= 45
    ).astype(np.int8)

    df["Within60Days"] = (
        df["StartDifference"] <= 60
    ).astype(np.int8)

    # Distance beyond the old 21-day boundary
    df["DaysBeyond21"] = (
        df["StartDifference"] - 21
    ).clip(lower=0).astype(np.float32)

    return df


train160 = add_tolerance_features(
    train160
)

dev160 = add_tolerance_features(
    dev160
)


# ============================================================
# 5. Update CandidateCount
# ============================================================

train160["CandidateCount"] = (
    train160
    .groupby("QueryId")["QueryId"]
    .transform("size")
    .astype(np.int16)
)

dev160["CandidateCount"] = (
    dev160
    .groupby("QueryId")["QueryId"]
    .transform("size")
    .astype(np.int16)
)


# ============================================================
# 6. Recompute within-query ranks
# ============================================================

train160_ranked = (
    add_candidate_rank_features(
        train160
    )
)

dev160_ranked = (
    add_candidate_rank_features(
        dev160
    )
)


# ============================================================
# 7. Keep training queries with positive candidate
# ============================================================

train160_positive_available = (
    train160_ranked
    .groupby("QueryId")[
        "IsCorrectCandidate"
    ]
    .transform("max")
)

train160_model = (
    train160_ranked.loc[
        train160_positive_available == 1
    ]
    .reset_index(drop=True)
)


# ============================================================
# 8. Features
# ============================================================

NON_FEATURE_COLUMNS_160 = [
    "QueryId",
    "CandidateClassId",
    "TrueClassId",
    "IsCorrectCandidate",
]

FEATURES_160 = [
    c
    for c in train160_model.columns
    if c not in NON_FEATURE_COLUMNS_160
]


X_train160 = (
    train160_model[
        FEATURES_160
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .astype(np.float32)
)

y_train160 = (
    train160_model[
        "IsCorrectCandidate"
    ]
    .astype(np.int8)
)


X_dev160 = (
    dev160_ranked[
        FEATURES_160
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .astype(np.float32)
)


print(
    "Training queries:",
    train160_model[
        "QueryId"
    ].nunique()
)

print(
    "Features:",
    len(FEATURES_160)
)

print(
    "New tolerance features:",
    [
        x for x in FEATURES_160
        if (
            "Within" in x
            or "Beyond21" in x
        )
    ]
)

Training queries: 1881
Features: 59
New tolerance features: ['Within7Days', 'Within14Days', 'Within21Days', 'Within30Days', 'Within45Days', 'Within60Days', 'DaysBeyond21']


In [35]:
# ============================================================
# Global Top160 HGB
# WITH tolerance-band features
# ============================================================

hgb160_tolerance = (
    HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_iter=400,
        max_leaf_nodes=7,
        min_samples_leaf=10,
        l2_regularization=1.0,

        early_stopping=True,
        validation_fraction=0.10,
        n_iter_no_change=20,

        random_state=RANDOM_STATE,
    )
)


hgb160_tolerance.fit(
    X_train160,
    y_train160
)


prob160 = (
    hgb160_tolerance.predict_proba(
        X_dev160
    )[:, 1]
)


scored160 = (
    dev160_ranked[
        [
            "QueryId",
            "CandidateClassId",
            "TrueClassId",
            "IsCorrectCandidate",
        ]
    ]
    .copy()
)

scored160[
    "Probability"
] = prob160


best160 = (
    scored160
    .sort_values(
        [
            "QueryId",
            "Probability",
        ],
        ascending=[
            True,
            False,
        ]
    )
    .groupby(
        "QueryId",
        as_index=False,
    )
    .first()
)


best160[
    "PredictionCorrect"
] = (
    best160[
        "CandidateClassId"
    ]
    ==
    best160[
        "TrueClassId"
    ]
)


# ============================================================
# Coverage
# ============================================================

presence160 = (
    dev160_ranked
    .groupby("QueryId")[
        "IsCorrectCandidate"
    ]
    .max()
)


eval160 = (
    global320_query_info[
        [
            "QueryId",
            "AnimalSeenInHistory",
        ]
    ]
    .copy()
)


eval160[
    "CorrectCandidatePresent"
] = (
    eval160[
        "QueryId"
    ]
    .map(
        presence160
    )
    .fillna(0)
    .astype(bool)
)


eval160 = (
    eval160
    .merge(
        best160[
            [
                "QueryId",
                "PredictionCorrect",
            ]
        ],
        on="QueryId",
        how="left",
    )
)


eval160[
    "PredictionCorrect"
] = (
    eval160[
        "PredictionCorrect"
    ]
    .fillna(False)
)


# ============================================================
# Metrics
# ============================================================

overall160 = (
    eval160[
        "PredictionCorrect"
    ].mean()
)

seen160 = (
    eval160.loc[
        eval160[
            "AnimalSeenInHistory"
        ],
        "PredictionCorrect",
    ].mean()
)

conditional160 = (
    eval160.loc[
        eval160[
            "CorrectCandidatePresent"
        ],
        "PredictionCorrect",
    ].mean()
)

coverage160 = (
    eval160[
        "CorrectCandidatePresent"
    ].mean()
)


print("=" * 60)

print(
    "Iterations:",
    hgb160_tolerance.n_iter_
)

print(
    "Candidate coverage:",
    f"{coverage160:.2%}"
)

print(
    "Overall accuracy:",
    f"{overall160:.2%}"
)

print(
    "Seen accuracy:",
    f"{seen160:.2%}"
)

print(
    "Conditional accuracy:",
    f"{conditional160:.2%}"
)

print(
    "Correct predictions:",
    int(
        eval160[
            "PredictionCorrect"
        ].sum()
    ),
    "/",
    len(eval160)
)

Iterations: 361
Candidate coverage: 70.40%
Overall accuracy: 27.00%
Seen accuracy: 31.18%
Conditional accuracy: 38.35%
Correct predictions: 135 / 500


In [37]:
# ============================================================
# ERROR ANALYSIS
#
# Compare:
#   TRUE candidate
# vs
#   WRONG predicted Top-1 candidate
#
# Only for queries where:
#   1. true candidate is present
#   2. HGB prediction is wrong
# ============================================================


# ------------------------------------------------------------
# 1. Add probability back to full dev Top160 table
# ------------------------------------------------------------

error_analysis_table = (
    dev160_ranked
    .copy()
)

error_analysis_table[
    "HGBProbability"
] = prob160


# ------------------------------------------------------------
# 2. Identify HGB Top-1 row for each query
# ------------------------------------------------------------

predicted_rows = (
    error_analysis_table
    .sort_values(
        [
            "QueryId",
            "HGBProbability",
        ],
        ascending=[
            True,
            False,
        ]
    )
    .groupby(
        "QueryId",
        as_index=False
    )
    .first()
)


# ------------------------------------------------------------
# 3. Get TRUE candidate row
# ------------------------------------------------------------

true_rows = (
    error_analysis_table.loc[
        error_analysis_table[
            "IsCorrectCandidate"
        ] == 1
    ]
    .copy()
)


# ------------------------------------------------------------
# 4. Keep only queries where:
#    correct candidate exists
#    but prediction is wrong
# ------------------------------------------------------------

wrong_query_ids = (
    predicted_rows.loc[
        predicted_rows[
            "CandidateClassId"
        ]
        !=
        predicted_rows[
            "TrueClassId"
        ],
        "QueryId"
    ]
)


wrong_true_rows = (
    true_rows.loc[
        true_rows[
            "QueryId"
        ].isin(
            wrong_query_ids
        )
    ]
    .set_index("QueryId")
)


wrong_pred_rows = (
    predicted_rows.loc[
        predicted_rows[
            "QueryId"
        ].isin(
            wrong_query_ids
        )
    ]
    .set_index("QueryId")
)


# Only common queries
common_wrong_queries = (
    wrong_true_rows.index
    .intersection(
        wrong_pred_rows.index
    )
)

wrong_true_rows = (
    wrong_true_rows.loc[
        common_wrong_queries
    ]
)

wrong_pred_rows = (
    wrong_pred_rows.loc[
        common_wrong_queries
    ]
)


print(
    "Wrong predictions with true candidate available:",
    len(common_wrong_queries)
)

# ============================================================
# Compare numeric features:
#
# For each feature:
#   how often does TRUE candidate have a LOWER value
#   than the WRONG predicted candidate?
#
# This is especially meaningful for:
#   Distance
#   Difference
#   Rank
#   Residual absolute-error features
# ============================================================


comparison_features = []

for column in FEATURES_160:

    if column not in wrong_true_rows.columns:
        continue

    if not pd.api.types.is_numeric_dtype(
        wrong_true_rows[column]
    ):
        continue

    column_lower = column.lower()

    # Focus on features where "smaller" is generally better
    if any(
        keyword in column_lower
        for keyword in [
            "distance",
            "difference",
            "rank",
            "residual",
        ]
    ):
        comparison_features.append(
            column
        )


feature_diagnostics = []


for feature in comparison_features:

    true_values = pd.to_numeric(
        wrong_true_rows[
            feature
        ],
        errors="coerce"
    )

    pred_values = pd.to_numeric(
        wrong_pred_rows[
            feature
        ],
        errors="coerce"
    )


    valid = (
        true_values.notna()
        &
        pred_values.notna()
    )

    if valid.sum() == 0:
        continue


    true_valid = (
        true_values[
            valid
        ]
    )

    pred_valid = (
        pred_values[
            valid
        ]
    )


    # Positive means TRUE candidate has smaller value
    improvement = (
        pred_valid
        -
        true_valid
    )


    feature_diagnostics.append({

        "Feature":
            feature,

        "ValidQueries":
            int(
                valid.sum()
            ),

        "TrueBetterPct":
            (
                true_valid
                <
                pred_valid
            ).mean() * 100,

        "TrueEqualPct":
            (
                true_valid
                ==
                pred_valid
            ).mean() * 100,

        "MedianTrue":
            true_valid.median(),

        "MedianWrongPrediction":
            pred_valid.median(),

        "MedianAdvantage":
            improvement.median(),
    })


feature_diagnostics_df = (
    pd.DataFrame(
        feature_diagnostics
    )
    .sort_values(
        "TrueBetterPct",
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    feature_diagnostics_df.head(25)
)

Wrong predictions with true candidate available: 217


,Feature,ValidQueries,TrueBetterPct,TrueEqualPct,MedianTrue,MedianWrongPrediction,MedianAdvantage
0,Residual_FlowShapeRatio_Rank,217,47.926267,0.000000,0.393750,0.318750,-0.043750
1,Residual_FlowShapeRatio,217,47.926267,0.000000,0.375562,0.349860,-0.028396
2,Residual_YieldFirst2Min_Session_Rank,217,42.396313,0.000000,0.337500,0.268750,-0.043750
3,Residual_YieldFirst2Min_Session,217,42.396313,0.000000,0.607817,0.489816,-0.097963
4,Residual_Flow30_60Session_Rank,217,40.092166,3.225806,0.343750,0.250000,-0.043750
5,Residual_Flow30_60Session,217,40.092166,3.225806,0.579565,0.418575,-0.096594
6,CowPrototypeEuclidean_Rank,217,39.170507,0.000000,0.268750,0.187500,-0.081250
7,Residual_DaysInMilk_Rank,217,39.170507,27.188940,0.484375,0.493750,0.000000
8,Residual_YieldPerMinute,217,39.170507,0.000000,0.634591,0.530443,-0.150592
9,Residual_DaysInMilk,217,39.170507,27.188940,1.115819,1.082723,0.000000


In [38]:
# ============================================================
# DIAGNOSTIC
# Is our reference history too stale?
#
# Compare TRUE cow's reference availability using:
#
# OLD reference:
#     reranker_history
#
# EXPANDED reference:
#     reranker_history + reranker_train_queries
#
# IMPORTANT:
# reranker_train_queries ends BEFORE dev starts,
# so this does NOT use dev labels.
# ============================================================

# ------------------------------------------------------------
# 1. Only keep cows appearing in the 500 dev queries
#    This keeps the diagnostic lightweight.
# ------------------------------------------------------------

diagnostic_animal_ids = (
    pilot_dev_queries["AnimalId"]
    .dropna()
    .unique()
)


diagnostic_columns = [
    "AnimalId",
    "EventDate",
    "LactationNumber",
    "DaysInMilk",
]


old_reference_diag = (
    reranker_history.loc[
        reranker_history["AnimalId"].isin(
            diagnostic_animal_ids
        ),
        diagnostic_columns,
    ]
    .copy()
)


recent_reference_diag = (
    reranker_train_queries.loc[
        reranker_train_queries["AnimalId"].isin(
            diagnostic_animal_ids
        ),
        diagnostic_columns,
    ]
    .copy()
)


# ------------------------------------------------------------
# 2. Make sure dates are datetime
# ------------------------------------------------------------

old_reference_diag["EventDate"] = pd.to_datetime(
    old_reference_diag["EventDate"]
)

recent_reference_diag["EventDate"] = pd.to_datetime(
    recent_reference_diag["EventDate"]
)

dev_diag = (
    pilot_dev_queries[
        diagnostic_columns
    ]
    .copy()
)

dev_diag["EventDate"] = pd.to_datetime(
    dev_diag["EventDate"]
)


# ------------------------------------------------------------
# 3. Build lookup dictionaries
#    key = (AnimalId, LactationNumber)
# ------------------------------------------------------------

def build_reference_lookup(df):

    lookup = {}

    for key, group in df.groupby(
        [
            "AnimalId",
            "LactationNumber",
        ],
        sort=False,
    ):

        group = (
            group
            .sort_values("EventDate")
        )

        lookup[key] = {
            "dates":
                group["EventDate"]
                .to_numpy(),

            "dim":
                group["DaysInMilk"]
                .to_numpy(
                    dtype=np.float32
                ),
        }

    return lookup


old_lookup = build_reference_lookup(
    old_reference_diag
)


expanded_reference_diag = pd.concat(
    [
        old_reference_diag,
        recent_reference_diag,
    ],
    ignore_index=True,
)


expanded_lookup = build_reference_lookup(
    expanded_reference_diag
)


# ------------------------------------------------------------
# 4. For each dev query:
#
# How recently did we see the TRUE cow?
# How close in DIM was its nearest reference row?
# ------------------------------------------------------------

diagnostic_results = []


for query_index, row in dev_diag.iterrows():

    animal_id = row["AnimalId"]

    lactation = row["LactationNumber"]

    query_date = row["EventDate"]

    query_dim = float(
        row["DaysInMilk"]
    )

    key = (
        animal_id,
        lactation,
    )


    result = {
        "QueryIndex":
            query_index,

        "AnimalId":
            animal_id,

        "QueryDate":
            query_date,

        "DaysInMilk":
            query_dim,
    }


    # ========================================================
    # OLD reference
    # ========================================================

    old_data = old_lookup.get(
        key
    )

    if old_data is not None:

        old_dates = old_data[
            "dates"
        ]

        old_dims = old_data[
            "dim"
        ]

        # strictly before query
        old_mask = (
            old_dates
            <
            np.datetime64(
                query_date
            )
        )

        if old_mask.any():

            valid_dates = (
                old_dates[
                    old_mask
                ]
            )

            valid_dims = (
                old_dims[
                    old_mask
                ]
            )

            last_date = (
                valid_dates.max()
            )

            result[
                "OldDaysSinceLastSeen"
            ] = (
                query_date
                -
                pd.Timestamp(
                    last_date
                )
            ).days

            result[
                "OldMinDIMGap"
            ] = float(
                np.min(
                    np.abs(
                        valid_dims
                        -
                        query_dim
                    )
                )
            )

        else:

            result[
                "OldDaysSinceLastSeen"
            ] = np.nan

            result[
                "OldMinDIMGap"
            ] = np.nan

    else:

        result[
            "OldDaysSinceLastSeen"
        ] = np.nan

        result[
            "OldMinDIMGap"
        ] = np.nan


    # ========================================================
    # EXPANDED reference
    # ========================================================

    expanded_data = (
        expanded_lookup.get(
            key
        )
    )

    if expanded_data is not None:

        expanded_dates = (
            expanded_data[
                "dates"
            ]
        )

        expanded_dims = (
            expanded_data[
                "dim"
            ]
        )

        expanded_mask = (
            expanded_dates
            <
            np.datetime64(
                query_date
            )
        )

        if expanded_mask.any():

            valid_dates = (
                expanded_dates[
                    expanded_mask
                ]
            )

            valid_dims = (
                expanded_dims[
                    expanded_mask
                ]
            )

            last_date = (
                valid_dates.max()
            )

            result[
                "ExpandedDaysSinceLastSeen"
            ] = (
                query_date
                -
                pd.Timestamp(
                    last_date
                )
            ).days

            result[
                "ExpandedMinDIMGap"
            ] = float(
                np.min(
                    np.abs(
                        valid_dims
                        -
                        query_dim
                    )
                )
            )

        else:

            result[
                "ExpandedDaysSinceLastSeen"
            ] = np.nan

            result[
                "ExpandedMinDIMGap"
            ] = np.nan

    else:

        result[
            "ExpandedDaysSinceLastSeen"
        ] = np.nan

        result[
            "ExpandedMinDIMGap"
        ] = np.nan


    diagnostic_results.append(
        result
    )


reference_recency_df = pd.DataFrame(
    diagnostic_results
)


# ============================================================
# 5. Summary
# ============================================================

print("=" * 65)

print(
    "OLD reference median days since true cow last seen:",
    reference_recency_df[
        "OldDaysSinceLastSeen"
    ].median()
)

print(
    "EXPANDED reference median days since true cow last seen:",
    reference_recency_df[
        "ExpandedDaysSinceLastSeen"
    ].median()
)

print()

print(
    "OLD reference median minimum DIM gap:",
    reference_recency_df[
        "OldMinDIMGap"
    ].median()
)

print(
    "EXPANDED reference median minimum DIM gap:",
    reference_recency_df[
        "ExpandedMinDIMGap"
    ].median()
)

print()

for days in [
    7,
    14,
    30,
    60,
    90,
]:

    old_pct = (
        reference_recency_df[
            "OldDaysSinceLastSeen"
        ]
        <= days
    ).mean()

    expanded_pct = (
        reference_recency_df[
            "ExpandedDaysSinceLastSeen"
        ]
        <= days
    ).mean()

    print(
        f"True cow seen within {days:>2} days | "
        f"OLD: {old_pct:.2%} | "
        f"EXPANDED: {expanded_pct:.2%}"
    )

OLD reference median days since true cow last seen: 123.0
EXPANDED reference median days since true cow last seen: 46.0

OLD reference median minimum DIM gap: 123.0
EXPANDED reference median minimum DIM gap: 46.0

True cow seen within  7 days | OLD: 0.00% | EXPANDED: 10.60%
True cow seen within 14 days | OLD: 0.00% | EXPANDED: 14.00%
True cow seen within 30 days | OLD: 0.00% | EXPANDED: 21.80%
True cow seen within 60 days | OLD: 0.00% | EXPANDED: 59.60%
True cow seen within 90 days | OLD: 9.60% | EXPANDED: 84.60%


In [39]:
# ============================================================
# FRESH REFERENCE SYSTEM - STEP 1
#
# Create leakage-safe chronological reference snapshots.
#
# TRAIN:
#   history + early 70% of train period
#   -> later 30% used as reranker training queries
#
# DEV:
#   history + ALL train period
#   -> existing dev queries
# ============================================================


# ------------------------------------------------------------
# 1. Make sure EventDate is datetime
# ------------------------------------------------------------

fresh_train_period = (
    reranker_train_queries
    .copy()
)

fresh_train_period["EventDate"] = pd.to_datetime(
    fresh_train_period["EventDate"]
)


# ------------------------------------------------------------
# 2. Split by DATE, not random rows
# ------------------------------------------------------------

unique_train_dates = np.sort(
    fresh_train_period[
        "EventDate"
    ]
    .dt
    .normalize()
    .unique()
)


cutoff_position = int(
    len(unique_train_dates) * 0.70
)

fresh_reference_cutoff = pd.Timestamp(
    unique_train_dates[
        cutoff_position - 1
    ]
)


print(
    "Fresh reference cutoff:",
    fresh_reference_cutoff
)


# ============================================================
# 3. TRAIN reference
# ============================================================

early_train_reference_rows = (
    fresh_train_period.loc[
        fresh_train_period[
            "EventDate"
        ]
        .dt
        .normalize()
        <= fresh_reference_cutoff
    ]
    .copy()
)


fresh_train_reference = (
    pd.concat(
        [
            reranker_history,
            early_train_reference_rows,
        ],
        ignore_index=True,
    )
)


# ============================================================
# 4. TRAIN queries
#
# These occur strictly AFTER the reference cutoff.
# ============================================================

fresh_model_train_queries = (
    fresh_train_period.loc[
        fresh_train_period[
            "EventDate"
        ]
        .dt
        .normalize()
        >
        fresh_reference_cutoff
    ]
    .copy()
)


# ============================================================
# 5. DEV reference
#
# All labeled training-period rows are available before dev.
# ============================================================

fresh_dev_reference = (
    pd.concat(
        [
            reranker_history,
            fresh_train_period,
        ],
        ignore_index=True,
    )
)


# ============================================================
# 6. Basic diagnostics
# ============================================================

print("\n" + "=" * 65)

print("FRESH TRAIN REFERENCE")

print(
    "Rows:",
    f"{len(fresh_train_reference):,}"
)

print(
    "Date range:",
    fresh_train_reference[
        "EventDate"
    ].min(),
    "→",
    fresh_train_reference[
        "EventDate"
    ].max()
)

print(
    "AnimalIds:",
    fresh_train_reference[
        "AnimalId"
    ].nunique()
)


print("\n" + "=" * 65)

print("FRESH MODEL TRAIN QUERIES")

print(
    "Rows:",
    f"{len(fresh_model_train_queries):,}"
)

print(
    "Date range:",
    fresh_model_train_queries[
        "EventDate"
    ].min(),
    "→",
    fresh_model_train_queries[
        "EventDate"
    ].max()
)

print(
    "AnimalIds:",
    fresh_model_train_queries[
        "AnimalId"
    ].nunique()
)


print("\n" + "=" * 65)

print("FRESH DEV REFERENCE")

print(
    "Rows:",
    f"{len(fresh_dev_reference):,}"
)

print(
    "Date range:",
    fresh_dev_reference[
        "EventDate"
    ].min(),
    "→",
    fresh_dev_reference[
        "EventDate"
    ].max()
)

print(
    "AnimalIds:",
    fresh_dev_reference[
        "AnimalId"
    ].nunique()
)


# ============================================================
# 7. AnimalId coverage
# ============================================================

fresh_train_reference_ids = set(
    fresh_train_reference[
        "AnimalId"
    ]
    .dropna()
)

fresh_dev_reference_ids = set(
    fresh_dev_reference[
        "AnimalId"
    ]
    .dropna()
)


fresh_train_seen_rate = (
    fresh_model_train_queries[
        "AnimalId"
    ]
    .isin(
        fresh_train_reference_ids
    )
    .mean()
)


fresh_dev_seen_rate = (
    pilot_dev_queries[
        "AnimalId"
    ]
    .isin(
        fresh_dev_reference_ids
    )
    .mean()
)


print("\n" + "=" * 65)

print(
    "TRAIN-query AnimalId seen in fresh TRAIN reference:",
    f"{fresh_train_seen_rate:.2%}"
)

print(
    "DEV-query AnimalId seen in fresh DEV reference:",
    f"{fresh_dev_seen_rate:.2%}"
)

Fresh reference cutoff: 2020-08-27 00:00:00

FRESH TRAIN REFERENCE
Rows: 3,198,518
Date range: 2019-06-14 00:00:00 → 2020-08-27 00:00:00
AnimalIds: 6928

FRESH MODEL TRAIN QUERIES
Rows: 181,365
Date range: 2020-08-28 00:00:00 → 2020-09-20 00:00:00
AnimalIds: 4466

FRESH DEV REFERENCE
Rows: 3,379,883
Date range: 2019-06-14 00:00:00 → 2020-09-20 00:00:00
AnimalIds: 7111

TRAIN-query AnimalId seen in fresh TRAIN reference: 97.80%
DEV-query AnimalId seen in fresh DEV reference: 95.20%


In [40]:
# ============================================================
# NEXT DIAGNOSTIC
# Compare reference freshness:
#
# Fresh TRAIN queries vs DEV queries
#
# We only need the LAST known record for each
# (AnimalId, LactationNumber).
# ============================================================


def build_last_seen_table(reference_df):

    temp = (
        reference_df[
            [
                "AnimalId",
                "LactationNumber",
                "EventDate",
            ]
        ]
        .dropna(
            subset=[
                "AnimalId",
                "LactationNumber",
                "EventDate",
            ]
        )
        .copy()
    )

    temp["EventDate"] = pd.to_datetime(
        temp["EventDate"]
    )

    last_seen = (
        temp
        .groupby(
            [
                "AnimalId",
                "LactationNumber",
            ],
            as_index=False,
        )["EventDate"]
        .max()
        .rename(
            columns={
                "EventDate":
                    "LastReferenceDate"
            }
        )
    )

    return last_seen


# ============================================================
# 1. Build TRAIN and DEV last-seen lookup
# ============================================================

fresh_train_last_seen = (
    build_last_seen_table(
        fresh_train_reference
    )
)

fresh_dev_last_seen = (
    build_last_seen_table(
        fresh_dev_reference
    )
)


# ============================================================
# 2. Use a manageable TRAIN sample
# ============================================================

FRESHNESS_TRAIN_SAMPLE = 5000

freshness_train_queries = (
    fresh_model_train_queries
    .sample(
        n=min(
            FRESHNESS_TRAIN_SAMPLE,
            len(fresh_model_train_queries)
        ),
        random_state=2026,
    )
    [
        [
            "AnimalId",
            "LactationNumber",
            "EventDate",
        ]
    ]
    .copy()
)

freshness_dev_queries = (
    pilot_dev_queries[
        [
            "AnimalId",
            "LactationNumber",
            "EventDate",
        ]
    ]
    .copy()
)


freshness_train_queries["EventDate"] = (
    pd.to_datetime(
        freshness_train_queries[
            "EventDate"
        ]
    )
)

freshness_dev_queries["EventDate"] = (
    pd.to_datetime(
        freshness_dev_queries[
            "EventDate"
        ]
    )
)


# ============================================================
# 3. Join TRUE cow's most recent reference record
# ============================================================

train_freshness = (
    freshness_train_queries
    .merge(
        fresh_train_last_seen,
        on=[
            "AnimalId",
            "LactationNumber",
        ],
        how="left",
    )
)

dev_freshness = (
    freshness_dev_queries
    .merge(
        fresh_dev_last_seen,
        on=[
            "AnimalId",
            "LactationNumber",
        ],
        how="left",
    )
)


# ============================================================
# 4. Days since TRUE cow was last seen
# ============================================================

train_freshness[
    "DaysSinceLastSeen"
] = (
    train_freshness[
        "EventDate"
    ]
    -
    train_freshness[
        "LastReferenceDate"
    ]
).dt.days


dev_freshness[
    "DaysSinceLastSeen"
] = (
    dev_freshness[
        "EventDate"
    ]
    -
    dev_freshness[
        "LastReferenceDate"
    ]
).dt.days


# Safety:
# Reference must never be after query
assert (
    train_freshness[
        "DaysSinceLastSeen"
    ]
    .dropna()
    >= 0
).all()

assert (
    dev_freshness[
        "DaysSinceLastSeen"
    ]
    .dropna()
    >= 0
).all()


# ============================================================
# 5. Summary
# ============================================================

print("=" * 65)

print(
    "TRAIN median days since true cow last seen:",
    train_freshness[
        "DaysSinceLastSeen"
    ].median()
)

print(
    "DEV median days since true cow last seen:",
    dev_freshness[
        "DaysSinceLastSeen"
    ].median()
)

print()

print(
    "TRAIN mean days since true cow last seen:",
    train_freshness[
        "DaysSinceLastSeen"
    ].mean()
)

print(
    "DEV mean days since true cow last seen:",
    dev_freshness[
        "DaysSinceLastSeen"
    ].mean()
)

print()


for days in [
    7,
    14,
    30,
    60,
    90,
]:

    train_pct = (
        train_freshness[
            "DaysSinceLastSeen"
        ]
        <= days
    ).mean()

    dev_pct = (
        dev_freshness[
            "DaysSinceLastSeen"
        ]
        <= days
    ).mean()

    print(
        f"True cow seen within {days:>2} days | "
        f"TRAIN: {train_pct:.2%} | "
        f"DEV: {dev_pct:.2%}"
    )


print()
print(
    "TRAIN missing same-lactation reference:",
    train_freshness[
        "LastReferenceDate"
    ].isna().mean()
)

print(
    "DEV missing same-lactation reference:",
    dev_freshness[
        "LastReferenceDate"
    ].isna().mean()
)

TRAIN median days since true cow last seen: 14.0
DEV median days since true cow last seen: 46.0

TRAIN mean days since true cow last seen: 14.039781098716059
DEV mean days since true cow last seen: 45.178823529411765

True cow seen within  7 days | TRAIN: 21.14% | DEV: 10.60%
True cow seen within 14 days | TRAIN: 47.80% | DEV: 14.00%
True cow seen within 30 days | TRAIN: 94.90% | DEV: 21.80%
True cow seen within 60 days | TRAIN: 94.96% | DEV: 59.60%
True cow seen within 90 days | TRAIN: 94.98% | DEV: 84.60%

TRAIN missing same-lactation reference: 0.0498
DEV missing same-lactation reference: 0.15


In [41]:
# ============================================================
# NEXT STEP
# Find a TRAIN reference cutoff whose freshness
# distribution is similar to DEV.
# ============================================================

TRAIN_REFERENCE_CUTOFFS = [
    "2020-07-10",
    "2020-07-20",
    "2020-07-31",
    "2020-08-10",
    "2020-08-20",
]


# Use the same 5,000 training queries as before
train_query_sample = (
    fresh_model_train_queries
    .sample(
        n=min(
            5000,
            len(fresh_model_train_queries)
        ),
        random_state=2026,
    )
    [
        [
            "AnimalId",
            "LactationNumber",
            "EventDate",
        ]
    ]
    .copy()
)

train_query_sample["EventDate"] = pd.to_datetime(
    train_query_sample["EventDate"]
)


cutoff_results = []


for cutoff_string in TRAIN_REFERENCE_CUTOFFS:

    cutoff = pd.Timestamp(
        cutoff_string
    )

    print(
        "Testing cutoff:",
        cutoff.date()
    )


    # --------------------------------------------------------
    # Build reference available by this date
    # --------------------------------------------------------

    cutoff_reference = pd.concat(
        [
            reranker_history,

            fresh_train_period.loc[
                fresh_train_period["EventDate"]
                <= cutoff
            ],
        ],
        ignore_index=True,
    )


    # --------------------------------------------------------
    # Last known record for each cow/lactation
    # --------------------------------------------------------

    cutoff_last_seen = (
        cutoff_reference[
            [
                "AnimalId",
                "LactationNumber",
                "EventDate",
            ]
        ]
        .dropna()
        .groupby(
            [
                "AnimalId",
                "LactationNumber",
            ],
            as_index=False,
        )["EventDate"]
        .max()
        .rename(
            columns={
                "EventDate":
                    "LastReferenceDate"
            }
        )
    )


    # --------------------------------------------------------
    # Match to training queries
    # --------------------------------------------------------

    temp = (
        train_query_sample
        .merge(
            cutoff_last_seen,
            on=[
                "AnimalId",
                "LactationNumber",
            ],
            how="left",
        )
    )


    temp["DaysSinceLastSeen"] = (
        temp["EventDate"]
        -
        temp["LastReferenceDate"]
    ).dt.days


    # Safety check
    valid_days = (
        temp["DaysSinceLastSeen"]
        .dropna()
    )

    assert (
        valid_days >= 0
    ).all()


    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    cutoff_results.append({

        "reference_cutoff":
            cutoff.date(),

        "median_days":
            temp[
                "DaysSinceLastSeen"
            ].median(),

        "mean_days":
            temp[
                "DaysSinceLastSeen"
            ].mean(),

        "within_14d":
            (
                temp[
                    "DaysSinceLastSeen"
                ]
                <= 14
            ).mean() * 100,

        "within_30d":
            (
                temp[
                    "DaysSinceLastSeen"
                ]
                <= 30
            ).mean() * 100,

        "within_60d":
            (
                temp[
                    "DaysSinceLastSeen"
                ]
                <= 60
            ).mean() * 100,

        "within_90d":
            (
                temp[
                    "DaysSinceLastSeen"
                ]
                <= 90
            ).mean() * 100,

        "missing_same_lactation":
            temp[
                "LastReferenceDate"
            ].isna().mean() * 100,
    })


cutoff_results_df = (
    pd.DataFrame(
        cutoff_results
    )
)

display(
    cutoff_results_df
)


print("\nDEV TARGET:")

print(
    "Median days: 46"
)

print(
    "Mean days: 45.18"
)

print(
    "Within 14d: 14.0%"
)

print(
    "Within 30d: 21.8%"
)

print(
    "Within 60d: 59.6%"
)

print(
    "Within 90d: 84.6%"
)

print(
    "Missing same lactation: 15.0%"
)

Testing cutoff: 2020-07-10
Testing cutoff: 2020-07-20
Testing cutoff: 2020-07-31
Testing cutoff: 2020-08-10
Testing cutoff: 2020-08-20


,reference_cutoff,median_days,mean_days,within_14d,within_30d,within_60d,within_90d,missing_same_lactation
0,2020-07-10,62.0,61.878281,0.00,0.00,32.52,77.50,22.28
1,2020-07-20,52.0,51.911967,0.00,0.00,72.82,81.36,18.44
2,2020-07-31,41.0,40.876054,0.00,5.98,85.06,85.24,14.64
3,2020-08-10,32.0,31.338066,0.00,39.54,88.52,88.62,11.26
4,2020-08-20,21.0,20.990302,21.14,86.08,92.70,92.76,7.20



DEV TARGET:
Median days: 46
Mean days: 45.18
Within 14d: 14.0%
Within 30d: 21.8%
Within 60d: 59.6%
Within 90d: 84.6%
Missing same lactation: 15.0%


In [42]:
# ============================================================
# MULTI-SNAPSHOT DIAGNOSTIC
#
# Build query-snapshot pairs and check whether we have enough
# examples in each freshness group to mimic DEV.
# ============================================================

SNAPSHOT_DATES = [
    "2020-07-10",
    "2020-07-20",
    "2020-07-31",
    "2020-08-10",
    "2020-08-20",
]

# Use a larger but still manageable query sample
SNAPSHOT_QUERY_SAMPLE = 20000

snapshot_query_sample = (
    fresh_model_train_queries
    .sample(
        n=min(
            SNAPSHOT_QUERY_SAMPLE,
            len(fresh_model_train_queries)
        ),
        random_state=2026,
    )
    [
        [
            "AnimalId",
            "LactationNumber",
            "EventDate",
        ]
    ]
    .copy()
)

snapshot_query_sample["EventDate"] = pd.to_datetime(
    snapshot_query_sample["EventDate"]
)

# Preserve original query identity
snapshot_query_sample["OriginalQueryIndex"] = (
    snapshot_query_sample.index
)


# ============================================================
# Freshness bin function
# ============================================================

def assign_freshness_bin(days):

    if pd.isna(days):
        return "MissingSameLactation"

    if days <= 14:
        return "00_0_14"

    if days <= 30:
        return "01_15_30"

    if days <= 60:
        return "02_31_60"

    if days <= 90:
        return "03_61_90"

    return "04_Over90"


# ============================================================
# Build query-snapshot pairs
# ============================================================

snapshot_pair_tables = []


for snapshot_string in SNAPSHOT_DATES:

    snapshot_date = pd.Timestamp(
        snapshot_string
    )

    # --------------------------------------------------------
    # Reference available at this snapshot
    # --------------------------------------------------------

    snapshot_reference = pd.concat(
        [
            reranker_history,

            fresh_train_period.loc[
                fresh_train_period["EventDate"]
                <= snapshot_date
            ],
        ],
        ignore_index=True,
    )


    # --------------------------------------------------------
    # Last seen for each AnimalId + LactationNumber
    # --------------------------------------------------------

    snapshot_last_seen = (
        snapshot_reference[
            [
                "AnimalId",
                "LactationNumber",
                "EventDate",
            ]
        ]
        .dropna()
        .groupby(
            [
                "AnimalId",
                "LactationNumber",
            ],
            as_index=False,
        )["EventDate"]
        .max()
        .rename(
            columns={
                "EventDate":
                    "LastReferenceDate"
            }
        )
    )


    temp = (
        snapshot_query_sample
        .merge(
            snapshot_last_seen,
            on=[
                "AnimalId",
                "LactationNumber",
            ],
            how="left",
        )
    )


    # Reference snapshot must be before query
    temp = (
        temp.loc[
            temp["EventDate"]
            >
            snapshot_date
        ]
        .copy()
    )


    temp["SnapshotDate"] = (
        snapshot_date
    )


    temp["DaysSinceLastSeen"] = (
        temp["EventDate"]
        -
        temp["LastReferenceDate"]
    ).dt.days


    temp["FreshnessBin"] = (
        temp["DaysSinceLastSeen"]
        .apply(
            assign_freshness_bin
        )
    )


    snapshot_pair_tables.append(
        temp
    )


snapshot_training_pool = (
    pd.concat(
        snapshot_pair_tables,
        ignore_index=True,
    )
)


# ============================================================
# Count available examples
# ============================================================

freshness_order = [
    "00_0_14",
    "01_15_30",
    "02_31_60",
    "03_61_90",
    "04_Over90",
    "MissingSameLactation",
]


pool_counts = (
    snapshot_training_pool[
        "FreshnessBin"
    ]
    .value_counts()
    .reindex(
        freshness_order,
        fill_value=0,
    )
)


pool_percent = (
    pool_counts
    /
    len(snapshot_training_pool)
    * 100
)


snapshot_pool_summary = pd.DataFrame(
    {
        "AvailablePairs":
            pool_counts,

        "PoolPercent":
            pool_percent,

        "DEV_TargetPercent":
            [
                14.0,
                7.8,
                37.8,
                25.0,
                0.4,
                15.0,
            ],

        # If we later build 2,000 training query-snapshot pairs
        "NeededFor2000":
            [
                280,
                156,
                756,
                500,
                8,
                300,
            ],
    }
)


display(
    snapshot_pool_summary
)


print(
    "Total query-snapshot pairs:",
    f"{len(snapshot_training_pool):,}"
)

print(
    "Unique original queries:",
    snapshot_training_pool[
        "OriginalQueryIndex"
    ].nunique()
)

,AvailablePairs,PoolPercent,DEV_TargetPercent,NeededFor2000
FreshnessBin,,,,
00_0_14,4173,4.173,14.0,280
01_15_30,22024,22.024,7.8,156
02_31_60,48243,48.243,37.8,756
03_61_90,10909,10.909,25.0,500
04_Over90,113,0.113,0.4,8
MissingSameLactation,14538,14.538,15.0,300


Total query-snapshot pairs: 100,000
Unique original queries: 20000


In [44]:
# ============================================================
# MULTI-SNAPSHOT TRAINING SET
# STEP 2
#
# Sample exactly 2,000 query-snapshot pairs
# matching the DEV freshness distribution.
#
# IMPORTANT:
# Each original query can appear only once.
# ============================================================

TARGET_BIN_COUNTS = {
    "00_0_14": 280,
    "01_15_30": 156,
    "02_31_60": 756,
    "03_61_90": 500,
    "04_Over90": 8,
    "MissingSameLactation": 300,
}


# ------------------------------------------------------------
# Sample rarer / harder bins first
# ------------------------------------------------------------

SAMPLING_ORDER = [
    "04_Over90",
    "00_0_14",
    "03_61_90",
    "MissingSameLactation",
    "01_15_30",
    "02_31_60",
]


selected_parts = []

used_original_queries = set()


for i, freshness_bin in enumerate(
    SAMPLING_ORDER
):

    target_n = TARGET_BIN_COUNTS[
        freshness_bin
    ]

    # --------------------------------------------------------
    # Candidate pairs for this bin
    # excluding queries already selected
    # --------------------------------------------------------

    available = (
        snapshot_training_pool.loc[
            (
                snapshot_training_pool[
                    "FreshnessBin"
                ]
                == freshness_bin
            )
            &
            (
                ~snapshot_training_pool[
                    "OriginalQueryIndex"
                ].isin(
                    used_original_queries
                )
            )
        ]
        .copy()
    )


    # --------------------------------------------------------
    # A query may appear under more than one snapshot
    # within the same freshness bin.
    #
    # Randomly shuffle, then keep one snapshot per query.
    # --------------------------------------------------------

    available = (
        available
        .sample(
            frac=1,
            random_state=2026 + i,
        )
        .drop_duplicates(
            subset=[
                "OriginalQueryIndex"
            ],
            keep="first",
        )
    )


    print(
        freshness_bin,
        "| needed:",
        target_n,
        "| available unique queries:",
        len(available)
    )


    if len(available) < target_n:

        raise ValueError(
            f"Not enough unique queries for "
            f"{freshness_bin}: "
            f"need {target_n}, "
            f"have {len(available)}"
        )


    selected = (
        available
        .sample(
            n=target_n,
            random_state=3000 + i,
        )
        .copy()
    )


    selected_parts.append(
        selected
    )


    used_original_queries.update(
        selected[
            "OriginalQueryIndex"
        ].tolist()
    )


# ============================================================
# Combine selected pairs
# ============================================================

matched_snapshot_train = (
    pd.concat(
        selected_parts,
        ignore_index=True,
    )
    .sample(
        frac=1,
        random_state=2026,
    )
    .reset_index(drop=True)
)


# ============================================================
# Safety checks
# ============================================================

assert (
    len(matched_snapshot_train)
    == 2000
)

assert (
    matched_snapshot_train[
        "OriginalQueryIndex"
    ].nunique()
    == 2000
)

assert (
    matched_snapshot_train[
        "EventDate"
    ]
    >
    matched_snapshot_train[
        "SnapshotDate"
    ]
).all()


# ============================================================
# Distribution check
# ============================================================

matched_bin_summary = (
    matched_snapshot_train[
        "FreshnessBin"
    ]
    .value_counts()
    .reindex(
        freshness_order
    )
    .rename(
        "Count"
    )
    .to_frame()
)

matched_bin_summary[
    "Percent"
] = (
    matched_bin_summary[
        "Count"
    ]
    /
    len(
        matched_snapshot_train
    )
    * 100
)


print("\n" + "=" * 65)

print(
    "Total matched training pairs:",
    len(
        matched_snapshot_train
    )
)

print(
    "Unique original queries:",
    matched_snapshot_train[
        "OriginalQueryIndex"
    ].nunique()
)

display(
    matched_bin_summary
)


# ============================================================
# Snapshot-date distribution
# ============================================================

snapshot_distribution = (
    matched_snapshot_train[
        "SnapshotDate"
    ]
    .value_counts()
    .sort_index()
    .rename(
        "Count"
    )
    .to_frame()
)

snapshot_distribution[
    "Percent"
] = (
    snapshot_distribution[
        "Count"
    ]
    /
    len(
        matched_snapshot_train
    )
    * 100
)


print(
    "\nSnapshot distribution:"
)

display(
    snapshot_distribution
)


# ============================================================
# Freshness statistics
# ============================================================

print(
    "Matched TRAIN median days since last seen:",
    matched_snapshot_train[
        "DaysSinceLastSeen"
    ].median()
)

print(
    "Matched TRAIN mean days since last seen:",
    matched_snapshot_train[
        "DaysSinceLastSeen"
    ].mean()
)

print(
    "Matched TRAIN missing same-lactation:",
    matched_snapshot_train[
        "LastReferenceDate"
    ].isna().mean()
)

04_Over90 | needed: 8 | available unique queries: 40
00_0_14 | needed: 280 | available unique queries: 4173
03_61_90 | needed: 500 | available unique queries: 9209
MissingSameLactation | needed: 300 | available unique queries: 4300
01_15_30 | needed: 156 | available unique queries: 16133
02_31_60 | needed: 756 | available unique queries: 16409

Total matched training pairs: 2000
Unique original queries: 2000


,Count,Percent
FreshnessBin,,
00_0_14,280,14.0
01_15_30,156,7.8
02_31_60,756,37.8
03_61_90,500,25.0
04_Over90,8,0.4
MissingSameLactation,300,15.0



Snapshot distribution:


,Count,Percent
SnapshotDate,,
2020-07-10,681,34.05
2020-07-20,344,17.20
2020-07-31,310,15.50
2020-08-10,222,11.10
2020-08-20,443,22.15


Matched TRAIN median days since last seen: 46.0
Matched TRAIN mean days since last seen: 43.69764705882353
Matched TRAIN missing same-lactation: 0.15


In [45]:
# ============================================================
# MULTI-SNAPSHOT REFERENCE SYSTEM
# STEP 3
#
# Create:
#   1. Full query rows for the matched 2,000 training queries
#   2. Five independent reference datasets
#   3. Query groups assigned to each snapshot
#
# NO feature construction yet.
# ============================================================


# ============================================================
# 1. Recover the FULL original query rows
# ============================================================

matched_query_metadata = (
    matched_snapshot_train[
        [
            "OriginalQueryIndex",
            "SnapshotDate",
            "FreshnessBin",
            "DaysSinceLastSeen",
        ]
    ]
    .copy()
)


matched_full_train_queries = (
    fresh_model_train_queries
    .loc[
        matched_query_metadata[
            "OriginalQueryIndex"
        ]
        .tolist()
    ]
    .copy()
)


# Preserve original index as a normal column
matched_full_train_queries[
    "OriginalQueryIndex"
] = matched_full_train_queries.index


# Add snapshot assignment back
matched_full_train_queries = (
    matched_full_train_queries
    .merge(
        matched_query_metadata,
        on="OriginalQueryIndex",
        how="left",
        validate="one_to_one",
    )
)


# ============================================================
# 2. Basic safety checks
# ============================================================

assert len(
    matched_full_train_queries
) == 2000

assert (
    matched_full_train_queries[
        "OriginalQueryIndex"
    ].nunique()
    == 2000
)

assert (
    matched_full_train_queries[
        "SnapshotDate"
    ].notna()
).all()

assert (
    pd.to_datetime(
        matched_full_train_queries[
            "EventDate"
        ]
    )
    >
    pd.to_datetime(
        matched_full_train_queries[
            "SnapshotDate"
        ]
    )
).all()


# ============================================================
# 3. Build the FIVE snapshot reference datasets
#
# Each reference contains:
#   reranker_history
#   +
#   train-period rows up to SnapshotDate
# ============================================================

snapshot_reference_sets = {}

snapshot_query_groups = {}


for snapshot_string in SNAPSHOT_DATES:

    snapshot_date = pd.Timestamp(
        snapshot_string
    )


    # --------------------------------------------------------
    # Reference rows available by snapshot date
    # --------------------------------------------------------

    snapshot_reference = (
        pd.concat(
            [
                reranker_history,

                fresh_train_period.loc[
                    fresh_train_period[
                        "EventDate"
                    ]
                    <= snapshot_date
                ],
            ],
            ignore_index=True,
        )
        .copy()
    )


    # --------------------------------------------------------
    # Queries assigned to this snapshot
    # --------------------------------------------------------

    snapshot_queries = (
        matched_full_train_queries.loc[
            matched_full_train_queries[
                "SnapshotDate"
            ]
            == snapshot_date
        ]
        .copy()
    )


    snapshot_reference_sets[
        snapshot_date
    ] = snapshot_reference

    snapshot_query_groups[
        snapshot_date
    ] = snapshot_queries


# ============================================================
# 4. Summary table
# ============================================================

snapshot_system_summary = []


for snapshot_date in sorted(
    snapshot_reference_sets.keys()
):

    reference_df = (
        snapshot_reference_sets[
            snapshot_date
        ]
    )

    query_df = (
        snapshot_query_groups[
            snapshot_date
        ]
    )


    reference_ids = set(
        reference_df[
            "AnimalId"
        ]
        .dropna()
    )


    query_seen_rate = (
        query_df[
            "AnimalId"
        ]
        .isin(
            reference_ids
        )
        .mean()
    )


    # Safety:
    # reference must not extend beyond snapshot
    reference_max_date = pd.to_datetime(
        reference_df[
            "EventDate"
        ]
    ).max()


    assert (
        reference_max_date
        <= snapshot_date
    )


    # Safety:
    # all assigned queries must occur AFTER snapshot
    assert (
        pd.to_datetime(
            query_df[
                "EventDate"
            ]
        )
        >
        snapshot_date
    ).all()


    snapshot_system_summary.append({

        "SnapshotDate":
            snapshot_date.date(),

        "ReferenceRows":
            len(
                reference_df
            ),

        "ReferenceAnimalIds":
            reference_df[
                "AnimalId"
            ].nunique(),

        "AssignedQueries":
            len(
                query_df
            ),

        "QueryAnimalSeenRate":
            query_seen_rate * 100,

        "ReferenceMaxDate":
            reference_max_date.date(),
    })


snapshot_system_summary_df = (
    pd.DataFrame(
        snapshot_system_summary
    )
)


display(
    snapshot_system_summary_df
)


print(
    "Total assigned queries:",
    sum(
        len(x)
        for x
        in snapshot_query_groups.values()
    )
)

print(
    "Unique assigned queries:",
    matched_full_train_queries[
        "OriginalQueryIndex"
    ].nunique()
)

,SnapshotDate,ReferenceRows,ReferenceAnimalIds,AssignedQueries,QueryAnimalSeenRate,ReferenceMaxDate
0,2020-07-10,2830363,6690,681,94.126285,2020-07-10
1,2020-07-20,2914658,6746,344,92.732558,2020-07-20
2,2020-07-31,3002472,6810,310,95.161290,2020-07-31
3,2020-08-10,3080351,6855,222,95.945946,2020-08-10
4,2020-08-20,3149078,6903,443,96.839729,2020-08-20


Total assigned queries: 2000
Unique assigned queries: 2000


In [46]:
# ============================================================
# MULTI-SNAPSHOT REFERENCE SYSTEM
# STEP 4
#
# Build ONE reference system first:
#     2020-07-10
#
# We are only testing that the fresh-reference machinery works.
# DO NOT build candidates yet.
# ============================================================

TEST_SNAPSHOT_DATE = pd.Timestamp(
    "2020-07-10"
)

test_snapshot_reference = (
    snapshot_reference_sets[
        TEST_SNAPSHOT_DATE
    ]
)

test_snapshot_queries = (
    snapshot_query_groups[
        TEST_SNAPSHOT_DATE
    ]
)


print(
    "Reference rows:",
    f"{len(test_snapshot_reference):,}"
)

print(
    "Assigned queries:",
    len(test_snapshot_queries)
)


# ============================================================
# 1. AnimalId class mapping
# ============================================================

snapshot_class_values = pd.Index(
    test_snapshot_reference[
        "AnimalId"
    ]
    .astype(object)
    .unique()
)

snapshot_class_to_number = {
    animal_id: class_number
    for class_number, animal_id
    in enumerate(
        snapshot_class_values
    )
}

snapshot_number_to_class = {
    class_number: animal_id
    for class_number, animal_id
    in enumerate(
        snapshot_class_values
    )
}


# ============================================================
# 2. Origin date
# ============================================================

snapshot_origin_date = (
    test_snapshot_reference[
        "EventDate"
    ]
    .min()
)


# ============================================================
# 3. Structural profiles
# ============================================================

(
    snapshot_profiles,
    snapshot_global_lactation_interval,
) = build_structural_profiles(
    test_snapshot_reference,
    snapshot_class_to_number,
    snapshot_origin_date,
    include_projection=True,
)

snapshot_profile_lookup = (
    make_profile_lookup(
        snapshot_profiles
    )
)


# ============================================================
# 4. Sample reference rows
#
# Same strategy as previous model.
# ============================================================

snapshot_reference_indices = (
    sample_reference_rows(
        test_snapshot_reference,
        max_rows_per_group=
            REFERENCE_ROWS_PER_GROUP,
        recent_fraction=0.80,
        random_state=RANDOM_STATE,
    )
)


snapshot_reference_raw = (
    test_snapshot_reference
    .loc[
        snapshot_reference_indices
    ]
    .copy()
)


# ============================================================
# 5. Distance preprocessing
# ============================================================

(
    X_snapshot_reference,
    snapshot_numeric_columns,
    snapshot_numeric_fill,
    snapshot_scaler,
    snapshot_reproduction_columns,
    snapshot_feature_names,
) = fit_distance_preprocessor(
    snapshot_reference_raw
)


# ============================================================
# 6. Reference metadata
# ============================================================

snapshot_reference_class_ids = (
    snapshot_reference_raw[
        "AnimalId"
    ]
    .astype(object)
    .map(
        snapshot_class_to_number
    )
    .astype(np.int32)
    .to_numpy()
)


snapshot_reference_lactations = (
    snapshot_reference_raw[
        "LactationNumber"
    ]
    .round()
    .astype(np.int16)
    .to_numpy()
)


snapshot_reference_dim = (
    snapshot_reference_raw[
        "DaysInMilk"
    ]
    .astype(np.float32)
    .to_numpy()
)


snapshot_reference_event_day = (
    event_day_from_origin(
        snapshot_reference_raw,
        snapshot_origin_date,
    )
    .to_numpy()
)


# ============================================================
# 7. Exact AnimalId-lactation index
# ============================================================

(
    snapshot_reference_exact_index,
    snapshot_reference_class_lactations,
) = build_reference_index(
    snapshot_reference_class_ids,
    snapshot_reference_lactations,
)


# ============================================================
# 8. Whole-cow prototypes
# ============================================================

snapshot_cow_prototype = {}


for class_id in np.unique(
    snapshot_reference_class_ids
):

    positions = np.flatnonzero(
        snapshot_reference_class_ids
        == class_id
    )

    snapshot_cow_prototype[
        int(class_id)
    ] = np.median(
        X_snapshot_reference[
            positions
        ],
        axis=0,
    ).astype(np.float32)


# ============================================================
# 9. Lactation + DIM-bin prototypes
# ============================================================

snapshot_reference_dim_bin = np.floor(
    snapshot_reference_dim
    /
    DIM_BIN_WIDTH
).astype(np.int16)


snapshot_lactation_prototype = {}


for (
    class_id,
    lactation_number
), positions in (
    snapshot_reference_exact_index.items()
):

    bins_here = (
        snapshot_reference_dim_bin[
            positions
        ]
    )

    prototype_by_bin = {}


    for dim_bin in np.unique(
        bins_here
    ):

        bin_positions = (
            positions[
                bins_here
                == dim_bin
            ]
        )


        prototype_by_bin[
            int(dim_bin)
        ] = np.median(
            X_snapshot_reference[
                bin_positions
            ],
            axis=0,
        ).astype(np.float32)


    snapshot_lactation_prototype[
        (
            int(class_id),
            int(lactation_number),
        )
    ] = prototype_by_bin


# ============================================================
# 10. Diagnostics
# ============================================================

query_seen_rate = (
    test_snapshot_queries[
        "AnimalId"
    ]
    .astype(object)
    .isin(
        snapshot_class_values
    )
    .mean()
)


print("\n" + "=" * 65)

print(
    "Snapshot:",
    TEST_SNAPSHOT_DATE.date()
)

print(
    "AnimalId classes:",
    f"{len(snapshot_class_values):,}"
)

print(
    "Structural profiles:",
    f"{len(snapshot_profiles):,}"
)

print(
    "Reference rows sampled:",
    f"{len(snapshot_reference_raw):,}"
)

print(
    "Reference matrix:",
    X_snapshot_reference.shape
)

print(
    "Distance features:",
    len(snapshot_feature_names)
)

print(
    "AnimalId-lactation groups:",
    f"{len(snapshot_reference_exact_index):,}"
)

print(
    "Whole-cow prototypes:",
    f"{len(snapshot_cow_prototype):,}"
)

print(
    "Lactation/DIM prototype groups:",
    f"{len(snapshot_lactation_prototype):,}"
)

print(
    "Assigned query AnimalId seen:",
    f"{query_seen_rate:.2%}"
)

print(
    "Estimated reference-matrix memory:",
    f"{X_snapshot_reference.nbytes / 1024**2:.2f} MB"
)

Reference rows: 2,830,363
Assigned queries: 681

Snapshot: 2020-07-10
AnimalId classes: 6,690
Structural profiles: 16,495
Reference rows sampled: 1,802,025
Reference matrix: (1802025, 13)
Distance features: 13
AnimalId-lactation groups: 9,807
Whole-cow prototypes: 6,690
Lactation/DIM prototype groups: 9,807
Assigned query AnimalId seen: 94.13%
Estimated reference-matrix memory: 89.36 MB


In [49]:
# ============================================================
# CHECK GLOBAL VARIABLES USED BY CURRENT CANDIDATE FUNCTION
# ============================================================

candidate_function_names = (
    build_candidate_features_for_one_query
    .__code__
    .co_names
)

reranker_dependencies = sorted(
    {
        name
        for name in candidate_function_names
        if (
            name.startswith("reranker_")
            or name.startswith("RERANKER_")
        )
    }
)

print(
    "Reranker globals used by "
    "build_candidate_features_for_one_query:"
)

for name in reranker_dependencies:

    print(
        f"{name:45s}",
        "DEFINED"
        if name in globals()
        else "MISSING"
    )


print(
    "\nTotal reranker dependencies:",
    len(reranker_dependencies)
)

# ============================================================
# Also inspect important helper functions
# ============================================================

functions_to_check = [
    build_candidate_features_for_one_query,
    get_candidate_profiles,
    choose_reference_lactation,
]

for func in functions_to_check:

    print("\n" + "=" * 70)
    print("FUNCTION:", func.__name__)

    names = sorted(
        {
            name
            for name in func.__code__.co_names
            if (
                name.startswith("reranker_")
                or name.startswith("RERANKER_")
            )
        }
    )

    for name in names:
        print(name)

Reranker globals used by build_candidate_features_for_one_query:
RERANKER_CANDIDATE_PARAMS                     DEFINED
reranker_class_to_number                      DEFINED
reranker_cow_prototype                        DEFINED
reranker_feature_names                        DEFINED
reranker_lactation_prototype                  DEFINED
reranker_numeric_columns                      DEFINED
reranker_numeric_fill                         DEFINED
reranker_origin_date                          DEFINED
reranker_profile_lookup                       DEFINED
reranker_reference_class_lactations           DEFINED
reranker_reference_dim                        DEFINED
reranker_reference_exact_index                DEFINED
reranker_reproduction_columns                 DEFINED
reranker_scaler                               DEFINED

Total reranker dependencies: 14

FUNCTION: build_candidate_features_for_one_query
RERANKER_CANDIDATE_PARAMS
reranker_class_to_number
reranker_cow_prototype
reranker_feature_names

In [50]:
# ============================================================
# FINAL DEPENDENCY AUDIT
#
# Find ALL notebook-level globals used directly or indirectly
# by build_candidate_features_for_one_query()
#
# This catches names such as:
#     X_reranker_reference
# which were missed by the previous prefix-only check.
# ============================================================

import inspect
import types


def collect_notebook_dependencies(
    func,
    visited=None,
    level=0,
    max_depth=4,
):

    if visited is None:
        visited = set()

    if func in visited:
        return {}

    visited.add(func)

    dependencies = {}

    try:
        closure = inspect.getclosurevars(
            func
        )
    except Exception:
        return dependencies


    # --------------------------------------------------------
    # Direct globals
    # --------------------------------------------------------

    for name, obj in closure.globals.items():

        dependencies[name] = obj


    # --------------------------------------------------------
    # Recursively inspect notebook-defined helper functions
    # --------------------------------------------------------

    if level < max_depth:

        for name, obj in closure.globals.items():

            if (
                inspect.isfunction(obj)
                and obj.__module__ == "__main__"
            ):

                child_dependencies = (
                    collect_notebook_dependencies(
                        obj,
                        visited=visited,
                        level=level + 1,
                        max_depth=max_depth,
                    )
                )

                dependencies.update(
                    child_dependencies
                )


    return dependencies


all_candidate_dependencies = (
    collect_notebook_dependencies(
        build_candidate_features_for_one_query
    )
)


# ============================================================
# Show DATA-LIKE globals only
#
# Ignore imported modules and helper functions.
# ============================================================

data_dependencies = {}


for name, obj in all_candidate_dependencies.items():

    if inspect.ismodule(obj):
        continue

    if inspect.isfunction(obj):
        continue

    if inspect.isclass(obj):
        continue

    data_dependencies[name] = obj


print(
    "DATA-LIKE GLOBAL DEPENDENCIES"
)

print("=" * 70)


for name in sorted(
    data_dependencies
):

    obj = data_dependencies[
        name
    ]

    obj_type = type(
        obj
    ).__name__


    # Try to show shape / size
    extra = ""

    if hasattr(
        obj,
        "shape"
    ):

        try:
            extra = (
                f" shape={obj.shape}"
            )
        except Exception:
            pass

    elif isinstance(
        obj,
        dict
    ):

        extra = (
            f" len={len(obj):,}"
        )

    elif isinstance(
        obj,
        (list, tuple, set)
    ):

        extra = (
            f" len={len(obj):,}"
        )


    print(
        f"{name:45s} "
        f"{obj_type:20s}"
        f"{extra}"
    )


print(
    "\nTotal data-like globals:",
    len(data_dependencies)
)

DATA-LIKE GLOBAL DEPENDENCIES
BASE_NUMERIC_FEATURES                         list                 len=6
DIM_BIN_WIDTH                                 int                 
RERANKER_CANDIDATE_PARAMS                     dict                 len=4
X_reranker_reference                          ndarray              shape=(1783366, 13)
reranker_class_to_number                      dict                 len=6,667
reranker_cow_prototype                        dict                 len=6,667
reranker_feature_names                        list                 len=13
reranker_lactation_prototype                  dict                 len=9,731
reranker_numeric_columns                      list                 len=9
reranker_numeric_fill                         Series               shape=(9,)
reranker_origin_date                          Timestamp           
reranker_profile_lookup                       dict                 len=10
reranker_reference_class_lactations           dict                 len=6,

In [51]:
# ============================================================
# MULTI-SNAPSHOT REFERENCE SYSTEM
# STEP 5
#
# Generate candidates for ONLY the 2020-07-10 snapshot group
# using the FRESH reference system.
#
# Globals are temporarily replaced and then restored.
# ============================================================

FRESH_TEST_PREFIX = "fresh0710_query"


# ============================================================
# 1. Map old reranker globals -> fresh snapshot objects
# ============================================================

fresh_snapshot_globals = {

    "reranker_class_to_number":
        snapshot_class_to_number,

    "reranker_cow_prototype":
        snapshot_cow_prototype,

    "reranker_feature_names":
        snapshot_feature_names,

    "reranker_lactation_prototype":
        snapshot_lactation_prototype,

    "reranker_numeric_columns":
        snapshot_numeric_columns,

    "reranker_numeric_fill":
        snapshot_numeric_fill,

    "reranker_origin_date":
        snapshot_origin_date,

    "reranker_profile_lookup":
        snapshot_profile_lookup,

    "reranker_reference_class_lactations":
        snapshot_reference_class_lactations,

    "reranker_reference_dim":
        snapshot_reference_dim,

    "reranker_reference_exact_index":
        snapshot_reference_exact_index,

    "reranker_reproduction_columns":
        snapshot_reproduction_columns,

    "reranker_scaler":
        snapshot_scaler,

    "X_reranker_reference":
        X_snapshot_reference,
}


# ============================================================
# 2. Backup current OLD globals
# ============================================================

old_reranker_globals = {
    name: globals()[name]
    for name in fresh_snapshot_globals
}


# ============================================================
# 3. Temporarily activate fresh snapshot
# ============================================================

try:

    for name, value in fresh_snapshot_globals.items():
        globals()[name] = value


    print("Fresh snapshot temporarily activated.")
    print(
        "Snapshot:",
        TEST_SNAPSHOT_DATE.date()
    )

    print(
        "Candidate max:",
        RERANKER_CANDIDATE_PARAMS[
            "max_candidates"
        ]
    )

    print(
        "Reference matrix now:",
        X_reranker_reference.shape
    )

    print(
        "Class count now:",
        len(
            reranker_class_to_number
        )
    )


    # ========================================================
    # 4. Generate candidate features
    # ========================================================

    fresh0710_candidate_pairs = (
        build_candidate_dataset(
            test_snapshot_queries,
            prefix=FRESH_TEST_PREFIX,
            progress_every=50,
        )
    )


finally:

    # ========================================================
    # 5. ALWAYS restore original globals
    # ========================================================

    for name, value in old_reranker_globals.items():
        globals()[name] = value

    print(
        "\nOriginal reranker globals restored."
    )

Fresh snapshot temporarily activated.
Snapshot: 2020-07-10
Candidate max: 320
Reference matrix now: (1802025, 13)
Class count now: 6690
fresh0710_query: processed 50 / 681 queries
fresh0710_query: processed 100 / 681 queries
fresh0710_query: processed 150 / 681 queries
fresh0710_query: processed 200 / 681 queries
fresh0710_query: processed 250 / 681 queries
fresh0710_query: processed 300 / 681 queries
fresh0710_query: processed 350 / 681 queries
fresh0710_query: processed 400 / 681 queries
fresh0710_query: processed 450 / 681 queries
fresh0710_query: processed 500 / 681 queries
fresh0710_query: processed 550 / 681 queries
fresh0710_query: processed 600 / 681 queries
fresh0710_query: processed 650 / 681 queries
fresh0710_query: processed 681 / 681 queries

Original reranker globals restored.


In [52]:
# ============================================================
# STEP 6
# Evaluate FRESH 2020-07-10 candidate coverage
# ============================================================

fresh0710_presence = (
    fresh0710_candidate_pairs
    .groupby("QueryId")[
        "IsCorrectCandidate"
    ]
    .max()
)


# Query IDs generated by build_candidate_dataset
fresh0710_query_ids = [
    f"{FRESH_TEST_PREFIX}_{idx}"
    for idx in test_snapshot_queries.index
]


fresh0710_query_eval = (
    test_snapshot_queries[
        ["AnimalId"]
    ]
    .copy()
)

fresh0710_query_eval[
    "QueryId"
] = fresh0710_query_ids


# ------------------------------------------------------------
# True AnimalId seen in THIS snapshot reference?
# ------------------------------------------------------------

fresh0710_query_eval[
    "AnimalSeenInSnapshot"
] = (
    fresh0710_query_eval[
        "AnimalId"
    ]
    .astype(object)
    .isin(
        snapshot_class_values
    )
)


# ------------------------------------------------------------
# Correct candidate present?
# ------------------------------------------------------------

fresh0710_query_eval[
    "CorrectCandidatePresent"
] = (
    fresh0710_query_eval[
        "QueryId"
    ]
    .map(
        fresh0710_presence
    )
    .fillna(0)
    .astype(bool)
)


# ============================================================
# Metrics
# ============================================================

fresh0710_overall_coverage = (
    fresh0710_query_eval[
        "CorrectCandidatePresent"
    ].mean()
)


fresh0710_seen_coverage = (
    fresh0710_query_eval.loc[
        fresh0710_query_eval[
            "AnimalSeenInSnapshot"
        ],
        "CorrectCandidatePresent",
    ]
    .mean()
)


fresh0710_candidate_counts = (
    fresh0710_candidate_pairs
    .groupby("QueryId")
    .size()
)


print("=" * 65)

print(
    "Queries:",
    len(
        fresh0710_query_eval
    )
)

print(
    "Candidate rows:",
    f"{len(fresh0710_candidate_pairs):,}"
)

print(
    "Mean candidates/query:",
    f"{fresh0710_candidate_counts.mean():.2f}"
)

print(
    "AnimalId seen rate:",
    f"{fresh0710_query_eval['AnimalSeenInSnapshot'].mean():.2%}"
)

print(
    "Overall candidate coverage:",
    f"{fresh0710_overall_coverage:.2%}"
)

print(
    "Seen-AnimalId candidate coverage:",
    f"{fresh0710_seen_coverage:.2%}"
)

print(
    "Correct queries:",
    int(
        fresh0710_query_eval[
            "CorrectCandidatePresent"
        ].sum()
    ),
    "/",
    len(
        fresh0710_query_eval
    )
)

Queries: 681
Candidate rows: 217,352
Mean candidates/query: 319.17
AnimalId seen rate: 94.13%
Overall candidate coverage: 89.28%
Seen-AnimalId candidate coverage: 94.85%
Correct queries: 608 / 681


In [53]:
# ============================================================
# PAIRED TEST
#
# SAME 681 queries:
#
# OLD fixed reference
# vs
# FRESH 2020-07-10 snapshot reference
#
# Current globals should already be restored to OLD system.
# ============================================================

OLD_TEST_PREFIX = "old0710_query"


print(
    "Current OLD reference matrix:",
    X_reranker_reference.shape
)

print(
    "Current OLD class count:",
    len(reranker_class_to_number)
)


# ============================================================
# 1. Generate OLD-reference candidates
# ============================================================

old0710_candidate_pairs = (
    build_candidate_dataset(
        test_snapshot_queries,
        prefix=OLD_TEST_PREFIX,
        progress_every=50,
    )
)


# ============================================================
# 2. OLD candidate presence
# ============================================================

old0710_presence = (
    old0710_candidate_pairs
    .groupby("QueryId")[
        "IsCorrectCandidate"
    ]
    .max()
)


# ============================================================
# 3. Build a paired query-level comparison
# ============================================================

paired0710 = (
    test_snapshot_queries[
        ["AnimalId"]
    ]
    .copy()
)


# OLD class availability
paired0710[
    "OldAnimalSeen"
] = (
    paired0710[
        "AnimalId"
    ]
    .astype(object)
    .isin(
        reranker_class_to_number
    )
)


# FRESH class availability
paired0710[
    "FreshAnimalSeen"
] = (
    paired0710[
        "AnimalId"
    ]
    .astype(object)
    .isin(
        snapshot_class_values
    )
)


# ------------------------------------------------------------
# Candidate presence
# ------------------------------------------------------------

paired0710[
    "OldCorrectPresent"
] = [
    bool(
        old0710_presence.get(
            f"{OLD_TEST_PREFIX}_{idx}",
            0,
        )
    )
    for idx in paired0710.index
]


paired0710[
    "FreshCorrectPresent"
] = [
    bool(
        fresh0710_presence.get(
            f"{FRESH_TEST_PREFIX}_{idx}",
            0,
        )
    )
    for idx in paired0710.index
]


# ============================================================
# 4. Metrics
# ============================================================

old_overall = (
    paired0710[
        "OldCorrectPresent"
    ].mean()
)

fresh_overall = (
    paired0710[
        "FreshCorrectPresent"
    ].mean()
)


old_seen_coverage = (
    paired0710.loc[
        paired0710[
            "OldAnimalSeen"
        ],
        "OldCorrectPresent",
    ]
    .mean()
)


fresh_seen_coverage = (
    paired0710.loc[
        paired0710[
            "FreshAnimalSeen"
        ],
        "FreshCorrectPresent",
    ]
    .mean()
)


# ============================================================
# 5. Paired transitions
# ============================================================

fresh_recovered = (
    (~paired0710["OldCorrectPresent"])
    &
    (paired0710["FreshCorrectPresent"])
).sum()


fresh_lost = (
    (paired0710["OldCorrectPresent"])
    &
    (~paired0710["FreshCorrectPresent"])
).sum()


both_correct = (
    paired0710["OldCorrectPresent"]
    &
    paired0710["FreshCorrectPresent"]
).sum()


both_missing = (
    (~paired0710["OldCorrectPresent"])
    &
    (~paired0710["FreshCorrectPresent"])
).sum()


# ============================================================
# 6. Results
# ============================================================

print("\n" + "=" * 65)

print(
    "OLD overall coverage:",
    f"{old_overall:.2%}"
)

print(
    "FRESH overall coverage:",
    f"{fresh_overall:.2%}"
)

print(
    "Coverage gain:",
    f"{(fresh_overall - old_overall) * 100:.2f} pp"
)

print()

print(
    "OLD seen-AnimalId coverage:",
    f"{old_seen_coverage:.2%}"
)

print(
    "FRESH seen-AnimalId coverage:",
    f"{fresh_seen_coverage:.2%}"
)

print()

print(
    "Recovered by FRESH:",
    int(fresh_recovered)
)

print(
    "Lost after FRESH:",
    int(fresh_lost)
)

print(
    "Correct in both:",
    int(both_correct)
)

print(
    "Missing in both:",
    int(both_missing)
)

Current OLD reference matrix: (1783366, 13)
Current OLD class count: 6667
old0710_query: processed 50 / 681 queries
old0710_query: processed 100 / 681 queries
old0710_query: processed 150 / 681 queries
old0710_query: processed 200 / 681 queries
old0710_query: processed 250 / 681 queries
old0710_query: processed 300 / 681 queries
old0710_query: processed 350 / 681 queries
old0710_query: processed 400 / 681 queries
old0710_query: processed 450 / 681 queries
old0710_query: processed 500 / 681 queries
old0710_query: processed 550 / 681 queries
old0710_query: processed 600 / 681 queries
old0710_query: processed 650 / 681 queries
old0710_query: processed 681 / 681 queries

OLD overall coverage: 88.11%
FRESH overall coverage: 89.28%
Coverage gain: 1.17 pp

OLD seen-AnimalId coverage: 94.49%
FRESH seen-AnimalId coverage: 94.85%

Recovered by FRESH: 8
Lost after FRESH: 0
Correct in both: 600
Missing in both: 73


In [54]:
# ============================================================
# MULTI-SNAPSHOT REFERENCE SYSTEM
# STEP 7
#
# Build the REMAINING FOUR fresh snapshots sequentially:
#
#   2020-07-20
#   2020-07-31
#   2020-08-10
#   2020-08-20
#
# For each snapshot:
#   1. Build fresh reference system
#   2. Temporarily activate fresh globals
#   3. Build candidate features
#   4. Save candidates to disk
#   5. Restore old globals
#   6. Delete large objects from memory
# ============================================================

import gc


# ============================================================
# 0. Save the already-built 2020-07-10 candidates
# ============================================================

fresh0710_cache_path = (
    CACHE_DIR
    /
    "fresh_snapshot_candidates_20200710.pkl"
)

if not fresh0710_cache_path.exists():

    fresh0710_candidate_pairs.to_pickle(
        fresh0710_cache_path
    )

    print(
        "Saved existing 2020-07-10 candidates:",
        fresh0710_cache_path
    )

else:

    print(
        "2020-07-10 cache already exists."
    )


# ============================================================
# 1. Remaining snapshots
# ============================================================

REMAINING_SNAPSHOT_DATES = [
    pd.Timestamp("2020-07-20"),
    pd.Timestamp("2020-07-31"),
    pd.Timestamp("2020-08-10"),
    pd.Timestamp("2020-08-20"),
]


snapshot_candidate_summaries = []


# ============================================================
# 2. Backup OLD reranker globals ONCE
# ============================================================

REFERENCE_GLOBAL_NAMES = [
    "reranker_class_to_number",
    "reranker_cow_prototype",
    "reranker_feature_names",
    "reranker_lactation_prototype",
    "reranker_numeric_columns",
    "reranker_numeric_fill",
    "reranker_origin_date",
    "reranker_profile_lookup",
    "reranker_reference_class_lactations",
    "reranker_reference_dim",
    "reranker_reference_exact_index",
    "reranker_reproduction_columns",
    "reranker_scaler",
    "X_reranker_reference",
]


original_reranker_globals = {
    name: globals()[name]
    for name in REFERENCE_GLOBAL_NAMES
}


# ============================================================
# 3. Process snapshots ONE AT A TIME
# ============================================================

for snapshot_date in REMAINING_SNAPSHOT_DATES:

    print("\n" + "=" * 75)
    print(
        "BUILDING SNAPSHOT:",
        snapshot_date.date()
    )
    print("=" * 75)


    snapshot_tag = (
        snapshot_date.strftime(
            "%Y%m%d"
        )
    )

    prefix = (
        "fresh"
        +
        snapshot_date.strftime(
            "%m%d"
        )
        +
        "_query"
    )

    cache_path = (
        CACHE_DIR
        /
        f"fresh_snapshot_candidates_{snapshot_tag}.pkl"
    )


    # --------------------------------------------------------
    # Reference and assigned queries
    # --------------------------------------------------------

    reference_df = (
        snapshot_reference_sets[
            snapshot_date
        ]
    )

    query_df = (
        snapshot_query_groups[
            snapshot_date
        ]
    )


    print(
        "Reference rows:",
        f"{len(reference_df):,}"
    )

    print(
        "Assigned queries:",
        len(query_df)
    )


    # ========================================================
    # 4. If candidate cache already exists, just load it
    # ========================================================

    if cache_path.exists():

        print(
            "Loading cached candidates..."
        )

        candidate_pairs = (
            pd.read_pickle(
                cache_path
            )
        )


    else:

        # ====================================================
        # 5. Class mapping
        # ====================================================

        class_values = pd.Index(
            reference_df[
                "AnimalId"
            ]
            .astype(object)
            .unique()
        )


        class_to_number = {
            animal_id: class_number
            for class_number, animal_id
            in enumerate(
                class_values
            )
        }


        # ====================================================
        # 6. Origin date
        # ====================================================

        origin_date = (
            reference_df[
                "EventDate"
            ]
            .min()
        )


        # ====================================================
        # 7. Structural profiles
        # ====================================================

        (
            structural_profiles,
            global_lactation_interval,
        ) = build_structural_profiles(
            reference_df,
            class_to_number,
            origin_date,
            include_projection=True,
        )


        profile_lookup = (
            make_profile_lookup(
                structural_profiles
            )
        )


        print(
            "Structural profiles:",
            f"{len(structural_profiles):,}"
        )


        # ====================================================
        # 8. Sample reference rows
        # ====================================================

        reference_indices = (
            sample_reference_rows(
                reference_df,
                max_rows_per_group=
                    REFERENCE_ROWS_PER_GROUP,
                recent_fraction=0.80,
                random_state=RANDOM_STATE,
            )
        )


        reference_raw = (
            reference_df
            .loc[
                reference_indices
            ]
            .copy()
        )


        print(
            "Sampled reference rows:",
            f"{len(reference_raw):,}"
        )


        # ====================================================
        # 9. Fit distance preprocessing
        # ====================================================

        (
            X_reference,
            numeric_columns,
            numeric_fill,
            scaler,
            reproduction_columns,
            feature_names,
        ) = fit_distance_preprocessor(
            reference_raw
        )


        print(
            "Reference matrix:",
            X_reference.shape
        )


        # ====================================================
        # 10. Reference metadata
        # ====================================================

        reference_class_ids = (
            reference_raw[
                "AnimalId"
            ]
            .astype(object)
            .map(
                class_to_number
            )
            .astype(np.int32)
            .to_numpy()
        )


        reference_lactations = (
            reference_raw[
                "LactationNumber"
            ]
            .round()
            .astype(np.int16)
            .to_numpy()
        )


        reference_dim = (
            reference_raw[
                "DaysInMilk"
            ]
            .astype(np.float32)
            .to_numpy()
        )


        # ====================================================
        # 11. Exact AnimalId-lactation index
        # ====================================================

        (
            reference_exact_index,
            reference_class_lactations,
        ) = build_reference_index(
            reference_class_ids,
            reference_lactations,
        )


        # ====================================================
        # 12. Whole-cow prototypes
        # ====================================================

        cow_prototype = {}


        for class_id in np.unique(
            reference_class_ids
        ):

            positions = np.flatnonzero(
                reference_class_ids
                == class_id
            )

            cow_prototype[
                int(class_id)
            ] = np.median(
                X_reference[
                    positions
                ],
                axis=0,
            ).astype(np.float32)


        # ====================================================
        # 13. Lactation + DIM-bin prototypes
        # ====================================================

        reference_dim_bin = np.floor(
            reference_dim
            /
            DIM_BIN_WIDTH
        ).astype(np.int16)


        lactation_prototype = {}


        for (
            class_id,
            lactation_number
        ), positions in (
            reference_exact_index.items()
        ):

            bins_here = (
                reference_dim_bin[
                    positions
                ]
            )

            prototype_by_bin = {}


            for dim_bin in np.unique(
                bins_here
            ):

                bin_positions = (
                    positions[
                        bins_here
                        == dim_bin
                    ]
                )


                prototype_by_bin[
                    int(dim_bin)
                ] = np.median(
                    X_reference[
                        bin_positions
                    ],
                    axis=0,
                ).astype(np.float32)


            lactation_prototype[
                (
                    int(class_id),
                    int(lactation_number),
                )
            ] = prototype_by_bin


        # ====================================================
        # 14. Fresh snapshot global mapping
        # ====================================================

        fresh_globals = {

            "reranker_class_to_number":
                class_to_number,

            "reranker_cow_prototype":
                cow_prototype,

            "reranker_feature_names":
                feature_names,

            "reranker_lactation_prototype":
                lactation_prototype,

            "reranker_numeric_columns":
                numeric_columns,

            "reranker_numeric_fill":
                numeric_fill,

            "reranker_origin_date":
                origin_date,

            "reranker_profile_lookup":
                profile_lookup,

            "reranker_reference_class_lactations":
                reference_class_lactations,

            "reranker_reference_dim":
                reference_dim,

            "reranker_reference_exact_index":
                reference_exact_index,

            "reranker_reproduction_columns":
                reproduction_columns,

            "reranker_scaler":
                scaler,

            "X_reranker_reference":
                X_reference,
        }


        # ====================================================
        # 15. Temporarily activate snapshot
        # ====================================================

        try:

            for name, value in (
                fresh_globals.items()
            ):

                globals()[name] = value


            print(
                "Fresh globals activated."
            )

            print(
                "Classes:",
                len(
                    reranker_class_to_number
                )
            )

            print(
                "Reference matrix:",
                X_reranker_reference.shape
            )


            # ================================================
            # Build candidate features
            # ================================================

            candidate_pairs = (
                build_candidate_dataset(
                    query_df,
                    prefix=prefix,
                    progress_every=50,
                )
            )


        finally:

            # ================================================
            # ALWAYS restore original globals
            # ================================================

            for name, value in (
                original_reranker_globals.items()
            ):

                globals()[name] = value


            print(
                "Original globals restored."
            )


        # ====================================================
        # 16. Save candidate table
        # ====================================================

        candidate_pairs.to_pickle(
            cache_path
        )


        print(
            "Candidate cache saved:",
            cache_path
        )


    # ========================================================
    # 17. Coverage diagnostics
    # ========================================================

    candidate_presence = (
        candidate_pairs
        .groupby("QueryId")[
            "IsCorrectCandidate"
        ]
        .max()
    )


    expected_query_ids = [
        f"{prefix}_{idx}"
        for idx in query_df.index
    ]


    query_eval = (
        query_df[
            ["AnimalId"]
        ]
        .copy()
    )


    query_eval[
        "QueryId"
    ] = expected_query_ids


    reference_animal_ids = set(
        reference_df[
            "AnimalId"
        ]
        .dropna()
        .astype(object)
    )


    query_eval[
        "AnimalSeenInSnapshot"
    ] = (
        query_eval[
            "AnimalId"
        ]
        .astype(object)
        .isin(
            reference_animal_ids
        )
    )


    query_eval[
        "CorrectCandidatePresent"
    ] = (
        query_eval[
            "QueryId"
        ]
        .map(
            candidate_presence
        )
        .fillna(0)
        .astype(bool)
    )


    overall_coverage = (
        query_eval[
            "CorrectCandidatePresent"
        ]
        .mean()
    )


    seen_coverage = (
        query_eval.loc[
            query_eval[
                "AnimalSeenInSnapshot"
            ],
            "CorrectCandidatePresent",
        ]
        .mean()
    )


    candidate_counts = (
        candidate_pairs
        .groupby("QueryId")
        .size()
    )


    summary_row = {

        "SnapshotDate":
            snapshot_date.date(),

        "Queries":
            len(query_df),

        "CandidateRows":
            len(candidate_pairs),

        "MeanCandidates":
            candidate_counts.mean(),

        "AnimalSeenRate":
            query_eval[
                "AnimalSeenInSnapshot"
            ].mean() * 100,

        "OverallCoverage":
            overall_coverage * 100,

        "SeenCoverage":
            seen_coverage * 100,

        "CorrectQueries":
            int(
                query_eval[
                    "CorrectCandidatePresent"
                ].sum()
            ),

        "CacheFile":
            cache_path.name,
    }


    snapshot_candidate_summaries.append(
        summary_row
    )


    print("\nSnapshot results:")

    print(
        "Mean candidates/query:",
        f"{candidate_counts.mean():.2f}"
    )

    print(
        "AnimalId seen rate:",
        f"{query_eval['AnimalSeenInSnapshot'].mean():.2%}"
    )

    print(
        "Overall candidate coverage:",
        f"{overall_coverage:.2%}"
    )

    print(
        "Seen-AnimalId candidate coverage:",
        f"{seen_coverage:.2%}"
    )

    print(
        "Correct queries:",
        int(
            query_eval[
                "CorrectCandidatePresent"
            ].sum()
        ),
        "/",
        len(query_eval)
    )


    # ========================================================
    # 18. MEMORY CLEANUP
    # ========================================================

    del candidate_pairs
    del candidate_presence
    del candidate_counts
    del query_eval

    # Objects only exist if this snapshot was newly built
    for variable_name in [
        "class_values",
        "class_to_number",
        "structural_profiles",
        "global_lactation_interval",
        "profile_lookup",
        "reference_indices",
        "reference_raw",
        "X_reference",
        "numeric_columns",
        "numeric_fill",
        "scaler",
        "reproduction_columns",
        "feature_names",
        "reference_class_ids",
        "reference_lactations",
        "reference_dim",
        "reference_exact_index",
        "reference_class_lactations",
        "reference_dim_bin",
        "cow_prototype",
        "lactation_prototype",
        "fresh_globals",
    ]:

        if variable_name in globals():

            del globals()[
                variable_name
            ]


    gc.collect()

    print(
        "Memory cleanup complete."
    )


# ============================================================
# 19. Final summary
# ============================================================

fresh_snapshot_candidate_summary_df = (
    pd.DataFrame(
        snapshot_candidate_summaries
    )
)


print("\n" + "=" * 75)

print(
    "ALL REMAINING SNAPSHOTS COMPLETED"
)

print("=" * 75)


display(
    fresh_snapshot_candidate_summary_df
)

Saved existing 2020-07-10 candidates: xgboost_cache/fresh_snapshot_candidates_20200710.pkl

BUILDING SNAPSHOT: 2020-07-20
Reference rows: 2,914,658
Assigned queries: 344
Structural profiles: 16,712
Sampled reference rows: 1,838,191
Reference matrix: (1838191, 13)
Fresh globals activated.
Classes: 6746
Reference matrix: (1838191, 13)
fresh0720_query: processed 50 / 344 queries
fresh0720_query: processed 100 / 344 queries
fresh0720_query: processed 150 / 344 queries
fresh0720_query: processed 200 / 344 queries
fresh0720_query: processed 250 / 344 queries
fresh0720_query: processed 300 / 344 queries
fresh0720_query: processed 344 / 344 queries
Original globals restored.
Candidate cache saved: xgboost_cache/fresh_snapshot_candidates_20200720.pkl

Snapshot results:
Mean candidates/query: 319.19
AnimalId seen rate: 92.73%
Overall candidate coverage: 87.21%
Seen-AnimalId candidate coverage: 94.04%
Correct queries: 300 / 344
Memory cleanup complete.

BUILDING SNAPSHOT: 2020-07-31
Reference row

,SnapshotDate,Queries,CandidateRows,MeanCandidates,AnimalSeenRate,OverallCoverage,SeenCoverage,CorrectQueries,CacheFile
0,2020-07-20,344,109801,319.188953,92.732558,87.209302,94.043887,300,fresh_snapshot_candidates_20200720.pkl
1,2020-07-31,310,98755,318.564516,95.161290,91.612903,96.271186,284,fresh_snapshot_candidates_20200731.pkl
2,2020-08-10,222,71040,320.000000,95.945946,92.792793,96.713615,206,fresh_snapshot_candidates_20200810.pkl
3,2020-08-20,443,141681,319.821670,96.839729,95.711061,98.834499,424,fresh_snapshot_candidates_20200820.pkl


In [56]:
# ============================================================
# MULTI-SNAPSHOT TRAINING SET
# STEP 8
#
# Load and combine all FIVE snapshot candidate tables.
#
# Important:
# CandidateClassId values are snapshot-local.
# This is OK because CandidateClassId / TrueClassId are NOT
# model features.
# ============================================================

import gc


FRESH_SNAPSHOT_CACHE_FILES = [
    "fresh_snapshot_candidates_20200710.pkl",
    "fresh_snapshot_candidates_20200720.pkl",
    "fresh_snapshot_candidates_20200731.pkl",
    "fresh_snapshot_candidates_20200810.pkl",
    "fresh_snapshot_candidates_20200820.pkl",
]


snapshot_candidate_tables = []

schema_reference = None

merge_summary = []


for filename in FRESH_SNAPSHOT_CACHE_FILES:

    path = (
        CACHE_DIR
        /
        filename
    )

    print(
        "Loading:",
        filename
    )

    df_snapshot = pd.read_pickle(
        path
    )


    # --------------------------------------------------------
    # Schema check
    # --------------------------------------------------------

    if schema_reference is None:

        schema_reference = list(
            df_snapshot.columns
        )

    else:

        assert (
            list(df_snapshot.columns)
            ==
            schema_reference
        ), (
            f"Column mismatch in {filename}"
        )


    # --------------------------------------------------------
    # Basic snapshot diagnostics
    # --------------------------------------------------------

    snapshot_presence = (
        df_snapshot
        .groupby("QueryId")[
            "IsCorrectCandidate"
        ]
        .max()
    )


    merge_summary.append({

        "File":
            filename,

        "Rows":
            len(df_snapshot),

        "Queries":
            df_snapshot[
                "QueryId"
            ].nunique(),

        "PositiveQueries":
            int(
                snapshot_presence.sum()
            ),

        "Columns":
            len(
                df_snapshot.columns
            ),
    })


    snapshot_candidate_tables.append(
        df_snapshot
    )


# ============================================================
# Combine
# ============================================================

fresh_multisnapshot_train = (
    pd.concat(
        snapshot_candidate_tables,
        ignore_index=True,
    )
)


# Free individual tables
del snapshot_candidate_tables
gc.collect()


# ============================================================
# Safety checks
# ============================================================

query_count = (
    fresh_multisnapshot_train[
        "QueryId"
    ].nunique()
)

candidate_presence = (
    fresh_multisnapshot_train
    .groupby("QueryId")[
        "IsCorrectCandidate"
    ]
    .max()
)

positive_queries = int(
    candidate_presence.sum()
)


assert query_count == 2000

assert positive_queries == 1822


# ============================================================
# Candidate-count diagnostics
# ============================================================

candidate_counts = (
    fresh_multisnapshot_train
    .groupby("QueryId")
    .size()
)


print("\n" + "=" * 70)

display(
    pd.DataFrame(
        merge_summary
    )
)


print(
    "Combined rows:",
    f"{len(fresh_multisnapshot_train):,}"
)

print(
    "Combined queries:",
    query_count
)

print(
    "Mean candidates/query:",
    f"{candidate_counts.mean():.2f}"
)

print(
    "Queries with correct candidate:",
    positive_queries
)

print(
    "Queries without correct candidate:",
    query_count
    -
    positive_queries
)

print(
    "Overall training candidate coverage:",
    f"{positive_queries / query_count:.2%}"
)

print(
    "Combined shape:",
    fresh_multisnapshot_train.shape
)

print(
    "Memory:",
    f"{fresh_multisnapshot_train.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)


# ============================================================
# Check QueryId uniqueness across snapshots
#
# Each QueryId should belong to exactly one snapshot prefix.
# ============================================================

print(
    "\nDuplicate candidate rows:",
    fresh_multisnapshot_train[
        [
            "QueryId",
            "CandidateClassId",
        ]
    ].duplicated().sum()
)

Loading: fresh_snapshot_candidates_20200710.pkl
Loading: fresh_snapshot_candidates_20200720.pkl
Loading: fresh_snapshot_candidates_20200731.pkl
Loading: fresh_snapshot_candidates_20200810.pkl
Loading: fresh_snapshot_candidates_20200820.pkl



,File,Rows,Queries,PositiveQueries,Columns
0,fresh_snapshot_candidates_20200710.pkl,217352,681,608,56
1,fresh_snapshot_candidates_20200720.pkl,109801,344,300,56
2,fresh_snapshot_candidates_20200731.pkl,98755,310,284,56
3,fresh_snapshot_candidates_20200810.pkl,71040,222,206,56
4,fresh_snapshot_candidates_20200820.pkl,141681,443,424,56


Combined rows: 638,629
Combined queries: 2000
Mean candidates/query: 319.31
Queries with correct candidate: 1822
Queries without correct candidate: 178
Overall training candidate coverage: 91.10%
Combined shape: (638629, 56)
Memory: 256.07 MB

Duplicate candidate rows: 0


In [57]:
# ============================================================
# FRESH DEV REFERENCE SYSTEM
# STEP 9
#
# Reference:
#   reranker_history
#   +
#   ALL reranker_train_queries
#
# Cutoff:
#   2020-09-20
#
# Queries:
#   existing 500 pilot_dev_queries
#
# NO candidate generation yet.
# ============================================================

import gc


FRESH_DEV_CUTOFF = pd.Timestamp(
    "2020-09-20"
)


# ============================================================
# 1. DEV reference dataset
# ============================================================

fresh_dev_reference_system = (
    pd.concat(
        [
            reranker_history,
            reranker_train_queries,
        ],
        ignore_index=True,
    )
    .copy()
)


fresh_dev_reference_system[
    "EventDate"
] = pd.to_datetime(
    fresh_dev_reference_system[
        "EventDate"
    ]
)


# Safety
assert (
    fresh_dev_reference_system[
        "EventDate"
    ].max()
    <=
    FRESH_DEV_CUTOFF
)

assert (
    pd.to_datetime(
        pilot_dev_queries[
            "EventDate"
        ]
    ).min()
    >
    FRESH_DEV_CUTOFF
)


print(
    "Fresh DEV reference rows:",
    f"{len(fresh_dev_reference_system):,}"
)

print(
    "Reference date range:",
    fresh_dev_reference_system[
        "EventDate"
    ].min(),
    "→",
    fresh_dev_reference_system[
        "EventDate"
    ].max()
)


# ============================================================
# 2. Class mapping
# ============================================================

fresh_dev_class_values = pd.Index(
    fresh_dev_reference_system[
        "AnimalId"
    ]
    .astype(object)
    .unique()
)


fresh_dev_class_to_number = {
    animal_id: class_number

    for class_number, animal_id
    in enumerate(
        fresh_dev_class_values
    )
}


fresh_dev_number_to_class = {
    class_number: animal_id

    for class_number, animal_id
    in enumerate(
        fresh_dev_class_values
    )
}


# ============================================================
# 3. Origin date
# ============================================================

fresh_dev_origin_date = (
    fresh_dev_reference_system[
        "EventDate"
    ]
    .min()
)


# ============================================================
# 4. Structural profiles
# ============================================================

(
    fresh_dev_profiles,
    fresh_dev_global_lactation_interval,
) = build_structural_profiles(
    fresh_dev_reference_system,
    fresh_dev_class_to_number,
    fresh_dev_origin_date,
    include_projection=True,
)


fresh_dev_profile_lookup = (
    make_profile_lookup(
        fresh_dev_profiles
    )
)


# ============================================================
# 5. Sample reference rows
# ============================================================

fresh_dev_reference_indices = (
    sample_reference_rows(
        fresh_dev_reference_system,

        max_rows_per_group=
            REFERENCE_ROWS_PER_GROUP,

        recent_fraction=0.80,

        random_state=RANDOM_STATE,
    )
)


fresh_dev_reference_raw = (
    fresh_dev_reference_system
    .loc[
        fresh_dev_reference_indices
    ]
    .copy()
)


# ============================================================
# 6. Fit distance preprocessing
# ============================================================

(
    X_fresh_dev_reference,
    fresh_dev_numeric_columns,
    fresh_dev_numeric_fill,
    fresh_dev_scaler,
    fresh_dev_reproduction_columns,
    fresh_dev_feature_names,
) = fit_distance_preprocessor(
    fresh_dev_reference_raw
)


# ============================================================
# 7. Reference metadata
# ============================================================

fresh_dev_reference_class_ids = (
    fresh_dev_reference_raw[
        "AnimalId"
    ]
    .astype(object)
    .map(
        fresh_dev_class_to_number
    )
    .astype(np.int32)
    .to_numpy()
)


fresh_dev_reference_lactations = (
    fresh_dev_reference_raw[
        "LactationNumber"
    ]
    .round()
    .astype(np.int16)
    .to_numpy()
)


fresh_dev_reference_dim = (
    fresh_dev_reference_raw[
        "DaysInMilk"
    ]
    .astype(np.float32)
    .to_numpy()
)


# ============================================================
# 8. Exact AnimalId-lactation index
# ============================================================

(
    fresh_dev_reference_exact_index,
    fresh_dev_reference_class_lactations,
) = build_reference_index(
    fresh_dev_reference_class_ids,
    fresh_dev_reference_lactations,
)


# ============================================================
# 9. Whole-cow prototypes
# ============================================================

fresh_dev_cow_prototype = {}


for class_id in np.unique(
    fresh_dev_reference_class_ids
):

    positions = np.flatnonzero(
        fresh_dev_reference_class_ids
        ==
        class_id
    )

    fresh_dev_cow_prototype[
        int(class_id)
    ] = np.median(
        X_fresh_dev_reference[
            positions
        ],
        axis=0,
    ).astype(np.float32)


# ============================================================
# 10. Lactation + DIM-bin prototypes
# ============================================================

fresh_dev_reference_dim_bin = np.floor(
    fresh_dev_reference_dim
    /
    DIM_BIN_WIDTH
).astype(np.int16)


fresh_dev_lactation_prototype = {}


for (
    class_id,
    lactation_number
), positions in (
    fresh_dev_reference_exact_index.items()
):

    bins_here = (
        fresh_dev_reference_dim_bin[
            positions
        ]
    )

    prototype_by_bin = {}


    for dim_bin in np.unique(
        bins_here
    ):

        bin_positions = (
            positions[
                bins_here
                ==
                dim_bin
            ]
        )


        prototype_by_bin[
            int(dim_bin)
        ] = np.median(
            X_fresh_dev_reference[
                bin_positions
            ],
            axis=0,
        ).astype(np.float32)


    fresh_dev_lactation_prototype[
        (
            int(class_id),
            int(lactation_number),
        )
    ] = prototype_by_bin


# ============================================================
# 11. DEV AnimalId availability
# ============================================================

fresh_dev_seen_rate = (
    pilot_dev_queries[
        "AnimalId"
    ]
    .astype(object)
    .isin(
        fresh_dev_class_values
    )
    .mean()
)


# ============================================================
# 12. Diagnostics
# ============================================================

print("\n" + "=" * 70)

print(
    "Fresh DEV classes:",
    f"{len(fresh_dev_class_values):,}"
)

print(
    "Structural profiles:",
    f"{len(fresh_dev_profiles):,}"
)

print(
    "Sampled reference rows:",
    f"{len(fresh_dev_reference_raw):,}"
)

print(
    "Reference matrix:",
    X_fresh_dev_reference.shape
)

print(
    "Distance features:",
    len(
        fresh_dev_feature_names
    )
)

print(
    "AnimalId-lactation groups:",
    f"{len(fresh_dev_reference_exact_index):,}"
)

print(
    "Whole-cow prototypes:",
    f"{len(fresh_dev_cow_prototype):,}"
)

print(
    "Lactation/DIM prototype groups:",
    f"{len(fresh_dev_lactation_prototype):,}"
)

print(
    "DEV AnimalId seen rate:",
    f"{fresh_dev_seen_rate:.2%}"
)

print(
    "Reference matrix memory:",
    f"{X_fresh_dev_reference.nbytes / 1024**2:.2f} MB"
)

Fresh DEV reference rows: 3,379,883
Reference date range: 2019-06-14 00:00:00 → 2020-09-20 00:00:00

Fresh DEV classes: 7,111
Structural profiles: 18,068
Sampled reference rows: 2,040,891
Reference matrix: (2040891, 13)
Distance features: 13
AnimalId-lactation groups: 10,959
Whole-cow prototypes: 7,111
Lactation/DIM prototype groups: 10,959
DEV AnimalId seen rate: 95.20%
Reference matrix memory: 101.21 MB


In [58]:
# ============================================================
# FRESH DEV CANDIDATES
#
# Use the FRESH DEV reference system
# to rebuild candidates for the SAME 500 dev queries.
#
# Globals are temporarily replaced and then restored.
# ============================================================

FRESH_DEV_PREFIX = "freshdev_query"

FRESH_DEV_CACHE_PATH = (
    CACHE_DIR
    /
    "fresh_dev_candidates_500_global320.pkl"
)


# ============================================================
# 1. Fresh DEV global mapping
# ============================================================

fresh_dev_globals = {

    "reranker_class_to_number":
        fresh_dev_class_to_number,

    "reranker_cow_prototype":
        fresh_dev_cow_prototype,

    "reranker_feature_names":
        fresh_dev_feature_names,

    "reranker_lactation_prototype":
        fresh_dev_lactation_prototype,

    "reranker_numeric_columns":
        fresh_dev_numeric_columns,

    "reranker_numeric_fill":
        fresh_dev_numeric_fill,

    "reranker_origin_date":
        fresh_dev_origin_date,

    "reranker_profile_lookup":
        fresh_dev_profile_lookup,

    "reranker_reference_class_lactations":
        fresh_dev_reference_class_lactations,

    "reranker_reference_dim":
        fresh_dev_reference_dim,

    "reranker_reference_exact_index":
        fresh_dev_reference_exact_index,

    "reranker_reproduction_columns":
        fresh_dev_reproduction_columns,

    "reranker_scaler":
        fresh_dev_scaler,

    "X_reranker_reference":
        X_fresh_dev_reference,
}


# ============================================================
# 2. Backup current globals
# ============================================================

old_dev_globals = {
    name: globals()[name]
    for name in fresh_dev_globals
}


# ============================================================
# 3. Build / load fresh DEV candidates
# ============================================================

if FRESH_DEV_CACHE_PATH.exists():

    print(
        "Loading cached fresh DEV candidates..."
    )

    fresh_dev_candidate_pairs = (
        pd.read_pickle(
            FRESH_DEV_CACHE_PATH
        )
    )

else:

    try:

        # ----------------------------------------------------
        # Activate FRESH DEV reference
        # ----------------------------------------------------

        for name, value in (
            fresh_dev_globals.items()
        ):
            globals()[name] = value


        print(
            "Fresh DEV reference activated."
        )

        print(
            "Classes:",
            len(reranker_class_to_number)
        )

        print(
            "Reference matrix:",
            X_reranker_reference.shape
        )

        print(
            "Max candidates:",
            RERANKER_CANDIDATE_PARAMS[
                "max_candidates"
            ]
        )


        # ----------------------------------------------------
        # Generate candidates
        # ----------------------------------------------------

        fresh_dev_candidate_pairs = (
            build_candidate_dataset(
                pilot_dev_queries,
                prefix=FRESH_DEV_PREFIX,
                progress_every=50,
            )
        )


        fresh_dev_candidate_pairs.to_pickle(
            FRESH_DEV_CACHE_PATH
        )

        print(
            "Saved:",
            FRESH_DEV_CACHE_PATH
        )


    finally:

        # ----------------------------------------------------
        # ALWAYS restore original globals
        # ----------------------------------------------------

        for name, value in (
            old_dev_globals.items()
        ):
            globals()[name] = value


        print(
            "Original reranker globals restored."
        )

Fresh DEV reference activated.
Classes: 7111
Reference matrix: (2040891, 13)
Max candidates: 320
freshdev_query: processed 50 / 500 queries
freshdev_query: processed 100 / 500 queries
freshdev_query: processed 150 / 500 queries
freshdev_query: processed 200 / 500 queries
freshdev_query: processed 250 / 500 queries
freshdev_query: processed 300 / 500 queries
freshdev_query: processed 350 / 500 queries
freshdev_query: processed 400 / 500 queries
freshdev_query: processed 450 / 500 queries
freshdev_query: processed 500 / 500 queries
Saved: xgboost_cache/fresh_dev_candidates_500_global320.pkl
Original reranker globals restored.


In [59]:
# ============================================================
# FRESH DEV COVERAGE
# ============================================================

fresh_dev_presence = (
    fresh_dev_candidate_pairs
    .groupby("QueryId")[
        "IsCorrectCandidate"
    ]
    .max()
)


fresh_dev_query_ids = [
    f"{FRESH_DEV_PREFIX}_{idx}"
    for idx in pilot_dev_queries.index
]


fresh_dev_eval = (
    pilot_dev_queries[
        ["AnimalId"]
    ]
    .copy()
)


fresh_dev_eval[
    "QueryId"
] = fresh_dev_query_ids


# ============================================================
# AnimalId availability
# ============================================================

fresh_dev_eval[
    "AnimalSeenInFreshReference"
] = (
    fresh_dev_eval[
        "AnimalId"
    ]
    .astype(object)
    .isin(
        fresh_dev_class_values
    )
)


# ============================================================
# Correct candidate presence
# ============================================================

fresh_dev_eval[
    "CorrectCandidatePresent"
] = (
    fresh_dev_eval[
        "QueryId"
    ]
    .map(
        fresh_dev_presence
    )
    .fillna(0)
    .astype(bool)
)


# ============================================================
# Metrics
# ============================================================

fresh_dev_overall_coverage = (
    fresh_dev_eval[
        "CorrectCandidatePresent"
    ].mean()
)


fresh_dev_seen_coverage = (
    fresh_dev_eval.loc[
        fresh_dev_eval[
            "AnimalSeenInFreshReference"
        ],
        "CorrectCandidatePresent",
    ]
    .mean()
)


fresh_dev_candidate_counts = (
    fresh_dev_candidate_pairs
    .groupby("QueryId")
    .size()
)


print("=" * 70)

print(
    "DEV queries:",
    len(fresh_dev_eval)
)

print(
    "Candidate rows:",
    f"{len(fresh_dev_candidate_pairs):,}"
)

print(
    "Mean candidates/query:",
    f"{fresh_dev_candidate_counts.mean():.2f}"
)

print(
    "AnimalId seen rate:",
    f"{fresh_dev_eval['AnimalSeenInFreshReference'].mean():.2%}"
)

print(
    "Overall candidate coverage:",
    f"{fresh_dev_overall_coverage:.2%}"
)

print(
    "Seen-AnimalId candidate coverage:",
    f"{fresh_dev_seen_coverage:.2%}"
)

print(
    "Correct queries:",
    int(
        fresh_dev_eval[
            "CorrectCandidatePresent"
        ].sum()
    ),
    "/",
    len(fresh_dev_eval)
)

DEV queries: 500
Candidate rows: 159,935
Mean candidates/query: 319.87
AnimalId seen rate: 95.20%
Overall candidate coverage: 92.40%
Seen-AnimalId candidate coverage: 97.06%
Correct queries: 462 / 500


In [60]:
# ============================================================
# NEXT STEP
# Fresh-reference candidate coverage at different Top-K values
#
# No model training yet.
# ============================================================

FRESH_K_VALUES = [
    80,
    120,
    160,
    200,
    240,
    280,
    320,
]

fresh_k_results = []


for candidate_k in FRESH_K_VALUES:

    # --------------------------------------------------------
    # DEV
    # --------------------------------------------------------

    dev_k = (
        fresh_dev_candidate_pairs.loc[
            fresh_dev_candidate_pairs[
                "CandidateStartRank"
            ]
            <= candidate_k
        ]
        .copy()
    )


    dev_presence = (
        dev_k
        .groupby("QueryId")[
            "IsCorrectCandidate"
        ]
        .max()
    )


    dev_correct = (
        pd.Series(
            fresh_dev_query_ids
        )
        .map(
            dev_presence
        )
        .fillna(0)
        .astype(bool)
    )


    dev_overall_coverage = (
        dev_correct.mean()
    )


    # Seen-only coverage
    dev_seen_mask = (
        fresh_dev_eval[
            "AnimalSeenInFreshReference"
        ]
        .to_numpy()
    )

    dev_seen_coverage = (
        dev_correct.to_numpy()[
            dev_seen_mask
        ]
        .mean()
    )


    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    train_k = (
        fresh_multisnapshot_train.loc[
            fresh_multisnapshot_train[
                "CandidateStartRank"
            ]
            <= candidate_k
        ]
        .copy()
    )


    train_presence = (
        train_k
        .groupby("QueryId")[
            "IsCorrectCandidate"
        ]
        .max()
    )


    train_coverage = (
        train_presence.sum()
        /
        2000
    )


    # --------------------------------------------------------
    # Candidate counts
    # --------------------------------------------------------

    dev_candidate_counts = (
        dev_k
        .groupby("QueryId")
        .size()
    )


    fresh_k_results.append({

        "candidate_top_k":
            candidate_k,

        "train_coverage":
            train_coverage * 100,

        "dev_coverage":
            dev_overall_coverage * 100,

        "dev_seen_coverage":
            dev_seen_coverage * 100,

        "dev_correct_queries":
            int(
                dev_correct.sum()
            ),

        "mean_dev_candidates":
            dev_candidate_counts.mean(),
    })


fresh_k_results_df = (
    pd.DataFrame(
        fresh_k_results
    )
)


display(
    fresh_k_results_df
)

,candidate_top_k,train_coverage,dev_coverage,dev_seen_coverage,dev_correct_queries,mean_dev_candidates
0,80,87.55,87.6,92.016807,438,80.00
1,120,88.25,88.2,92.647059,441,120.00
2,160,88.85,89.6,94.117647,448,160.00
3,200,89.40,90.0,94.537815,450,200.00
4,240,90.20,91.0,95.588235,455,240.00
5,280,90.70,92.4,97.058824,462,279.95
6,320,91.10,92.4,97.058824,462,319.87


In [61]:
# ============================================================
# FRESH TOP-160 HGB
#
# TRAIN:
#   5 leakage-safe, freshness-matched snapshots
#
# DEV:
#   fresh reference through 2020-09-20
#
# Candidate set:
#   Global structural Top-160
# ============================================================

FRESH_MODEL_K = 160


# ============================================================
# 1. TRAIN Top-160
# ============================================================

fresh_train160 = (
    fresh_multisnapshot_train.loc[
        fresh_multisnapshot_train[
            "CandidateStartRank"
        ]
        <= FRESH_MODEL_K
    ]
    .copy()
    .reset_index(drop=True)
)


# CandidateCount must describe the reduced candidate set
fresh_train160["CandidateCount"] = (
    fresh_train160
    .groupby("QueryId")["QueryId"]
    .transform("size")
    .astype(np.int16)
)


# Recompute within-query rank features
# using ONLY the Top-160 candidates
fresh_train160_ranked = (
    add_candidate_rank_features(
        fresh_train160
    )
)


# ============================================================
# 2. Keep training queries where true candidate exists
# ============================================================

fresh_train_positive_available = (
    fresh_train160_ranked
    .groupby("QueryId")[
        "IsCorrectCandidate"
    ]
    .transform("max")
)


fresh_train160_model = (
    fresh_train160_ranked.loc[
        fresh_train_positive_available == 1
    ]
    .reset_index(drop=True)
)


# ============================================================
# 3. DEV Top-160
# ============================================================

fresh_dev160 = (
    fresh_dev_candidate_pairs.loc[
        fresh_dev_candidate_pairs[
            "CandidateStartRank"
        ]
        <= FRESH_MODEL_K
    ]
    .copy()
    .reset_index(drop=True)
)


fresh_dev160["CandidateCount"] = (
    fresh_dev160
    .groupby("QueryId")["QueryId"]
    .transform("size")
    .astype(np.int16)
)


fresh_dev160_ranked = (
    add_candidate_rank_features(
        fresh_dev160
    )
)


# ============================================================
# 4. Feature list
# ============================================================

FRESH_NON_FEATURE_COLUMNS = [
    "QueryId",
    "CandidateClassId",
    "TrueClassId",
    "IsCorrectCandidate",
]


FRESH_RERANKER_FEATURES = [
    column
    for column in fresh_train160_model.columns
    if column not in FRESH_NON_FEATURE_COLUMNS
]


# Make sure DEV has exactly the required model features
missing_dev_features = [
    column
    for column in FRESH_RERANKER_FEATURES
    if column not in fresh_dev160_ranked.columns
]

assert len(missing_dev_features) == 0, (
    f"Missing DEV features: {missing_dev_features}"
)


# ============================================================
# 5. Matrices
# ============================================================

X_fresh_train160 = (
    fresh_train160_model[
        FRESH_RERANKER_FEATURES
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .astype(np.float32)
)


y_fresh_train160 = (
    fresh_train160_model[
        "IsCorrectCandidate"
    ]
    .astype(np.int8)
)


X_fresh_dev160 = (
    fresh_dev160_ranked[
        FRESH_RERANKER_FEATURES
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .astype(np.float32)
)


print("=" * 70)

print(
    "Training queries with correct candidate:",
    fresh_train160_model["QueryId"].nunique()
)

print(
    "Training candidate rows:",
    f"{len(fresh_train160_model):,}"
)

print(
    "DEV candidate rows:",
    f"{len(fresh_dev160_ranked):,}"
)

print(
    "Features:",
    len(FRESH_RERANKER_FEATURES)
)

print(
    "Positive training rows:",
    int(y_fresh_train160.sum())
)

print(
    "Train matrix:",
    X_fresh_train160.shape
)

print(
    "DEV matrix:",
    X_fresh_dev160.shape
)

Training queries with correct candidate: 1777
Training candidate rows: 284,097
DEV candidate rows: 80,000
Features: 52
Positive training rows: 1777
Train matrix: (284097, 52)
DEV matrix: (80000, 52)


In [62]:
from sklearn.ensemble import HistGradientBoostingClassifier


# ============================================================
# 6. Train Fresh Top-160 HGB
# ============================================================

fresh_hgb160 = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=400,
    max_leaf_nodes=7,
    min_samples_leaf=10,
    l2_regularization=1.0,

    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=20,

    random_state=RANDOM_STATE,
)


# No sample weighting
fresh_hgb160.fit(
    X_fresh_train160,
    y_fresh_train160
)


print(
    "Fresh Top-160 HGB trained."
)

print(
    "Iterations used:",
    fresh_hgb160.n_iter_
)

Fresh Top-160 HGB trained.
Iterations used: 340


In [63]:
# ============================================================
# 7. Predict DEV
# ============================================================

fresh_prob160 = (
    fresh_hgb160.predict_proba(
        X_fresh_dev160
    )[:, 1]
)


fresh_scored160 = (
    fresh_dev160_ranked[
        [
            "QueryId",
            "CandidateClassId",
            "TrueClassId",
            "IsCorrectCandidate",
        ]
    ]
    .copy()
)


fresh_scored160[
    "Probability"
] = fresh_prob160


# Best candidate in each query
fresh_best160 = (
    fresh_scored160
    .sort_values(
        [
            "QueryId",
            "Probability",
        ],
        ascending=[
            True,
            False,
        ]
    )
    .groupby(
        "QueryId",
        as_index=False,
    )
    .first()
)


fresh_best160[
    "PredictionCorrect"
] = (
    fresh_best160[
        "CandidateClassId"
    ]
    ==
    fresh_best160[
        "TrueClassId"
    ]
)


# ============================================================
# 8. Top-160 candidate presence
# ============================================================

fresh_presence160 = (
    fresh_dev160_ranked
    .groupby("QueryId")[
        "IsCorrectCandidate"
    ]
    .max()
)


# ============================================================
# 9. Evaluation table
# ============================================================

fresh_hgb160_eval = (
    fresh_dev_eval[
        [
            "QueryId",
            "AnimalSeenInFreshReference",
        ]
    ]
    .copy()
)


fresh_hgb160_eval[
    "CorrectCandidatePresent"
] = (
    fresh_hgb160_eval[
        "QueryId"
    ]
    .map(
        fresh_presence160
    )
    .fillna(0)
    .astype(bool)
)


fresh_hgb160_eval = (
    fresh_hgb160_eval
    .merge(
        fresh_best160[
            [
                "QueryId",
                "PredictionCorrect",
            ]
        ],
        on="QueryId",
        how="left",
    )
)


fresh_hgb160_eval[
    "PredictionCorrect"
] = (
    fresh_hgb160_eval[
        "PredictionCorrect"
    ]
    .fillna(False)
)


# ============================================================
# 10. Metrics
# ============================================================

fresh_hgb160_overall = (
    fresh_hgb160_eval[
        "PredictionCorrect"
    ].mean()
)


fresh_hgb160_seen = (
    fresh_hgb160_eval.loc[
        fresh_hgb160_eval[
            "AnimalSeenInFreshReference"
        ],
        "PredictionCorrect",
    ].mean()
)


fresh_hgb160_conditional = (
    fresh_hgb160_eval.loc[
        fresh_hgb160_eval[
            "CorrectCandidatePresent"
        ],
        "PredictionCorrect",
    ].mean()
)


fresh_hgb160_coverage = (
    fresh_hgb160_eval[
        "CorrectCandidatePresent"
    ].mean()
)


print("=" * 70)

print(
    "Fresh Top-160 candidate coverage:",
    f"{fresh_hgb160_coverage:.2%}"
)

print(
    "Fresh Top-160 HGB overall accuracy:",
    f"{fresh_hgb160_overall:.2%}"
)

print(
    "Fresh Top-160 HGB seen-AnimalId accuracy:",
    f"{fresh_hgb160_seen:.2%}"
)

print(
    "Fresh Top-160 HGB conditional ranking accuracy:",
    f"{fresh_hgb160_conditional:.2%}"
)

print(
    "Correct predictions:",
    int(
        fresh_hgb160_eval[
            "PredictionCorrect"
        ].sum()
    ),
    "/",
    len(fresh_hgb160_eval)
)

Fresh Top-160 candidate coverage: 89.60%
Fresh Top-160 HGB overall accuracy: 52.80%
Fresh Top-160 HGB seen-AnimalId accuracy: 55.46%
Fresh Top-160 HGB conditional ranking accuracy: 58.93%
Correct predictions: 264 / 500


In [64]:
# ============================================================
# CONFIRMATION VALIDATION SET
#
# Goal:
#   Create 2,000 NEW DEV queries that were never used
#   in the original 500-query pilot.
#
# We keep the model completely unchanged:
#   Top-K = 160
#   Same HGB
#   Same fresh DEV reference
#
# The 2,000 queries are spread across the whole DEV period.
# ============================================================

import numpy as np


CONFIRMATION_SIZE = 2000


# ============================================================
# 1. Remove the original 500 pilot queries
# ============================================================

remaining_dev_queries = (
    reranker_dev_queries.loc[
        ~reranker_dev_queries.index.isin(
            pilot_dev_queries.index
        )
    ]
    .copy()
)


print(
    "Remaining DEV queries:",
    f"{len(remaining_dev_queries):,}"
)


# ============================================================
# 2. Sort chronologically
# ============================================================

remaining_dev_queries[
    "EventDate"
] = pd.to_datetime(
    remaining_dev_queries[
        "EventDate"
    ]
)


remaining_dev_queries = (
    remaining_dev_queries
    .sort_values(
        "EventDate"
    )
)


# ============================================================
# 3. Select evenly across the full DEV period
#
# This is better than taking only the first 2,000,
# because later DEV queries are farther away from
# the reference cutoff and may be harder.
# ============================================================

confirmation_positions = (
    np.linspace(
        0,
        len(remaining_dev_queries) - 1,
        CONFIRMATION_SIZE,
    )
    .round()
    .astype(int)
)


confirmation_dev_queries = (
    remaining_dev_queries
    .iloc[
        confirmation_positions
    ]
    .copy()
)


# ============================================================
# 4. Safety checks
# ============================================================

assert (
    len(
        confirmation_dev_queries
    )
    ==
    CONFIRMATION_SIZE
)


assert (
    len(
        set(
            confirmation_dev_queries.index
        )
        &
        set(
            pilot_dev_queries.index
        )
    )
    ==
    0
)


# ============================================================
# 5. Diagnostics
# ============================================================

confirmation_seen_rate = (
    confirmation_dev_queries[
        "AnimalId"
    ]
    .astype(object)
    .isin(
        fresh_dev_class_values
    )
    .mean()
)


print("\n" + "=" * 70)

print(
    "Confirmation queries:",
    len(
        confirmation_dev_queries
    )
)

print(
    "Overlap with old 500:",
    len(
        set(
            confirmation_dev_queries.index
        )
        &
        set(
            pilot_dev_queries.index
        )
    )
)

print(
    "Date range:",
    confirmation_dev_queries[
        "EventDate"
    ].min(),
    "→",
    confirmation_dev_queries[
        "EventDate"
    ].max()
)

print(
    "AnimalId seen rate:",
    f"{confirmation_seen_rate:.2%}"
)

Remaining DEV queries: 604,705

Confirmation queries: 2000
Overlap with old 500: 0
Date range: 2020-09-21 00:00:00 → 2020-12-13 00:00:00
AnimalId seen rate: 94.40%


In [65]:
# ============================================================
# CONFIRMATION TEST
#
# 2,000 completely new DEV queries
#
# IMPORTANT:
#   - SAME fresh DEV reference
#   - SAME Top-160
#   - SAME trained fresh_hgb160
#   - NO retraining
#   - NO parameter tuning
# ============================================================

CONFIRMATION_PREFIX = "confirmationdev_query"

CONFIRMATION_CACHE_PATH = (
    CACHE_DIR
    /
    "confirmation_dev_candidates_2000_global320.pkl"
)


# ============================================================
# 1. Fresh DEV global mapping
# ============================================================

confirmation_fresh_globals = {

    "reranker_class_to_number":
        fresh_dev_class_to_number,

    "reranker_cow_prototype":
        fresh_dev_cow_prototype,

    "reranker_feature_names":
        fresh_dev_feature_names,

    "reranker_lactation_prototype":
        fresh_dev_lactation_prototype,

    "reranker_numeric_columns":
        fresh_dev_numeric_columns,

    "reranker_numeric_fill":
        fresh_dev_numeric_fill,

    "reranker_origin_date":
        fresh_dev_origin_date,

    "reranker_profile_lookup":
        fresh_dev_profile_lookup,

    "reranker_reference_class_lactations":
        fresh_dev_reference_class_lactations,

    "reranker_reference_dim":
        fresh_dev_reference_dim,

    "reranker_reference_exact_index":
        fresh_dev_reference_exact_index,

    "reranker_reproduction_columns":
        fresh_dev_reproduction_columns,

    "reranker_scaler":
        fresh_dev_scaler,

    "X_reranker_reference":
        X_fresh_dev_reference,
}


# ============================================================
# 2. Backup current globals
# ============================================================

confirmation_old_globals = {
    name: globals()[name]
    for name in confirmation_fresh_globals
}


# ============================================================
# 3. Generate / load confirmation candidates
# ============================================================

if CONFIRMATION_CACHE_PATH.exists():

    print(
        "Loading cached confirmation candidates..."
    )

    confirmation_candidate_pairs = (
        pd.read_pickle(
            CONFIRMATION_CACHE_PATH
        )
    )

else:

    try:

        for name, value in (
            confirmation_fresh_globals.items()
        ):
            globals()[name] = value


        print(
            "Fresh DEV reference activated."
        )

        print(
            "Reference matrix:",
            X_reranker_reference.shape
        )

        print(
            "Confirmation queries:",
            len(confirmation_dev_queries)
        )


        confirmation_candidate_pairs = (
            build_candidate_dataset(
                confirmation_dev_queries,
                prefix=CONFIRMATION_PREFIX,
                progress_every=100,
            )
        )


        confirmation_candidate_pairs.to_pickle(
            CONFIRMATION_CACHE_PATH
        )


        print(
            "Saved candidate cache:",
            CONFIRMATION_CACHE_PATH
        )


    finally:

        for name, value in (
            confirmation_old_globals.items()
        ):
            globals()[name] = value


        print(
            "Original globals restored."
        )


print(
    "Candidate rows:",
    f"{len(confirmation_candidate_pairs):,}"
)

print(
    "Queries:",
    confirmation_candidate_pairs[
        "QueryId"
    ].nunique()
)

Fresh DEV reference activated.
Reference matrix: (2040891, 13)
Confirmation queries: 2000
confirmationdev_query: processed 100 / 2000 queries
confirmationdev_query: processed 200 / 2000 queries
confirmationdev_query: processed 300 / 2000 queries
confirmationdev_query: processed 400 / 2000 queries
confirmationdev_query: processed 500 / 2000 queries
confirmationdev_query: processed 600 / 2000 queries
confirmationdev_query: processed 700 / 2000 queries
confirmationdev_query: processed 800 / 2000 queries
confirmationdev_query: processed 900 / 2000 queries
confirmationdev_query: processed 1000 / 2000 queries
confirmationdev_query: processed 1100 / 2000 queries
confirmationdev_query: processed 1200 / 2000 queries
confirmationdev_query: processed 1300 / 2000 queries
confirmationdev_query: processed 1400 / 2000 queries
confirmationdev_query: processed 1500 / 2000 queries
confirmationdev_query: processed 1600 / 2000 queries
confirmationdev_query: processed 1700 / 2000 queries
confirmationdev_qu

In [66]:
# ============================================================
# 4. Restrict confirmation candidates to SAME Top-160
# ============================================================

confirmation160 = (
    confirmation_candidate_pairs.loc[
        confirmation_candidate_pairs[
            "CandidateStartRank"
        ]
        <= 160
    ]
    .copy()
    .reset_index(drop=True)
)


# Candidate count after Top-160 filtering
confirmation160[
    "CandidateCount"
] = (
    confirmation160
    .groupby("QueryId")[
        "QueryId"
    ]
    .transform("size")
    .astype(np.int16)
)


# Recompute rank features exactly as in training
confirmation160_ranked = (
    add_candidate_rank_features(
        confirmation160
    )
)


# ============================================================
# 5. Feature consistency check
# ============================================================

missing_confirmation_features = [
    column
    for column in FRESH_RERANKER_FEATURES
    if column
    not in confirmation160_ranked.columns
]


assert (
    len(missing_confirmation_features)
    ==
    0
), (
    f"Missing features: "
    f"{missing_confirmation_features}"
)


X_confirmation160 = (
    confirmation160_ranked[
        FRESH_RERANKER_FEATURES
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .astype(np.float32)
)


print(
    "Confirmation model matrix:",
    X_confirmation160.shape
)


# ============================================================
# 6. Predict using EXISTING model
# ============================================================

confirmation_prob160 = (
    fresh_hgb160.predict_proba(
        X_confirmation160
    )[:, 1]
)


confirmation_scored160 = (
    confirmation160_ranked[
        [
            "QueryId",
            "CandidateClassId",
            "TrueClassId",
            "IsCorrectCandidate",
        ]
    ]
    .copy()
)


confirmation_scored160[
    "Probability"
] = confirmation_prob160


# Highest probability candidate per query
confirmation_best160 = (
    confirmation_scored160
    .sort_values(
        [
            "QueryId",
            "Probability",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .groupby(
        "QueryId",
        as_index=False,
    )
    .first()
)


confirmation_best160[
    "PredictionCorrect"
] = (
    confirmation_best160[
        "CandidateClassId"
    ]
    ==
    confirmation_best160[
        "TrueClassId"
    ]
)


# ============================================================
# 7. Candidate presence
# ============================================================

confirmation_presence160 = (
    confirmation160_ranked
    .groupby("QueryId")[
        "IsCorrectCandidate"
    ]
    .max()
)


# ============================================================
# 8. Build query-level evaluation table
# ============================================================

confirmation_query_ids = [
    f"{CONFIRMATION_PREFIX}_{idx}"
    for idx in confirmation_dev_queries.index
]


confirmation_eval = (
    confirmation_dev_queries[
        ["AnimalId"]
    ]
    .copy()
)


confirmation_eval[
    "QueryId"
] = confirmation_query_ids


confirmation_eval[
    "AnimalSeenInFreshReference"
] = (
    confirmation_eval[
        "AnimalId"
    ]
    .astype(object)
    .isin(
        fresh_dev_class_values
    )
)


confirmation_eval[
    "CorrectCandidatePresent"
] = (
    confirmation_eval[
        "QueryId"
    ]
    .map(
        confirmation_presence160
    )
    .fillna(0)
    .astype(bool)
)


confirmation_eval = (
    confirmation_eval
    .merge(
        confirmation_best160[
            [
                "QueryId",
                "PredictionCorrect",
            ]
        ],
        on="QueryId",
        how="left",
    )
)


confirmation_eval[
    "PredictionCorrect"
] = (
    confirmation_eval[
        "PredictionCorrect"
    ]
    .fillna(False)
    .astype(bool)
)


# ============================================================
# 9. Final confirmation metrics
# ============================================================

confirmation_coverage = (
    confirmation_eval[
        "CorrectCandidatePresent"
    ].mean()
)


confirmation_overall_accuracy = (
    confirmation_eval[
        "PredictionCorrect"
    ].mean()
)


confirmation_seen_accuracy = (
    confirmation_eval.loc[
        confirmation_eval[
            "AnimalSeenInFreshReference"
        ],
        "PredictionCorrect",
    ].mean()
)


confirmation_conditional_accuracy = (
    confirmation_eval.loc[
        confirmation_eval[
            "CorrectCandidatePresent"
        ],
        "PredictionCorrect",
    ].mean()
)


confirmation_correct = int(
    confirmation_eval[
        "PredictionCorrect"
    ].sum()
)


print("\n" + "=" * 72)

print(
    "CONFIRMATION RESULTS — 2,000 NEW DEV QUERIES"
)

print("=" * 72)

print(
    "AnimalId seen rate:",
    f"{confirmation_eval['AnimalSeenInFreshReference'].mean():.2%}"
)

print(
    "Top-160 candidate coverage:",
    f"{confirmation_coverage:.2%}"
)

print(
    "Overall accuracy:",
    f"{confirmation_overall_accuracy:.2%}"
)

print(
    "Seen-AnimalId accuracy:",
    f"{confirmation_seen_accuracy:.2%}"
)

print(
    "Conditional ranking accuracy:",
    f"{confirmation_conditional_accuracy:.2%}"
)

print(
    "Correct predictions:",
    confirmation_correct,
    "/",
    len(confirmation_eval)
)

Confirmation model matrix: (319903, 52)

CONFIRMATION RESULTS — 2,000 NEW DEV QUERIES
AnimalId seen rate: 94.40%
Top-160 candidate coverage: 89.25%
Overall accuracy: 53.90%
Seen-AnimalId accuracy: 57.10%
Conditional ranking accuracy: 60.39%
Correct predictions: 1078 / 2000


In [67]:
# ============================================================
# FREEZE CURRENT CHAMPION MODEL
#
# Fresh multi-snapshot + Top160 + HGB
# Confirmed on 2,000 untouched internal DEV queries
# ============================================================

import joblib
import json


CHAMPION_MODEL_PATH = (
    CACHE_DIR
    /
    "champion_fresh_top160_hgb.joblib"
)

CHAMPION_FEATURES_PATH = (
    CACHE_DIR
    /
    "champion_fresh_top160_features.json"
)

CHAMPION_METADATA_PATH = (
    CACHE_DIR
    /
    "champion_fresh_top160_metadata.json"
)


# ============================================================
# 1. Save trained HGB
# ============================================================

joblib.dump(
    fresh_hgb160,
    CHAMPION_MODEL_PATH
)


# ============================================================
# 2. Save exact feature order
# ============================================================

with open(
    CHAMPION_FEATURES_PATH,
    "w"
) as f:

    json.dump(
        FRESH_RERANKER_FEATURES,
        f,
        indent=2,
    )


# ============================================================
# 3. Save experimental metadata
# ============================================================

champion_metadata = {

    "model":
        "HistGradientBoostingClassifier",

    "candidate_top_k":
        160,

    "learning_rate":
        0.05,

    "max_iter":
        400,

    "iterations_used":
        int(
            fresh_hgb160.n_iter_
        ),

    "max_leaf_nodes":
        7,

    "min_samples_leaf":
        10,

    "l2_regularization":
        1.0,

    "training_queries":
        2000,

    "training_candidate_coverage":
        0.8885,

    "pilot_dev_queries":
        500,

    "pilot_overall_accuracy":
        0.5280,

    "confirmation_dev_queries":
        2000,

    "confirmation_seen_rate":
        0.9440,

    "confirmation_candidate_coverage":
        0.8925,

    "confirmation_overall_accuracy":
        0.5390,

    "confirmation_seen_accuracy":
        0.5710,

    "confirmation_conditional_accuracy":
        0.6039,

    "confirmation_correct_predictions":
        1078,

    "fresh_dev_reference_cutoff":
        "2020-09-20",
}


with open(
    CHAMPION_METADATA_PATH,
    "w"
) as f:

    json.dump(
        champion_metadata,
        f,
        indent=2,
    )


print("Champion model frozen.")

print(
    "Model:",
    CHAMPION_MODEL_PATH
)

print(
    "Features:",
    CHAMPION_FEATURES_PATH
)

print(
    "Metadata:",
    CHAMPION_METADATA_PATH
)

Champion model frozen.
Model: xgboost_cache/champion_fresh_top160_hgb.joblib
Features: xgboost_cache/champion_fresh_top160_features.json
Metadata: xgboost_cache/champion_fresh_top160_metadata.json


In [69]:
# ============================================================
# OUTER SPLIT AUDIT — FIXED VERSION
# ============================================================

outer_split_candidates = []


# Make a static copy first
global_items = list(globals().items())


for name, obj in global_items:

    if not isinstance(obj, pd.DataFrame):
        continue

    if "AnimalId" not in obj.columns:
        continue

    if "EventDate" not in obj.columns:
        continue

    # Focus on substantial datasets
    if len(obj) < 100_000:
        continue

    event_dates = pd.to_datetime(
        obj["EventDate"],
        errors="coerce"
    )

    outer_split_candidates.append({

        "Variable":
            name,

        "Rows":
            len(obj),

        "StartDate":
            event_dates.min(),

        "EndDate":
            event_dates.max(),

        "UniqueAnimalIds":
            obj["AnimalId"].nunique(),

        "MissingAnimalId":
            int(
                obj["AnimalId"].isna().sum()
            ),
    })


outer_split_candidates_df = (
    pd.DataFrame(
        outer_split_candidates
    )
    .sort_values(
        "Rows",
        ascending=False,
    )
    .reset_index(drop=True)
)


display(
    outer_split_candidates_df
)

,Variable,Rows,StartDate,EndDate,UniqueAnimalIds,MissingAnimalId
0,data,6645172,2019-06-14,2021-10-24,9069,0
1,labelled_data,6645172,2019-06-14,2021-10-24,9069,0
2,obj,6645172,2019-06-14,2021-10-24,9069,0
3,data_train,3985088,2019-06-14,2020-12-13,7539,0
4,reranker_base,3985088,2019-06-14,2020-12-13,7539,0
5,fresh_dev_reference_system,3379883,2019-06-14,2020-09-20,7111,0
6,fresh_dev_reference,3379883,2019-06-14,2020-09-20,7111,0
7,fresh_train_reference,3198518,2019-06-14,2020-08-27,6928,0
8,cutoff_reference,3149078,2019-06-14,2020-08-20,6903,0
9,reference_df,3149078,2019-06-14,2020-08-20,6903,0


In [70]:
# ============================================================
# OUTER VALIDATION SAMPLE
#
# 2,000 queries evenly distributed across the ENTIRE
# outer validation period.
#
# Validation:
#   2020-12-14 → 2021-05-20
#
# No model tuning will be done on this sample.
# ============================================================

import numpy as np
import pandas as pd


OUTER_VALIDATION_SAMPLE_SIZE = 2000


# ============================================================
# 1. Make sure EventDate is datetime
# ============================================================

data_validation = (
    data_validation
    .copy()
)

data_validation[
    "EventDate"
] = pd.to_datetime(
    data_validation[
        "EventDate"
    ]
)


# ============================================================
# 2. Sort chronologically
# ============================================================

validation_sorted = (
    data_validation
    .sort_values(
        "EventDate"
    )
)


# ============================================================
# 3. Uniformly sample across the whole time period
# ============================================================

validation_positions = (
    np.linspace(
        0,
        len(validation_sorted) - 1,
        OUTER_VALIDATION_SAMPLE_SIZE,
    )
    .round()
    .astype(int)
)


outer_validation_queries = (
    validation_sorted
    .iloc[
        validation_positions
    ]
    .copy()
)


# ============================================================
# 4. Safety checks
# ============================================================

assert (
    len(outer_validation_queries)
    ==
    OUTER_VALIDATION_SAMPLE_SIZE
)

assert (
    outer_validation_queries[
        "EventDate"
    ].min()
    >
    data_train[
        "EventDate"
    ].max()
)


# ============================================================
# 5. Check how many validation AnimalIds are already known
#    from TRAIN only
# ============================================================

train_animal_ids = set(
    data_train[
        "AnimalId"
    ]
    .dropna()
    .astype(object)
)


outer_validation_seen_rate = (
    outer_validation_queries[
        "AnimalId"
    ]
    .astype(object)
    .isin(
        train_animal_ids
    )
    .mean()
)


# ============================================================
# 6. Diagnostics
# ============================================================

print("=" * 70)

print(
    "Outer validation queries:",
    len(
        outer_validation_queries
    )
)

print(
    "Date range:",
    outer_validation_queries[
        "EventDate"
    ].min(),
    "→",
    outer_validation_queries[
        "EventDate"
    ].max()
)

print(
    "TRAIN reference cutoff:",
    data_train[
        "EventDate"
    ].max()
)

print(
    "AnimalId seen in TRAIN:",
    f"{outer_validation_seen_rate:.2%}"
)

print(
    "Unique AnimalIds in sample:",
    outer_validation_queries[
        "AnimalId"
    ].nunique()
)

Outer validation queries: 2000
Date range: 2020-12-14 00:00:00 → 2021-05-20 00:00:00
TRAIN reference cutoff: 2020-12-13 00:00:00
AnimalId seen in TRAIN: 92.20%
Unique AnimalIds in sample: 1639


In [71]:
# ============================================================
# OUTER VALIDATION REFERENCE SYSTEM
#
# Reference:
#   data_train ONLY
#
# Cutoff:
#   2020-12-13
#
# Validation starts:
#   2020-12-14
#
# NO validation labels are used in the reference.
# ============================================================

import gc


OUTER_VAL_CUTOFF = pd.Timestamp(
    "2020-12-13"
)


# ============================================================
# 1. Reference dataset
# ============================================================

outer_val_reference_system = data_train


# Make sure EventDate is datetime
if not pd.api.types.is_datetime64_any_dtype(
    outer_val_reference_system["EventDate"]
):

    outer_val_reference_system = (
        outer_val_reference_system.copy()
    )

    outer_val_reference_system[
        "EventDate"
    ] = pd.to_datetime(
        outer_val_reference_system[
            "EventDate"
        ]
    )


# ============================================================
# 2. Leakage safety checks
# ============================================================

assert (
    outer_val_reference_system[
        "EventDate"
    ].max()
    ==
    OUTER_VAL_CUTOFF
)


assert (
    outer_validation_queries[
        "EventDate"
    ].min()
    >
    OUTER_VAL_CUTOFF
)


print(
    "Outer validation reference rows:",
    f"{len(outer_val_reference_system):,}"
)

print(
    "Reference date range:",
    outer_val_reference_system[
        "EventDate"
    ].min(),
    "→",
    outer_val_reference_system[
        "EventDate"
    ].max()
)


# ============================================================
# 3. Class mapping
# ============================================================

outer_val_class_values = pd.Index(
    outer_val_reference_system[
        "AnimalId"
    ]
    .astype(object)
    .unique()
)


outer_val_class_to_number = {
    animal_id: class_number
    for class_number, animal_id
    in enumerate(
        outer_val_class_values
    )
}


# ============================================================
# 4. Origin date
# ============================================================

outer_val_origin_date = (
    outer_val_reference_system[
        "EventDate"
    ].min()
)


# ============================================================
# 5. Structural profiles
# ============================================================

(
    outer_val_profiles,
    outer_val_global_lactation_interval,
) = build_structural_profiles(
    outer_val_reference_system,
    outer_val_class_to_number,
    outer_val_origin_date,
    include_projection=True,
)


outer_val_profile_lookup = (
    make_profile_lookup(
        outer_val_profiles
    )
)


# ============================================================
# 6. Sample reference rows
# ============================================================

outer_val_reference_indices = (
    sample_reference_rows(
        outer_val_reference_system,
        max_rows_per_group=
            REFERENCE_ROWS_PER_GROUP,
        recent_fraction=0.80,
        random_state=RANDOM_STATE,
    )
)


outer_val_reference_raw = (
    outer_val_reference_system
    .loc[
        outer_val_reference_indices
    ]
    .copy()
)


print(
    "Sampled reference rows:",
    f"{len(outer_val_reference_raw):,}"
)


# ============================================================
# 7. Distance preprocessing
# ============================================================

(
    X_outer_val_reference,
    outer_val_numeric_columns,
    outer_val_numeric_fill,
    outer_val_scaler,
    outer_val_reproduction_columns,
    outer_val_feature_names,
) = fit_distance_preprocessor(
    outer_val_reference_raw
)


# ============================================================
# 8. Reference metadata
# ============================================================

outer_val_reference_class_ids = (
    outer_val_reference_raw[
        "AnimalId"
    ]
    .astype(object)
    .map(
        outer_val_class_to_number
    )
    .astype(np.int32)
    .to_numpy()
)


outer_val_reference_lactations = (
    outer_val_reference_raw[
        "LactationNumber"
    ]
    .round()
    .astype(np.int16)
    .to_numpy()
)


outer_val_reference_dim = (
    outer_val_reference_raw[
        "DaysInMilk"
    ]
    .astype(np.float32)
    .to_numpy()
)


# ============================================================
# 9. Reference indices
# ============================================================

(
    outer_val_reference_exact_index,
    outer_val_reference_class_lactations,
) = build_reference_index(
    outer_val_reference_class_ids,
    outer_val_reference_lactations,
)


# ============================================================
# 10. Whole-cow prototypes
# ============================================================

outer_val_cow_prototype = {}


for class_id in np.unique(
    outer_val_reference_class_ids
):

    positions = np.flatnonzero(
        outer_val_reference_class_ids
        ==
        class_id
    )

    outer_val_cow_prototype[
        int(class_id)
    ] = np.median(
        X_outer_val_reference[
            positions
        ],
        axis=0,
    ).astype(np.float32)


# ============================================================
# 11. Lactation + DIM-bin prototypes
# ============================================================

outer_val_reference_dim_bin = np.floor(
    outer_val_reference_dim
    /
    DIM_BIN_WIDTH
).astype(np.int16)


outer_val_lactation_prototype = {}


for (
    class_id,
    lactation_number
), positions in (
    outer_val_reference_exact_index.items()
):

    bins_here = (
        outer_val_reference_dim_bin[
            positions
        ]
    )

    prototype_by_bin = {}


    for dim_bin in np.unique(
        bins_here
    ):

        bin_positions = (
            positions[
                bins_here
                ==
                dim_bin
            ]
        )


        prototype_by_bin[
            int(dim_bin)
        ] = np.median(
            X_outer_val_reference[
                bin_positions
            ],
            axis=0,
        ).astype(np.float32)


    outer_val_lactation_prototype[
        (
            int(class_id),
            int(lactation_number),
        )
    ] = prototype_by_bin


# ============================================================
# 12. Validation sample seen rate
# ============================================================

outer_val_seen_rate_check = (
    outer_validation_queries[
        "AnimalId"
    ]
    .astype(object)
    .isin(
        outer_val_class_values
    )
    .mean()
)


# ============================================================
# 13. Diagnostics
# ============================================================

print("\n" + "=" * 70)

print(
    "Outer validation classes:",
    f"{len(outer_val_class_values):,}"
)

print(
    "Structural profiles:",
    f"{len(outer_val_profiles):,}"
)

print(
    "Reference matrix:",
    X_outer_val_reference.shape
)

print(
    "Distance features:",
    len(
        outer_val_feature_names
    )
)

print(
    "AnimalId-lactation groups:",
    f"{len(outer_val_reference_exact_index):,}"
)

print(
    "Whole-cow prototypes:",
    f"{len(outer_val_cow_prototype):,}"
)

print(
    "Lactation/DIM prototype groups:",
    f"{len(outer_val_lactation_prototype):,}"
)

print(
    "Outer validation AnimalId seen rate:",
    f"{outer_val_seen_rate_check:.2%}"
)

print(
    "Reference matrix memory:",
    f"{X_outer_val_reference.nbytes / 1024**2:.2f} MB"
)

Outer validation reference rows: 3,985,088
Reference date range: 2019-06-14 00:00:00 → 2020-12-13 00:00:00
Sampled reference rows: 2,323,643

Outer validation classes: 7,539
Structural profiles: 19,733
Reference matrix: (2323643, 13)
Distance features: 13
AnimalId-lactation groups: 12,196
Whole-cow prototypes: 7,539
Lactation/DIM prototype groups: 12,196
Outer validation AnimalId seen rate: 92.20%
Reference matrix memory: 115.23 MB


In [72]:
# ============================================================
# OUTER VALIDATION CANDIDATE GENERATION
#
# Reference:
#   data_train through 2020-12-13 ONLY
#
# Queries:
#   2,000 uniformly sampled outer validation queries
#
# Candidate generation:
#   Global Top-320
#
# Model evaluation:
#   NOT YET
# ============================================================

OUTER_VAL_PREFIX = "outerval_query"

OUTER_VAL_CACHE_PATH = (
    CACHE_DIR
    /
    "outer_validation_candidates_2000_global320.pkl"
)


# ============================================================
# 1. Map reranker globals to OUTER VALIDATION reference
# ============================================================

outer_val_globals = {

    "reranker_class_to_number":
        outer_val_class_to_number,

    "reranker_cow_prototype":
        outer_val_cow_prototype,

    "reranker_feature_names":
        outer_val_feature_names,

    "reranker_lactation_prototype":
        outer_val_lactation_prototype,

    "reranker_numeric_columns":
        outer_val_numeric_columns,

    "reranker_numeric_fill":
        outer_val_numeric_fill,

    "reranker_origin_date":
        outer_val_origin_date,

    "reranker_profile_lookup":
        outer_val_profile_lookup,

    "reranker_reference_class_lactations":
        outer_val_reference_class_lactations,

    "reranker_reference_dim":
        outer_val_reference_dim,

    "reranker_reference_exact_index":
        outer_val_reference_exact_index,

    "reranker_reproduction_columns":
        outer_val_reproduction_columns,

    "reranker_scaler":
        outer_val_scaler,

    "X_reranker_reference":
        X_outer_val_reference,
}


# ============================================================
# 2. Backup current globals
# ============================================================

outer_val_old_globals = {
    name: globals()[name]
    for name in outer_val_globals
}


# ============================================================
# 3. Generate / load Global Top-320 candidates
# ============================================================

if OUTER_VAL_CACHE_PATH.exists():

    print(
        "Loading cached outer validation candidates..."
    )

    outer_val_candidate_pairs = (
        pd.read_pickle(
            OUTER_VAL_CACHE_PATH
        )
    )

else:

    try:

        # ----------------------------------------------------
        # Activate outer validation reference
        # ----------------------------------------------------

        for name, value in (
            outer_val_globals.items()
        ):

            globals()[name] = value


        print(
            "Outer validation reference activated."
        )

        print(
            "Classes:",
            len(
                reranker_class_to_number
            )
        )

        print(
            "Reference matrix:",
            X_reranker_reference.shape
        )

        print(
            "Queries:",
            len(
                outer_validation_queries
            )
        )

        print(
            "Global candidate maximum:",
            RERANKER_CANDIDATE_PARAMS[
                "max_candidates"
            ]
        )


        # ----------------------------------------------------
        # Build candidates
        # ----------------------------------------------------

        outer_val_candidate_pairs = (
            build_candidate_dataset(
                outer_validation_queries,
                prefix=OUTER_VAL_PREFIX,
                progress_every=100,
            )
        )


        # ----------------------------------------------------
        # Save cache
        # ----------------------------------------------------

        outer_val_candidate_pairs.to_pickle(
            OUTER_VAL_CACHE_PATH
        )


        print(
            "Saved candidate cache:",
            OUTER_VAL_CACHE_PATH
        )


    finally:

        # ----------------------------------------------------
        # ALWAYS restore original reranker globals
        # ----------------------------------------------------

        for name, value in (
            outer_val_old_globals.items()
        ):

            globals()[name] = value


        print(
            "Original reranker globals restored."
        )


print()

print(
    "Candidate rows:",
    f"{len(outer_val_candidate_pairs):,}"
)

print(
    "Queries generated:",
    outer_val_candidate_pairs[
        "QueryId"
    ].nunique()
)

Outer validation reference activated.
Classes: 7539
Reference matrix: (2323643, 13)
Queries: 2000
Global candidate maximum: 320
outerval_query: processed 100 / 2000 queries
outerval_query: processed 200 / 2000 queries
outerval_query: processed 300 / 2000 queries
outerval_query: processed 400 / 2000 queries
outerval_query: processed 500 / 2000 queries
outerval_query: processed 600 / 2000 queries
outerval_query: processed 700 / 2000 queries
outerval_query: processed 800 / 2000 queries
outerval_query: processed 900 / 2000 queries
outerval_query: processed 1000 / 2000 queries
outerval_query: processed 1100 / 2000 queries
outerval_query: processed 1200 / 2000 queries
outerval_query: processed 1300 / 2000 queries
outerval_query: processed 1400 / 2000 queries
outerval_query: processed 1500 / 2000 queries
outerval_query: processed 1600 / 2000 queries
outerval_query: processed 1700 / 2000 queries
outerval_query: processed 1800 / 2000 queries
outerval_query: processed 1900 / 2000 queries
outerva

In [73]:
# ============================================================
# OUTER VALIDATION TOP-160 COVERAGE
#
# IMPORTANT:
# No HGB prediction yet.
# ============================================================

OUTER_MODEL_K = 160


# ============================================================
# 1. Restrict to SAME Top-160 used by champion model
# ============================================================

outer_val160 = (
    outer_val_candidate_pairs.loc[
        outer_val_candidate_pairs[
            "CandidateStartRank"
        ]
        <= OUTER_MODEL_K
    ]
    .copy()
    .reset_index(drop=True)
)


# ============================================================
# 2. Correct candidate presence
# ============================================================

outer_val160_presence = (
    outer_val160
    .groupby("QueryId")[
        "IsCorrectCandidate"
    ]
    .max()
)


# ============================================================
# 3. Build expected query IDs
# ============================================================

outer_val_query_ids = [
    f"{OUTER_VAL_PREFIX}_{idx}"
    for idx in outer_validation_queries.index
]


outer_val_eval = (
    outer_validation_queries[
        ["AnimalId", "EventDate"]
    ]
    .copy()
)


outer_val_eval[
    "QueryId"
] = outer_val_query_ids


# ============================================================
# 4. Was true AnimalId seen in TRAIN?
# ============================================================

outer_val_eval[
    "AnimalSeenInTrain"
] = (
    outer_val_eval[
        "AnimalId"
    ]
    .astype(object)
    .isin(
        outer_val_class_values
    )
)


# ============================================================
# 5. Was true cow inside Top-160?
# ============================================================

outer_val_eval[
    "CorrectCandidatePresent"
] = (
    outer_val_eval[
        "QueryId"
    ]
    .map(
        outer_val160_presence
    )
    .fillna(0)
    .astype(bool)
)


# ============================================================
# 6. Metrics
# ============================================================

outer_val160_coverage = (
    outer_val_eval[
        "CorrectCandidatePresent"
    ].mean()
)


outer_val160_seen_coverage = (
    outer_val_eval.loc[
        outer_val_eval[
            "AnimalSeenInTrain"
        ],
        "CorrectCandidatePresent",
    ].mean()
)


outer_val160_counts = (
    outer_val160
    .groupby("QueryId")
    .size()
)


print("\n" + "=" * 72)

print(
    "OUTER VALIDATION — TOP-160 RETRIEVAL"
)

print("=" * 72)

print(
    "Queries:",
    len(
        outer_val_eval
    )
)

print(
    "AnimalId seen in TRAIN:",
    f"{outer_val_eval['AnimalSeenInTrain'].mean():.2%}"
)

print(
    "Top-160 candidate coverage:",
    f"{outer_val160_coverage:.2%}"
)

print(
    "Seen-AnimalId candidate coverage:",
    f"{outer_val160_seen_coverage:.2%}"
)

print(
    "Correct candidate present:",
    int(
        outer_val_eval[
            "CorrectCandidatePresent"
        ].sum()
    ),
    "/",
    len(
        outer_val_eval
    )
)

print(
    "Mean candidates/query:",
    f"{outer_val160_counts.mean():.2f}"
)


OUTER VALIDATION — TOP-160 RETRIEVAL
Queries: 2000
AnimalId seen in TRAIN: 92.20%
Top-160 candidate coverage: 82.05%
Seen-AnimalId candidate coverage: 88.99%
Correct candidate present: 1641 / 2000
Mean candidates/query: 159.91


In [74]:
# ============================================================
# OUTER VALIDATION — FROZEN CHAMPION EVALUATION
#
# IMPORTANT:
#   - Load frozen model from disk
#   - Load frozen feature order from disk
#   - SAME Top-160
#   - NO retraining
#   - NO tuning
# ============================================================

import joblib
import json
import numpy as np


# ============================================================
# 1. Reload frozen champion
# ============================================================

frozen_champion = joblib.load(
    CACHE_DIR
    /
    "champion_fresh_top160_hgb.joblib"
)


with open(
    CACHE_DIR
    /
    "champion_fresh_top160_features.json",
    "r"
) as f:

    frozen_feature_order = json.load(f)


print(
    "Frozen model loaded."
)

print(
    "Frozen features:",
    len(frozen_feature_order)
)


# ============================================================
# 2. Recompute CandidateCount for Top-160 only
# ============================================================

outer_val160[
    "CandidateCount"
] = (
    outer_val160
    .groupby("QueryId")[
        "QueryId"
    ]
    .transform("size")
    .astype(np.int16)
)


# ============================================================
# 3. Recompute rank features inside Top-160
# ============================================================

outer_val160_ranked = (
    add_candidate_rank_features(
        outer_val160
    )
)


# ============================================================
# 4. Exact feature consistency check
# ============================================================

missing_outer_features = [
    feature
    for feature in frozen_feature_order
    if feature
    not in outer_val160_ranked.columns
]


assert (
    len(missing_outer_features)
    ==
    0
), (
    f"Missing frozen features: "
    f"{missing_outer_features}"
)


# ============================================================
# 5. Build model matrix
# ============================================================

X_outer_val160 = (
    outer_val160_ranked[
        frozen_feature_order
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .astype(np.float32)
)


print(
    "Outer validation matrix:",
    X_outer_val160.shape
)


# ============================================================
# 6. Frozen-model prediction
# ============================================================

outer_val_prob160 = (
    frozen_champion.predict_proba(
        X_outer_val160
    )[:, 1]
)


outer_val_scored160 = (
    outer_val160_ranked[
        [
            "QueryId",
            "CandidateClassId",
            "TrueClassId",
            "IsCorrectCandidate",
        ]
    ]
    .copy()
)


outer_val_scored160[
    "Probability"
] = outer_val_prob160


# ============================================================
# 7. Pick highest-probability candidate per query
# ============================================================

outer_val_best160 = (
    outer_val_scored160
    .sort_values(
        [
            "QueryId",
            "Probability",
        ],
        ascending=[
            True,
            False,
        ]
    )
    .groupby(
        "QueryId",
        as_index=False,
    )
    .first()
)


outer_val_best160[
    "PredictionCorrect"
] = (
    outer_val_best160[
        "CandidateClassId"
    ]
    ==
    outer_val_best160[
        "TrueClassId"
    ]
)


# ============================================================
# 8. Merge predictions into query-level evaluation
# ============================================================

outer_val_final_eval = (
    outer_val_eval
    .merge(
        outer_val_best160[
            [
                "QueryId",
                "PredictionCorrect",
            ]
        ],
        on="QueryId",
        how="left",
    )
)


outer_val_final_eval[
    "PredictionCorrect"
] = (
    outer_val_final_eval[
        "PredictionCorrect"
    ]
    .fillna(False)
    .astype(bool)
)


# ============================================================
# 9. Final metrics
# ============================================================

outer_val_overall_accuracy = (
    outer_val_final_eval[
        "PredictionCorrect"
    ].mean()
)


outer_val_seen_accuracy = (
    outer_val_final_eval.loc[
        outer_val_final_eval[
            "AnimalSeenInTrain"
        ],
        "PredictionCorrect",
    ].mean()
)


outer_val_conditional_accuracy = (
    outer_val_final_eval.loc[
        outer_val_final_eval[
            "CorrectCandidatePresent"
        ],
        "PredictionCorrect",
    ].mean()
)


outer_val_correct = int(
    outer_val_final_eval[
        "PredictionCorrect"
    ].sum()
)


print("\n" + "=" * 72)

print(
    "OUTER VALIDATION — FROZEN CHAMPION"
)

print("=" * 72)

print(
    "AnimalId seen in TRAIN:",
    f"{outer_val_final_eval['AnimalSeenInTrain'].mean():.2%}"
)

print(
    "Top-160 candidate coverage:",
    f"{outer_val_final_eval['CorrectCandidatePresent'].mean():.2%}"
)

print(
    "Overall accuracy:",
    f"{outer_val_overall_accuracy:.2%}"
)

print(
    "Seen-AnimalId accuracy:",
    f"{outer_val_seen_accuracy:.2%}"
)

print(
    "Conditional ranking accuracy:",
    f"{outer_val_conditional_accuracy:.2%}"
)

print(
    "Correct predictions:",
    outer_val_correct,
    "/",
    len(
        outer_val_final_eval
    )
)

Frozen model loaded.
Frozen features: 52
Outer validation matrix: (319826, 52)

OUTER VALIDATION — FROZEN CHAMPION
AnimalId seen in TRAIN: 92.20%
Top-160 candidate coverage: 82.05%
Overall accuracy: 38.50%
Seen-AnimalId accuracy: 41.76%
Conditional ranking accuracy: 46.92%
Correct predictions: 770 / 2000


In [75]:
# ============================================================
# OUTER VALIDATION TEMPORAL DRIFT DIAGNOSTIC
#
# Measure model performance as reference becomes older.
#
# Reference cutoff:
#   2020-12-13
#
# No retraining.
# ============================================================

temporal_eval = (
    outer_val_final_eval
    .copy()
)


# ============================================================
# 1. Days since fixed TRAIN reference cutoff
# ============================================================

temporal_eval[
    "DaysSinceReferenceCutoff"
] = (
    pd.to_datetime(
        temporal_eval["EventDate"]
    )
    -
    OUTER_VAL_CUTOFF
).dt.days


# ============================================================
# 2. Freshness windows
# ============================================================

freshness_bins = [
    0,
    30,
    60,
    90,
    120,
    160,
    9999,
]

freshness_labels = [
    "1-30 days",
    "31-60 days",
    "61-90 days",
    "91-120 days",
    "121-160 days",
    ">160 days",
]


temporal_eval[
    "FreshnessWindow"
] = pd.cut(
    temporal_eval[
        "DaysSinceReferenceCutoff"
    ],
    bins=freshness_bins,
    labels=freshness_labels,
    include_lowest=True,
    right=True,
)


# ============================================================
# 3. Calculate metrics by freshness window
# ============================================================

temporal_results = []


for window in freshness_labels:

    subset = (
        temporal_eval.loc[
            temporal_eval[
                "FreshnessWindow"
            ]
            ==
            window
        ]
        .copy()
    )

    if len(subset) == 0:
        continue


    seen_rate = (
        subset[
            "AnimalSeenInTrain"
        ].mean()
    )


    coverage = (
        subset[
            "CorrectCandidatePresent"
        ].mean()
    )


    overall_accuracy = (
        subset[
            "PredictionCorrect"
        ].mean()
    )


    # Conditional ranking:
    # only queries where true candidate exists
    candidate_present_subset = (
        subset.loc[
            subset[
                "CorrectCandidatePresent"
            ]
        ]
    )


    if len(
        candidate_present_subset
    ) > 0:

        conditional_accuracy = (
            candidate_present_subset[
                "PredictionCorrect"
            ].mean()
        )

    else:

        conditional_accuracy = np.nan


    # Seen-AnimalId accuracy
    seen_subset = (
        subset.loc[
            subset[
                "AnimalSeenInTrain"
            ]
        ]
    )


    if len(seen_subset) > 0:

        seen_accuracy = (
            seen_subset[
                "PredictionCorrect"
            ].mean()
        )

    else:

        seen_accuracy = np.nan


    temporal_results.append({

        "FreshnessWindow":
            window,

        "Queries":
            len(subset),

        "MeanDaysSinceCutoff":
            subset[
                "DaysSinceReferenceCutoff"
            ].mean(),

        "AnimalSeenRate":
            seen_rate * 100,

        "CandidateCoverage":
            coverage * 100,

        "ConditionalAccuracy":
            conditional_accuracy * 100,

        "SeenAccuracy":
            seen_accuracy * 100,

        "OverallAccuracy":
            overall_accuracy * 100,
    })


outer_temporal_drift_df = (
    pd.DataFrame(
        temporal_results
    )
)


display(
    outer_temporal_drift_df
)

,FreshnessWindow,Queries,MeanDaysSinceCutoff,AnimalSeenRate,CandidateCoverage,ConditionalAccuracy,SeenAccuracy,OverallAccuracy
0,1-30 days,394,15.497462,98.730964,96.192893,74.670185,72.750643,71.827411
1,31-60 days,375,45.410667,96.266667,89.600000,56.547619,52.631579,50.666667
2,61-90 days,384,75.450521,92.708333,84.375000,37.654321,34.269663,31.770833
3,91-120 days,361,105.759003,88.365651,77.285319,31.182796,27.272727,24.099723
4,121-160 days,486,139.788066,86.213992,66.460905,27.244582,21.002387,18.106996


In [76]:
# ============================================================
# LONG-GAP MULTI-SNAPSHOT TRAINING PLAN
#
# Goal:
#   Train the reranker on the SAME reference-age pattern
#   observed in outer validation.
#
# IMPORTANT:
#   - Uses data_train ONLY
#   - Does NOT use data_validation labels for training
#   - Does NOT touch data_test
#   - No candidate generation yet
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. Historical snapshot cutoffs
#
# All are early enough that we can observe queries up to
# 160 days later while remaining inside data_train.
# ============================================================

LONGGAP_SNAPSHOT_DATES = [
    pd.Timestamp("2020-04-01"),
    pd.Timestamp("2020-05-01"),
    pd.Timestamp("2020-06-01"),
    pd.Timestamp("2020-07-01"),
]


# ============================================================
# 2. Match OUTER VALIDATION freshness distribution exactly
# ============================================================

LONGGAP_TARGETS = {
    "1-30 days": 394,
    "31-60 days": 375,
    "61-90 days": 384,
    "91-120 days": 361,
    "121-160 days": 486,
}


LONGGAP_BOUNDS = {
    "1-30 days": (1, 30),
    "31-60 days": (31, 60),
    "61-90 days": (61, 90),
    "91-120 days": (91, 120),
    "121-160 days": (121, 160),
}


# ============================================================
# 3. Prepare training data
# ============================================================

longgap_base = data_train.copy()

longgap_base["EventDate"] = pd.to_datetime(
    longgap_base["EventDate"]
)


rng = np.random.default_rng(
    RANDOM_STATE
)


selected_rows = []

used_query_indices = set()


# ============================================================
# 4. Sample each freshness bin across multiple snapshots
# ============================================================

for freshness_window, target_n in LONGGAP_TARGETS.items():

    low_day, high_day = (
        LONGGAP_BOUNDS[
            freshness_window
        ]
    )


    # Split target approximately evenly across snapshots
    base_n = (
        target_n
        //
        len(LONGGAP_SNAPSHOT_DATES)
    )

    remainder = (
        target_n
        %
        len(LONGGAP_SNAPSHOT_DATES)
    )


    snapshot_targets = [
        base_n
        +
        (
            1
            if i < remainder
            else 0
        )
        for i in range(
            len(LONGGAP_SNAPSHOT_DATES)
        )
    ]


    for snapshot_date, snapshot_target in zip(
        LONGGAP_SNAPSHOT_DATES,
        snapshot_targets,
    ):

        # ----------------------------------------------------
        # Calculate query lag relative to THIS snapshot
        # ----------------------------------------------------

        lag_days = (
            longgap_base["EventDate"]
            -
            snapshot_date
        ).dt.days


        eligible_mask = (
            lag_days.between(
                low_day,
                high_day,
                inclusive="both",
            )
        )


        eligible = (
            longgap_base.loc[
                eligible_mask
            ]
            .copy()
        )


        # Do not reuse the same original query twice
        eligible = (
            eligible.loc[
                ~eligible.index.isin(
                    used_query_indices
                )
            ]
        )


        if len(eligible) < snapshot_target:

            raise ValueError(
                f"Not enough eligible rows for "
                f"{snapshot_date.date()} / "
                f"{freshness_window}: "
                f"need {snapshot_target}, "
                f"have {len(eligible)}"
            )


        sampled_indices = rng.choice(
            eligible.index.to_numpy(),
            size=snapshot_target,
            replace=False,
        )


        sampled = (
            longgap_base.loc[
                sampled_indices
            ]
            .copy()
        )


        sampled[
            "OriginalQueryIndex"
        ] = sampled.index


        sampled[
            "SnapshotDate"
        ] = snapshot_date


        sampled[
            "DaysSinceSnapshot"
        ] = (
            sampled["EventDate"]
            -
            snapshot_date
        ).dt.days


        sampled[
            "FreshnessWindow"
        ] = freshness_window


        selected_rows.append(
            sampled
        )


        used_query_indices.update(
            sampled_indices.tolist()
        )


# ============================================================
# 5. Combine selected queries
# ============================================================

longgap_training_queries = (
    pd.concat(
        selected_rows,
        ignore_index=False,
    )
    .copy()
)


# ============================================================
# 6. Safety checks
# ============================================================

assert (
    len(longgap_training_queries)
    ==
    2000
)


assert (
    longgap_training_queries[
        "OriginalQueryIndex"
    ].nunique()
    ==
    2000
)


assert (
    (
        longgap_training_queries[
            "EventDate"
        ]
        >
        longgap_training_queries[
            "SnapshotDate"
        ]
    ).all()
)


assert (
    longgap_training_queries[
        "EventDate"
    ].max()
    <=
    data_train[
        "EventDate"
    ].max()
)


# ============================================================
# 7. Distribution check
# ============================================================

longgap_bin_summary = (
    longgap_training_queries
    .groupby(
        "FreshnessWindow",
        observed=True,
    )
    .agg(
        Queries=(
            "OriginalQueryIndex",
            "size",
        ),
        MeanDaysSinceSnapshot=(
            "DaysSinceSnapshot",
            "mean",
        ),
        MinDays=(
            "DaysSinceSnapshot",
            "min",
        ),
        MaxDays=(
            "DaysSinceSnapshot",
            "max",
        ),
        UniqueAnimalIds=(
            "AnimalId",
            "nunique",
        ),
    )
    .reindex(
        LONGGAP_TARGETS.keys()
    )
    .reset_index()
)


snapshot_summary = (
    longgap_training_queries
    .groupby(
        "SnapshotDate"
    )
    .agg(
        Queries=(
            "OriginalQueryIndex",
            "size",
        ),
        MinQueryDate=(
            "EventDate",
            "min",
        ),
        MaxQueryDate=(
            "EventDate",
            "max",
        ),
        UniqueAnimalIds=(
            "AnimalId",
            "nunique",
        ),
    )
    .reset_index()
)


print("=" * 72)

print(
    "LONG-GAP TRAINING PLAN"
)

print("=" * 72)

print(
    "Total queries:",
    len(
        longgap_training_queries
    )
)

print(
    "Unique original queries:",
    longgap_training_queries[
        "OriginalQueryIndex"
    ].nunique()
)

print(
    "Latest query date:",
    longgap_training_queries[
        "EventDate"
    ].max()
)

print(
    "data_train cutoff:",
    data_train[
        "EventDate"
    ].max()
)


print(
    "\nFreshness distribution:"
)

display(
    longgap_bin_summary
)


print(
    "\nSnapshot distribution:"
)

display(
    snapshot_summary
)

LONG-GAP TRAINING PLAN
Total queries: 2000
Unique original queries: 2000
Latest query date: 2020-12-08 00:00:00
data_train cutoff: 2020-12-13 00:00:00

Freshness distribution:


,FreshnessWindow,Queries,MeanDaysSinceSnapshot,MinDays,MaxDays,UniqueAnimalIds
0,1-30 days,394,16.187817,1,30,381
1,31-60 days,375,45.709333,31,60,357
2,61-90 days,384,75.604167,61,90,365
3,91-120 days,361,106.576177,91,120,347
4,121-160 days,486,140.368313,121,160,464



Snapshot distribution:


,SnapshotDate,Queries,MinQueryDate,MaxQueryDate,UniqueAnimalIds
0,2020-04-01,502,2020-04-03,2020-09-08,478
1,2020-05-01,501,2020-05-02,2020-09-30,464
2,2020-06-01,499,2020-06-02,2020-11-08,475
3,2020-07-01,498,2020-07-03,2020-12-08,479


In [77]:
# ============================================================
# LONG-GAP SNAPSHOT 1
#
# Snapshot cutoff:
#   2020-04-01
#
# Assigned training queries:
#   502
#
# Goal:
#   Build leakage-safe reference system only.
#   No model training yet.
# ============================================================

LONGGAP_TEST_SNAPSHOT = pd.Timestamp(
    "2020-04-01"
)


# ============================================================
# 1. Assigned queries
# ============================================================

longgap_test_queries = (
    longgap_training_queries.loc[
        longgap_training_queries[
            "SnapshotDate"
        ]
        ==
        LONGGAP_TEST_SNAPSHOT
    ]
    .copy()
)


# ============================================================
# 2. Reference = data_train up through snapshot cutoff
# ============================================================

longgap_snapshot_reference = (
    data_train.loc[
        pd.to_datetime(
            data_train["EventDate"]
        )
        <=
        LONGGAP_TEST_SNAPSHOT
    ]
    .copy()
)


longgap_snapshot_reference[
    "EventDate"
] = pd.to_datetime(
    longgap_snapshot_reference[
        "EventDate"
    ]
)


# ============================================================
# 3. Leakage safety
# ============================================================

assert (
    longgap_snapshot_reference[
        "EventDate"
    ].max()
    <=
    LONGGAP_TEST_SNAPSHOT
)


assert (
    longgap_test_queries[
        "EventDate"
    ].min()
    >
    LONGGAP_TEST_SNAPSHOT
)


# ============================================================
# 4. Class mapping
# ============================================================

longgap_snapshot_class_values = pd.Index(
    longgap_snapshot_reference[
        "AnimalId"
    ]
    .astype(object)
    .unique()
)


longgap_snapshot_class_to_number = {
    animal_id: class_number
    for class_number, animal_id
    in enumerate(
        longgap_snapshot_class_values
    )
}


# ============================================================
# 5. Origin date
# ============================================================

longgap_snapshot_origin_date = (
    longgap_snapshot_reference[
        "EventDate"
    ].min()
)


# ============================================================
# 6. Structural profiles
# ============================================================

(
    longgap_snapshot_profiles,
    longgap_snapshot_global_interval,
) = build_structural_profiles(
    longgap_snapshot_reference,
    longgap_snapshot_class_to_number,
    longgap_snapshot_origin_date,
    include_projection=True,
)


longgap_snapshot_profile_lookup = (
    make_profile_lookup(
        longgap_snapshot_profiles
    )
)


# ============================================================
# 7. Sample reference rows
# ============================================================

longgap_snapshot_reference_indices = (
    sample_reference_rows(
        longgap_snapshot_reference,
        max_rows_per_group=
            REFERENCE_ROWS_PER_GROUP,
        recent_fraction=0.80,
        random_state=RANDOM_STATE,
    )
)


longgap_snapshot_reference_raw = (
    longgap_snapshot_reference
    .loc[
        longgap_snapshot_reference_indices
    ]
    .copy()
)


# ============================================================
# 8. Distance preprocessing
# ============================================================

(
    X_longgap_snapshot_reference,
    longgap_snapshot_numeric_columns,
    longgap_snapshot_numeric_fill,
    longgap_snapshot_scaler,
    longgap_snapshot_reproduction_columns,
    longgap_snapshot_feature_names,
) = fit_distance_preprocessor(
    longgap_snapshot_reference_raw
)


# ============================================================
# 9. Metadata
# ============================================================

longgap_snapshot_reference_class_ids = (
    longgap_snapshot_reference_raw[
        "AnimalId"
    ]
    .astype(object)
    .map(
        longgap_snapshot_class_to_number
    )
    .astype(np.int32)
    .to_numpy()
)


longgap_snapshot_reference_lactations = (
    longgap_snapshot_reference_raw[
        "LactationNumber"
    ]
    .round()
    .astype(np.int16)
    .to_numpy()
)


longgap_snapshot_reference_dim = (
    longgap_snapshot_reference_raw[
        "DaysInMilk"
    ]
    .astype(np.float32)
    .to_numpy()
)


# ============================================================
# 10. Reference index
# ============================================================

(
    longgap_snapshot_reference_exact_index,
    longgap_snapshot_reference_class_lactations,
) = build_reference_index(
    longgap_snapshot_reference_class_ids,
    longgap_snapshot_reference_lactations,
)


# ============================================================
# 11. Whole-cow prototypes
# ============================================================

longgap_snapshot_cow_prototype = {}


for class_id in np.unique(
    longgap_snapshot_reference_class_ids
):

    positions = np.flatnonzero(
        longgap_snapshot_reference_class_ids
        ==
        class_id
    )

    longgap_snapshot_cow_prototype[
        int(class_id)
    ] = np.median(
        X_longgap_snapshot_reference[
            positions
        ],
        axis=0,
    ).astype(np.float32)


# ============================================================
# 12. Lactation / DIM-bin prototypes
# ============================================================

longgap_snapshot_reference_dim_bin = (
    np.floor(
        longgap_snapshot_reference_dim
        /
        DIM_BIN_WIDTH
    )
    .astype(np.int16)
)


longgap_snapshot_lactation_prototype = {}


for (
    class_id,
    lactation_number
), positions in (
    longgap_snapshot_reference_exact_index.items()
):

    bins_here = (
        longgap_snapshot_reference_dim_bin[
            positions
        ]
    )

    prototype_by_bin = {}


    for dim_bin in np.unique(
        bins_here
    ):

        bin_positions = (
            positions[
                bins_here
                ==
                dim_bin
            ]
        )

        prototype_by_bin[
            int(dim_bin)
        ] = np.median(
            X_longgap_snapshot_reference[
                bin_positions
            ],
            axis=0,
        ).astype(np.float32)


    longgap_snapshot_lactation_prototype[
        (
            int(class_id),
            int(lactation_number),
        )
    ] = prototype_by_bin


# ============================================================
# 13. Seen-AnimalId rate
# ============================================================

longgap_snapshot_seen_rate = (
    longgap_test_queries[
        "AnimalId"
    ]
    .astype(object)
    .isin(
        longgap_snapshot_class_values
    )
    .mean()
)


# ============================================================
# 14. Diagnostics
# ============================================================

print("=" * 72)

print(
    "Snapshot:",
    LONGGAP_TEST_SNAPSHOT.date()
)

print(
    "Reference rows:",
    f"{len(longgap_snapshot_reference):,}"
)

print(
    "Reference date range:",
    longgap_snapshot_reference[
        "EventDate"
    ].min(),
    "→",
    longgap_snapshot_reference[
        "EventDate"
    ].max()
)

print(
    "Assigned queries:",
    len(
        longgap_test_queries
    )
)

print(
    "Query date range:",
    longgap_test_queries[
        "EventDate"
    ].min(),
    "→",
    longgap_test_queries[
        "EventDate"
    ].max()
)

print(
    "AnimalId classes:",
    f"{len(longgap_snapshot_class_values):,}"
)

print(
    "Structural profiles:",
    f"{len(longgap_snapshot_profiles):,}"
)

print(
    "Sampled reference rows:",
    f"{len(longgap_snapshot_reference_raw):,}"
)

print(
    "Reference matrix:",
    X_longgap_snapshot_reference.shape
)

print(
    "AnimalId-lactation groups:",
    f"{len(longgap_snapshot_reference_exact_index):,}"
)

print(
    "Whole-cow prototypes:",
    f"{len(longgap_snapshot_cow_prototype):,}"
)

print(
    "Lactation/DIM prototype groups:",
    f"{len(longgap_snapshot_lactation_prototype):,}"
)

print(
    "Assigned-query AnimalId seen rate:",
    f"{longgap_snapshot_seen_rate:.2%}"
)

print(
    "Reference matrix memory:",
    f"{X_longgap_snapshot_reference.nbytes / 1024**2:.2f} MB"
)

Snapshot: 2020-04-01
Reference rows: 2,015,148
Reference date range: 2019-06-14 00:00:00 → 2020-04-01 00:00:00
Assigned queries: 502
Query date range: 2020-04-03 00:00:00 → 2020-09-08 00:00:00
AnimalId classes: 6,118
Structural profiles: 14,382
Sampled reference rows: 1,441,801
Reference matrix: (1441801, 13)
AnimalId-lactation groups: 8,266
Whole-cow prototypes: 6,118
Lactation/DIM prototype groups: 8,266
Assigned-query AnimalId seen rate: 89.64%
Reference matrix memory: 71.50 MB


In [78]:
# ============================================================
# LONG-GAP SNAPSHOT 1 — CANDIDATE GENERATION
#
# Snapshot:
#   2020-04-01
#
# Queries:
#   502
#
# Generate Global Top-320 first,
# then evaluate the champion's Top-160 retrieval.
# ============================================================

LONGGAP_TEST_PREFIX = "longgap0401_query"

LONGGAP_0401_CACHE_PATH = (
    CACHE_DIR
    /
    "longgap_candidates_20200401_global320.pkl"
)


# ============================================================
# 1. Map reranker globals to this snapshot
# ============================================================

longgap_snapshot_globals = {

    "reranker_class_to_number":
        longgap_snapshot_class_to_number,

    "reranker_cow_prototype":
        longgap_snapshot_cow_prototype,

    "reranker_feature_names":
        longgap_snapshot_feature_names,

    "reranker_lactation_prototype":
        longgap_snapshot_lactation_prototype,

    "reranker_numeric_columns":
        longgap_snapshot_numeric_columns,

    "reranker_numeric_fill":
        longgap_snapshot_numeric_fill,

    "reranker_origin_date":
        longgap_snapshot_origin_date,

    "reranker_profile_lookup":
        longgap_snapshot_profile_lookup,

    "reranker_reference_class_lactations":
        longgap_snapshot_reference_class_lactations,

    "reranker_reference_dim":
        longgap_snapshot_reference_dim,

    "reranker_reference_exact_index":
        longgap_snapshot_reference_exact_index,

    "reranker_reproduction_columns":
        longgap_snapshot_reproduction_columns,

    "reranker_scaler":
        longgap_snapshot_scaler,

    "X_reranker_reference":
        X_longgap_snapshot_reference,
}


# ============================================================
# 2. Backup current globals
# ============================================================

longgap_old_globals = {
    name: globals()[name]
    for name in longgap_snapshot_globals
}


# ============================================================
# 3. Generate / load candidates
# ============================================================

if LONGGAP_0401_CACHE_PATH.exists():

    print(
        "Loading cached 2020-04-01 candidates..."
    )

    longgap0401_candidate_pairs = (
        pd.read_pickle(
            LONGGAP_0401_CACHE_PATH
        )
    )

else:

    try:

        for name, value in (
            longgap_snapshot_globals.items()
        ):
            globals()[name] = value


        print(
            "Long-gap snapshot activated."
        )

        print(
            "Reference matrix:",
            X_reranker_reference.shape
        )

        print(
            "Assigned queries:",
            len(longgap_test_queries)
        )

        print(
            "Max candidates:",
            RERANKER_CANDIDATE_PARAMS[
                "max_candidates"
            ]
        )


        longgap0401_candidate_pairs = (
            build_candidate_dataset(
                longgap_test_queries,
                prefix=LONGGAP_TEST_PREFIX,
                progress_every=50,
            )
        )


        longgap0401_candidate_pairs.to_pickle(
            LONGGAP_0401_CACHE_PATH
        )


        print(
            "Saved:",
            LONGGAP_0401_CACHE_PATH
        )


    finally:

        for name, value in (
            longgap_old_globals.items()
        ):
            globals()[name] = value


        print(
            "Original reranker globals restored."
        )

Long-gap snapshot activated.
Reference matrix: (1441801, 13)
Assigned queries: 502
Max candidates: 320
longgap0401_query: processed 50 / 502 queries
longgap0401_query: processed 100 / 502 queries
longgap0401_query: processed 150 / 502 queries
longgap0401_query: processed 200 / 502 queries
longgap0401_query: processed 250 / 502 queries
longgap0401_query: processed 300 / 502 queries
longgap0401_query: processed 350 / 502 queries
longgap0401_query: processed 400 / 502 queries
longgap0401_query: processed 450 / 502 queries
longgap0401_query: processed 500 / 502 queries
longgap0401_query: processed 502 / 502 queries
Saved: xgboost_cache/longgap_candidates_20200401_global320.pkl
Original reranker globals restored.


In [79]:
# ============================================================
# LONG-GAP 2020-04-01 — TOP-160 RETRIEVAL DIAGNOSTIC
# ============================================================

LONGGAP_MODEL_K = 160


# ============================================================
# 1. Restrict to Top-160
# ============================================================

longgap0401_160 = (
    longgap0401_candidate_pairs.loc[
        longgap0401_candidate_pairs[
            "CandidateStartRank"
        ]
        <= LONGGAP_MODEL_K
    ]
    .copy()
)


# ============================================================
# 2. Candidate presence
# ============================================================

longgap0401_presence = (
    longgap0401_160
    .groupby("QueryId")[
        "IsCorrectCandidate"
    ]
    .max()
)


# ============================================================
# 3. Query-level evaluation table
# ============================================================

longgap0401_eval = (
    longgap_test_queries[
        [
            "AnimalId",
            "EventDate",
            "FreshnessWindow",
            "DaysSinceSnapshot",
        ]
    ]
    .copy()
)


longgap0401_eval[
    "QueryId"
] = [
    f"{LONGGAP_TEST_PREFIX}_{idx}"
    for idx in longgap0401_eval.index
]


longgap0401_eval[
    "AnimalSeenInSnapshot"
] = (
    longgap0401_eval[
        "AnimalId"
    ]
    .astype(object)
    .isin(
        longgap_snapshot_class_values
    )
)


longgap0401_eval[
    "CorrectCandidatePresent"
] = (
    longgap0401_eval[
        "QueryId"
    ]
    .map(
        longgap0401_presence
    )
    .fillna(0)
    .astype(bool)
)


# ============================================================
# 4. Overall metrics
# ============================================================

overall_coverage = (
    longgap0401_eval[
        "CorrectCandidatePresent"
    ].mean()
)


seen_coverage = (
    longgap0401_eval.loc[
        longgap0401_eval[
            "AnimalSeenInSnapshot"
        ],
        "CorrectCandidatePresent",
    ]
    .mean()
)


candidate_counts = (
    longgap0401_160
    .groupby("QueryId")
    .size()
)


print("=" * 72)

print(
    "LONG-GAP 2020-04-01 — TOP-160"
)

print("=" * 72)

print(
    "Queries:",
    len(longgap0401_eval)
)

print(
    "AnimalId seen rate:",
    f"{longgap0401_eval['AnimalSeenInSnapshot'].mean():.2%}"
)

print(
    "Top-160 candidate coverage:",
    f"{overall_coverage:.2%}"
)

print(
    "Seen-AnimalId candidate coverage:",
    f"{seen_coverage:.2%}"
)

print(
    "Correct candidate present:",
    int(
        longgap0401_eval[
            "CorrectCandidatePresent"
        ].sum()
    ),
    "/",
    len(longgap0401_eval)
)

print(
    "Mean candidates/query:",
    f"{candidate_counts.mean():.2f}"
)


# ============================================================
# 5. Coverage by freshness window
# ============================================================

longgap0401_by_freshness = (
    longgap0401_eval
    .groupby(
        "FreshnessWindow",
        observed=True,
    )
    .agg(
        Queries=(
            "QueryId",
            "size",
        ),

        MeanDays=(
            "DaysSinceSnapshot",
            "mean",
        ),

        AnimalSeenRate=(
            "AnimalSeenInSnapshot",
            "mean",
        ),

        CandidateCoverage=(
            "CorrectCandidatePresent",
            "mean",
        ),
    )
    .reset_index()
)


longgap0401_by_freshness[
    "AnimalSeenRate"
] *= 100

longgap0401_by_freshness[
    "CandidateCoverage"
] *= 100


print(
    "\nCoverage by freshness window:"
)

display(
    longgap0401_by_freshness
)

LONG-GAP 2020-04-01 — TOP-160
Queries: 502
AnimalId seen rate: 89.64%
Top-160 candidate coverage: 79.48%
Seen-AnimalId candidate coverage: 88.67%
Correct candidate present: 399 / 502
Mean candidates/query: 160.00

Coverage by freshness window:


,FreshnessWindow,Queries,MeanDays,AnimalSeenRate,CandidateCoverage
0,1-30 days,99,16.515152,97.979798,92.929293
1,121-160 days,122,138.229508,83.606557,69.672131
2,31-60 days,94,45.627660,92.553191,89.361702
3,61-90 days,96,75.489583,92.708333,81.250000
4,91-120 days,91,103.450549,82.417582,65.934066


In [80]:
# ============================================================
# LONG-GAP TRAINING
# BUILD REMAINING 3 SNAPSHOTS
#
#   2020-05-01
#   2020-06-01
#   2020-07-01
#
# For each snapshot:
#   1. Build leakage-safe reference
#   2. Build fresh KNN/prototype system
#   3. Generate Global Top-320 candidates
#   4. Save candidate table
#   5. Evaluate Top-160 coverage
#   6. Free memory
# ============================================================

import gc


REMAINING_LONGGAP_SNAPSHOTS = [
    pd.Timestamp("2020-05-01"),
    pd.Timestamp("2020-06-01"),
    pd.Timestamp("2020-07-01"),
]


# ============================================================
# Backup ORIGINAL reranker globals once
# ============================================================

LONGGAP_REFERENCE_GLOBALS = [
    "reranker_class_to_number",
    "reranker_cow_prototype",
    "reranker_feature_names",
    "reranker_lactation_prototype",
    "reranker_numeric_columns",
    "reranker_numeric_fill",
    "reranker_origin_date",
    "reranker_profile_lookup",
    "reranker_reference_class_lactations",
    "reranker_reference_dim",
    "reranker_reference_exact_index",
    "reranker_reproduction_columns",
    "reranker_scaler",
    "X_reranker_reference",
]


original_longgap_globals = {
    name: globals()[name]
    for name in LONGGAP_REFERENCE_GLOBALS
}


longgap_snapshot_summary = []
longgap_freshness_summary = []


# ============================================================
# Process snapshots sequentially
# ============================================================

for snapshot_date in REMAINING_LONGGAP_SNAPSHOTS:

    print("\n" + "=" * 75)

    print(
        "LONG-GAP SNAPSHOT:",
        snapshot_date.date()
    )

    print("=" * 75)


    snapshot_tag = (
        snapshot_date.strftime(
            "%Y%m%d"
        )
    )


    prefix = (
        "longgap"
        +
        snapshot_date.strftime("%m%d")
        +
        "_query"
    )


    cache_path = (
        CACHE_DIR
        /
        f"longgap_candidates_{snapshot_tag}_global320.pkl"
    )


    # ========================================================
    # 1. Assigned queries
    # ========================================================

    query_df = (
        longgap_training_queries.loc[
            longgap_training_queries[
                "SnapshotDate"
            ]
            ==
            snapshot_date
        ]
        .copy()
    )


    # ========================================================
    # 2. Leakage-safe reference
    # ========================================================

    reference_df = (
        data_train.loc[
            pd.to_datetime(
                data_train["EventDate"]
            )
            <=
            snapshot_date
        ]
        .copy()
    )


    reference_df[
        "EventDate"
    ] = pd.to_datetime(
        reference_df["EventDate"]
    )


    assert (
        reference_df[
            "EventDate"
        ].max()
        <=
        snapshot_date
    )


    assert (
        query_df[
            "EventDate"
        ].min()
        >
        snapshot_date
    )


    print(
        "Reference rows:",
        f"{len(reference_df):,}"
    )

    print(
        "Assigned queries:",
        len(query_df)
    )


    # ========================================================
    # 3. If cache exists, skip rebuilding reference system
    # ========================================================

    if cache_path.exists():

        print(
            "Loading cached candidates:",
            cache_path.name
        )

        candidate_pairs = (
            pd.read_pickle(
                cache_path
            )
        )

        snapshot_class_values_local = pd.Index(
            reference_df[
                "AnimalId"
            ]
            .astype(object)
            .unique()
        )


    else:

        # ====================================================
        # 4. Class mapping
        # ====================================================

        snapshot_class_values_local = pd.Index(
            reference_df[
                "AnimalId"
            ]
            .astype(object)
            .unique()
        )


        class_to_number = {
            animal_id: class_number
            for class_number, animal_id
            in enumerate(
                snapshot_class_values_local
            )
        }


        # ====================================================
        # 5. Origin date
        # ====================================================

        origin_date = (
            reference_df[
                "EventDate"
            ].min()
        )


        # ====================================================
        # 6. Structural profiles
        # ====================================================

        (
            structural_profiles,
            global_interval,
        ) = build_structural_profiles(
            reference_df,
            class_to_number,
            origin_date,
            include_projection=True,
        )


        profile_lookup = (
            make_profile_lookup(
                structural_profiles
            )
        )


        print(
            "AnimalId classes:",
            f"{len(snapshot_class_values_local):,}"
        )

        print(
            "Structural profiles:",
            f"{len(structural_profiles):,}"
        )


        # ====================================================
        # 7. Sample reference rows
        # ====================================================

        reference_indices = (
            sample_reference_rows(
                reference_df,

                max_rows_per_group=
                    REFERENCE_ROWS_PER_GROUP,

                recent_fraction=0.80,

                random_state=RANDOM_STATE,
            )
        )


        reference_raw = (
            reference_df
            .loc[
                reference_indices
            ]
            .copy()
        )


        # ====================================================
        # 8. Distance preprocessing
        # ====================================================

        (
            X_reference,
            numeric_columns,
            numeric_fill,
            scaler,
            reproduction_columns,
            feature_names,
        ) = fit_distance_preprocessor(
            reference_raw
        )


        print(
            "Reference matrix:",
            X_reference.shape
        )


        # ====================================================
        # 9. Metadata
        # ====================================================

        reference_class_ids = (
            reference_raw[
                "AnimalId"
            ]
            .astype(object)
            .map(
                class_to_number
            )
            .astype(np.int32)
            .to_numpy()
        )


        reference_lactations = (
            reference_raw[
                "LactationNumber"
            ]
            .round()
            .astype(np.int16)
            .to_numpy()
        )


        reference_dim = (
            reference_raw[
                "DaysInMilk"
            ]
            .astype(np.float32)
            .to_numpy()
        )


        # ====================================================
        # 10. Reference index
        # ====================================================

        (
            reference_exact_index,
            reference_class_lactations,
        ) = build_reference_index(
            reference_class_ids,
            reference_lactations,
        )


        # ====================================================
        # 11. Whole-cow prototypes
        # ====================================================

        cow_prototype = {}


        for class_id in np.unique(
            reference_class_ids
        ):

            positions = np.flatnonzero(
                reference_class_ids
                ==
                class_id
            )


            cow_prototype[
                int(class_id)
            ] = np.median(
                X_reference[
                    positions
                ],
                axis=0,
            ).astype(np.float32)


        # ====================================================
        # 12. Lactation + DIM-bin prototypes
        # ====================================================

        reference_dim_bin = np.floor(
            reference_dim
            /
            DIM_BIN_WIDTH
        ).astype(np.int16)


        lactation_prototype = {}


        for (
            class_id,
            lactation_number
        ), positions in (
            reference_exact_index.items()
        ):

            bins_here = (
                reference_dim_bin[
                    positions
                ]
            )


            prototype_by_bin = {}


            for dim_bin in np.unique(
                bins_here
            ):

                bin_positions = (
                    positions[
                        bins_here
                        ==
                        dim_bin
                    ]
                )


                prototype_by_bin[
                    int(dim_bin)
                ] = np.median(
                    X_reference[
                        bin_positions
                    ],
                    axis=0,
                ).astype(np.float32)


            lactation_prototype[
                (
                    int(class_id),
                    int(lactation_number),
                )
            ] = prototype_by_bin


        # ====================================================
        # 13. Temporarily activate this snapshot
        # ====================================================

        fresh_globals = {

            "reranker_class_to_number":
                class_to_number,

            "reranker_cow_prototype":
                cow_prototype,

            "reranker_feature_names":
                feature_names,

            "reranker_lactation_prototype":
                lactation_prototype,

            "reranker_numeric_columns":
                numeric_columns,

            "reranker_numeric_fill":
                numeric_fill,

            "reranker_origin_date":
                origin_date,

            "reranker_profile_lookup":
                profile_lookup,

            "reranker_reference_class_lactations":
                reference_class_lactations,

            "reranker_reference_dim":
                reference_dim,

            "reranker_reference_exact_index":
                reference_exact_index,

            "reranker_reproduction_columns":
                reproduction_columns,

            "reranker_scaler":
                scaler,

            "X_reranker_reference":
                X_reference,
        }


        try:

            for name, value in (
                fresh_globals.items()
            ):

                globals()[name] = value


            # ================================================
            # 14. Generate candidates
            # ================================================

            candidate_pairs = (
                build_candidate_dataset(
                    query_df,
                    prefix=prefix,
                    progress_every=50,
                )
            )


        finally:

            # ================================================
            # Restore original system
            # ================================================

            for name, value in (
                original_longgap_globals.items()
            ):

                globals()[name] = value


        # ====================================================
        # 15. Save candidate cache
        # ====================================================

        candidate_pairs.to_pickle(
            cache_path
        )


        print(
            "Saved:",
            cache_path.name
        )


    # ========================================================
    # 16. Top-160 retrieval
    # ========================================================

    candidate160 = (
        candidate_pairs.loc[
            candidate_pairs[
                "CandidateStartRank"
            ]
            <= 160
        ]
        .copy()
    )


    presence160 = (
        candidate160
        .groupby("QueryId")[
            "IsCorrectCandidate"
        ]
        .max()
    )


    eval_df = (
        query_df[
            [
                "AnimalId",
                "FreshnessWindow",
                "DaysSinceSnapshot",
            ]
        ]
        .copy()
    )


    eval_df[
        "QueryId"
    ] = [
        f"{prefix}_{idx}"
        for idx in eval_df.index
    ]


    eval_df[
        "AnimalSeen"
    ] = (
        eval_df[
            "AnimalId"
        ]
        .astype(object)
        .isin(
            snapshot_class_values_local
        )
    )


    eval_df[
        "CorrectCandidatePresent"
    ] = (
        eval_df[
            "QueryId"
        ]
        .map(
            presence160
        )
        .fillna(0)
        .astype(bool)
    )


    # ========================================================
    # 17. Snapshot-level summary
    # ========================================================

    overall_coverage = (
        eval_df[
            "CorrectCandidatePresent"
        ].mean()
    )


    seen_coverage = (
        eval_df.loc[
            eval_df[
                "AnimalSeen"
            ],
            "CorrectCandidatePresent",
        ]
        .mean()
    )


    snapshot_summary_row = {

        "SnapshotDate":
            snapshot_date.date(),

        "Queries":
            len(eval_df),

        "AnimalSeenRate":
            eval_df[
                "AnimalSeen"
            ].mean() * 100,

        "Top160Coverage":
            overall_coverage * 100,

        "SeenTop160Coverage":
            seen_coverage * 100,

        "CorrectQueries":
            int(
                eval_df[
                    "CorrectCandidatePresent"
                ].sum()
            ),

        "CandidateRows":
            len(candidate_pairs),
    }


    longgap_snapshot_summary.append(
        snapshot_summary_row
    )


    print(
        "AnimalId seen rate:",
        f"{eval_df['AnimalSeen'].mean():.2%}"
    )

    print(
        "Top-160 coverage:",
        f"{overall_coverage:.2%}"
    )

    print(
        "Seen Top-160 coverage:",
        f"{seen_coverage:.2%}"
    )


    # ========================================================
    # 18. Freshness-window summary
    # ========================================================

    by_window = (
        eval_df
        .groupby(
            "FreshnessWindow",
            observed=True,
        )
        .agg(
            Queries=(
                "QueryId",
                "size",
            ),

            MeanDays=(
                "DaysSinceSnapshot",
                "mean",
            ),

            AnimalSeenRate=(
                "AnimalSeen",
                "mean",
            ),

            CandidateCoverage=(
                "CorrectCandidatePresent",
                "mean",
            ),
        )
        .reset_index()
    )


    by_window[
        "AnimalSeenRate"
    ] *= 100


    by_window[
        "CandidateCoverage"
    ] *= 100


    by_window[
        "SnapshotDate"
    ] = snapshot_date.date()


    longgap_freshness_summary.append(
        by_window
    )


    # ========================================================
    # 19. Memory cleanup
    # ========================================================

    del candidate_pairs
    del candidate160
    del presence160
    del eval_df
    del reference_df
    del query_df

    gc.collect()


# ============================================================
# 20. Final summaries
# ============================================================

longgap_remaining_snapshot_summary_df = (
    pd.DataFrame(
        longgap_snapshot_summary
    )
)


longgap_remaining_freshness_df = (
    pd.concat(
        longgap_freshness_summary,
        ignore_index=True,
    )
)


print("\n" + "=" * 75)

print(
    "REMAINING LONG-GAP SNAPSHOTS COMPLETE"
)

print("=" * 75)


display(
    longgap_remaining_snapshot_summary_df
)


print(
    "\nCoverage by snapshot and freshness window:"
)

display(
    longgap_remaining_freshness_df
)


LONG-GAP SNAPSHOT: 2020-05-01
Reference rows: 2,257,505
Assigned queries: 501
AnimalId classes: 6,285
Structural profiles: 15,013
Reference matrix: (1552679, 13)
longgap0501_query: processed 50 / 501 queries
longgap0501_query: processed 100 / 501 queries
longgap0501_query: processed 150 / 501 queries
longgap0501_query: processed 200 / 501 queries
longgap0501_query: processed 250 / 501 queries
longgap0501_query: processed 300 / 501 queries
longgap0501_query: processed 350 / 501 queries
longgap0501_query: processed 400 / 501 queries
longgap0501_query: processed 450 / 501 queries
longgap0501_query: processed 500 / 501 queries
longgap0501_query: processed 501 / 501 queries
Saved: longgap_candidates_20200501_global320.pkl
AnimalId seen rate: 90.62%
Top-160 coverage: 80.64%
Seen Top-160 coverage: 88.99%

LONG-GAP SNAPSHOT: 2020-06-01
Reference rows: 2,510,577
Assigned queries: 499
AnimalId classes: 6,466
Structural profiles: 15,668
Reference matrix: (1662750, 13)
longgap0601_query: processe

,SnapshotDate,Queries,AnimalSeenRate,Top160Coverage,SeenTop160Coverage,CorrectQueries,CandidateRows
0,2020-05-01,501,90.618762,80.638723,88.986784,404,160181
1,2020-06-01,499,86.773547,77.955912,89.838337,389,159160
2,2020-07-01,498,91.767068,81.927711,89.277899,408,158702



Coverage by snapshot and freshness window:


,FreshnessWindow,Queries,MeanDays,AnimalSeenRate,CandidateCoverage,SnapshotDate
0,1-30 days,99,15.303030,98.989899,97.979798,2020-05-01
1,121-160 days,122,136.311475,86.065574,63.934426,2020-05-01
2,31-60 days,94,45.840426,91.489362,88.297872,2020-05-01
3,61-90 days,96,74.625000,91.666667,82.291667,2020-05-01
4,91-120 days,90,103.955556,85.555556,74.444444,2020-05-01
5,1-30 days,98,16.142857,96.938776,93.877551,2020-06-01
6,121-160 days,121,147.272727,80.991736,67.768595,2020-06-01
7,31-60 days,94,46.500000,90.425532,84.042553,2020-06-01
8,61-90 days,96,74.166667,84.375000,76.041667,2020-06-01
9,91-120 days,90,106.355556,82.222222,70.000000,2020-06-01


In [81]:
# ============================================================
# LONG-GAP TRAINING DATASET
# Combine all FOUR snapshot candidate tables
# ============================================================

import gc


LONGGAP_CACHE_FILES = [
    "longgap_candidates_20200401_global320.pkl",
    "longgap_candidates_20200501_global320.pkl",
    "longgap_candidates_20200601_global320.pkl",
    "longgap_candidates_20200701_global320.pkl",
]


longgap_candidate_tables = []

longgap_merge_summary = []

schema_reference = None


for filename in LONGGAP_CACHE_FILES:

    path = (
        CACHE_DIR
        /
        filename
    )

    print(
        "Loading:",
        filename
    )

    df_snapshot = (
        pd.read_pickle(
            path
        )
    )


    # ========================================================
    # Schema check
    # ========================================================

    if schema_reference is None:

        schema_reference = list(
            df_snapshot.columns
        )

    else:

        assert (
            list(df_snapshot.columns)
            ==
            schema_reference
        ), (
            f"Column mismatch in {filename}"
        )


    # ========================================================
    # Snapshot diagnostics
    # ========================================================

    snapshot_presence = (
        df_snapshot
        .loc[
            df_snapshot[
                "CandidateStartRank"
            ]
            <= 160
        ]
        .groupby("QueryId")[
            "IsCorrectCandidate"
        ]
        .max()
    )


    longgap_merge_summary.append({

        "File":
            filename,

        "Rows":
            len(df_snapshot),

        "Queries":
            df_snapshot[
                "QueryId"
            ].nunique(),

        "Top160PositiveQueries":
            int(
                snapshot_presence.sum()
            ),

        "Columns":
            len(
                df_snapshot.columns
            ),
    })


    longgap_candidate_tables.append(
        df_snapshot
    )


# ============================================================
# Combine Global Top-320 candidate tables
# ============================================================

longgap_multisnapshot_train = (
    pd.concat(
        longgap_candidate_tables,
        ignore_index=True,
    )
)


del longgap_candidate_tables

gc.collect()


# ============================================================
# Safety checks
# ============================================================

longgap_query_count = (
    longgap_multisnapshot_train[
        "QueryId"
    ].nunique()
)


assert (
    longgap_query_count
    ==
    2000
)


# ============================================================
# Top-160 version
# ============================================================

longgap_train160 = (
    longgap_multisnapshot_train.loc[
        longgap_multisnapshot_train[
            "CandidateStartRank"
        ]
        <= 160
    ]
    .copy()
    .reset_index(drop=True)
)


longgap_train160_presence = (
    longgap_train160
    .groupby("QueryId")[
        "IsCorrectCandidate"
    ]
    .max()
)


longgap_positive_queries = int(
    longgap_train160_presence.sum()
)


assert (
    longgap_positive_queries
    ==
    1600
)


# ============================================================
# Candidate counts
# ============================================================

longgap_candidate_counts = (
    longgap_train160
    .groupby("QueryId")
    .size()
)


# ============================================================
# Duplicate check
# ============================================================

duplicate_candidate_rows = (
    longgap_multisnapshot_train[
        [
            "QueryId",
            "CandidateClassId",
        ]
    ]
    .duplicated()
    .sum()
)


# ============================================================
# Results
# ============================================================

print("\n" + "=" * 72)

display(
    pd.DataFrame(
        longgap_merge_summary
    )
)


print(
    "Combined Global320 rows:",
    f"{len(longgap_multisnapshot_train):,}"
)

print(
    "Combined queries:",
    longgap_query_count
)

print(
    "Top160 rows:",
    f"{len(longgap_train160):,}"
)

print(
    "Mean Top160 candidates/query:",
    f"{longgap_candidate_counts.mean():.2f}"
)

print(
    "Top160 correct-candidate queries:",
    longgap_positive_queries
)

print(
    "Top160 missing-candidate queries:",
    2000
    -
    longgap_positive_queries
)

print(
    "Top160 training coverage:",
    f"{longgap_positive_queries / 2000:.2%}"
)

print(
    "Columns:",
    len(
        longgap_multisnapshot_train.columns
    )
)

print(
    "Duplicate candidate rows:",
    duplicate_candidate_rows
)

print(
    "Global320 memory:",
    f"{longgap_multisnapshot_train.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

Loading: longgap_candidates_20200401_global320.pkl
Loading: longgap_candidates_20200501_global320.pkl
Loading: longgap_candidates_20200601_global320.pkl
Loading: longgap_candidates_20200701_global320.pkl



,File,Rows,Queries,Top160PositiveQueries,Columns
0,longgap_candidates_20200401_global320.pkl,160342,502,399,56
1,longgap_candidates_20200501_global320.pkl,160181,501,404,56
2,longgap_candidates_20200601_global320.pkl,159160,499,389,56
3,longgap_candidates_20200701_global320.pkl,158702,498,408,56


Combined Global320 rows: 638,385
Combined queries: 2000
Top160 rows: 319,654
Mean Top160 candidates/query: 159.83
Top160 correct-candidate queries: 1600
Top160 missing-candidate queries: 400
Top160 training coverage: 80.00%
Columns: 56
Duplicate candidate rows: 0
Global320 memory: 259.35 MB


In [82]:
# ============================================================
# LONG-GAP HGB
#
# Only change:
#   TRAINING DATA
#
# Everything else stays fixed:
#   Top-K = 160
#   Same 52 features
#   Same HGB hyperparameters
#   Same 2,000 outer-validation queries
# ============================================================


# ============================================================
# 1. Recalculate CandidateCount inside Top-160
# ============================================================

longgap_train160 = (
    longgap_train160
    .copy()
    .reset_index(drop=True)
)


longgap_train160[
    "CandidateCount"
] = (
    longgap_train160
    .groupby("QueryId")["QueryId"]
    .transform("size")
    .astype(np.int16)
)


# ============================================================
# 2. Recompute rank features using ONLY Top-160 candidates
# ============================================================

longgap_train160_ranked = (
    add_candidate_rank_features(
        longgap_train160
    )
)


# ============================================================
# 3. Keep only queries for which the true candidate exists
#
# Same training rule used for previous champion.
# ============================================================

longgap_positive_available = (
    longgap_train160_ranked
    .groupby("QueryId")[
        "IsCorrectCandidate"
    ]
    .transform("max")
)


longgap_train160_model = (
    longgap_train160_ranked.loc[
        longgap_positive_available == 1
    ]
    .copy()
    .reset_index(drop=True)
)


print(
    "Training queries with correct candidate:",
    longgap_train160_model[
        "QueryId"
    ].nunique()
)

print(
    "Expected:",
    1600
)

Training queries with correct candidate: 1600
Expected: 1600


In [83]:
# ============================================================
# 4. Use EXACT SAME feature definition as champion
# ============================================================

LONGGAP_NON_FEATURE_COLUMNS = [
    "QueryId",
    "CandidateClassId",
    "TrueClassId",
    "IsCorrectCandidate",
]


LONGGAP_FEATURES = [
    column
    for column in longgap_train160_model.columns
    if column not in LONGGAP_NON_FEATURE_COLUMNS
]


print(
    "Long-gap features:",
    len(LONGGAP_FEATURES)
)

print(
    "Champion features:",
    len(frozen_feature_order)
)


# ============================================================
# 5. Require exact feature compatibility
# ============================================================

assert set(
    LONGGAP_FEATURES
) == set(
    frozen_feature_order
), (
    "Long-gap feature set does not match champion."
)


# IMPORTANT:
# use champion's exact feature order
LONGGAP_FEATURES = (
    frozen_feature_order.copy()
)


# ============================================================
# 6. Training matrices
# ============================================================

X_longgap_train = (
    longgap_train160_model[
        LONGGAP_FEATURES
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .astype(np.float32)
)


y_longgap_train = (
    longgap_train160_model[
        "IsCorrectCandidate"
    ]
    .astype(np.int8)
)


print(
    "Training matrix:",
    X_longgap_train.shape
)

print(
    "Positive rows:",
    int(
        y_longgap_train.sum()
    )
)

Long-gap features: 52
Champion features: 52
Training matrix: (255654, 52)
Positive rows: 1600


In [84]:
# ============================================================
# 7. Train LONG-GAP HGB
# ============================================================

from sklearn.ensemble import (
    HistGradientBoostingClassifier
)


longgap_hgb160 = (
    HistGradientBoostingClassifier(

        learning_rate=0.05,

        max_iter=400,

        max_leaf_nodes=7,

        min_samples_leaf=10,

        l2_regularization=1.0,

        early_stopping=True,

        validation_fraction=0.10,

        n_iter_no_change=20,

        random_state=RANDOM_STATE,
    )
)


longgap_hgb160.fit(
    X_longgap_train,
    y_longgap_train,
)


print(
    "Long-gap HGB trained."
)

print(
    "Iterations used:",
    longgap_hgb160.n_iter_
)

Long-gap HGB trained.
Iterations used: 349


In [85]:
# ============================================================
# 8. OUTER VALIDATION
#
# Reuse exactly the same outer_val160 candidates.
# ============================================================

outer_val160_for_longgap = (
    outer_val160
    .copy()
    .reset_index(drop=True)
)


outer_val160_for_longgap[
    "CandidateCount"
] = (
    outer_val160_for_longgap
    .groupby("QueryId")[
        "QueryId"
    ]
    .transform("size")
    .astype(np.int16)
)


outer_val160_longgap_ranked = (
    add_candidate_rank_features(
        outer_val160_for_longgap
    )
)


# ============================================================
# 9. Matrix using EXACT frozen feature order
# ============================================================

X_outer_longgap = (
    outer_val160_longgap_ranked[
        LONGGAP_FEATURES
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .astype(np.float32)
)


# ============================================================
# 10. Predict
# ============================================================

longgap_outer_prob = (
    longgap_hgb160.predict_proba(
        X_outer_longgap
    )[:, 1]
)


longgap_outer_scored = (
    outer_val160_longgap_ranked[
        [
            "QueryId",
            "CandidateClassId",
            "TrueClassId",
            "IsCorrectCandidate",
        ]
    ]
    .copy()
)


longgap_outer_scored[
    "Probability"
] = longgap_outer_prob


# ============================================================
# 11. Best candidate per query
# ============================================================

longgap_outer_best = (
    longgap_outer_scored
    .sort_values(
        [
            "QueryId",
            "Probability",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .groupby(
        "QueryId",
        as_index=False,
    )
    .first()
)


longgap_outer_best[
    "PredictionCorrect"
] = (
    longgap_outer_best[
        "CandidateClassId"
    ]
    ==
    longgap_outer_best[
        "TrueClassId"
    ]
)


# ============================================================
# 12. Query-level evaluation
# ============================================================

longgap_outer_eval = (
    outer_val_eval
    .merge(
        longgap_outer_best[
            [
                "QueryId",
                "PredictionCorrect",
            ]
        ],
        on="QueryId",
        how="left",
    )
)


longgap_outer_eval[
    "PredictionCorrect"
] = (
    longgap_outer_eval[
        "PredictionCorrect"
    ]
    .fillna(False)
    .astype(bool)
)


# ============================================================
# 13. Metrics
# ============================================================

longgap_outer_overall = (
    longgap_outer_eval[
        "PredictionCorrect"
    ].mean()
)


longgap_outer_seen = (
    longgap_outer_eval.loc[
        longgap_outer_eval[
            "AnimalSeenInTrain"
        ],
        "PredictionCorrect",
    ].mean()
)


longgap_outer_conditional = (
    longgap_outer_eval.loc[
        longgap_outer_eval[
            "CorrectCandidatePresent"
        ],
        "PredictionCorrect",
    ].mean()
)


longgap_outer_correct = int(
    longgap_outer_eval[
        "PredictionCorrect"
    ].sum()
)


print("\n" + "=" * 72)

print(
    "OUTER VALIDATION — LONG-GAP HGB"
)

print("=" * 72)

print(
    "Top-160 candidate coverage:",
    f"{longgap_outer_eval['CorrectCandidatePresent'].mean():.2%}"
)

print(
    "Overall accuracy:",
    f"{longgap_outer_overall:.2%}"
)

print(
    "Seen-AnimalId accuracy:",
    f"{longgap_outer_seen:.2%}"
)

print(
    "Conditional ranking accuracy:",
    f"{longgap_outer_conditional:.2%}"
)

print(
    "Correct predictions:",
    longgap_outer_correct,
    "/",
    len(longgap_outer_eval)
)


print("\n" + "-" * 72)

print(
    "OLD frozen champion overall:",
    "38.50%"
)

print(
    "OLD frozen champion conditional:",
    "46.92%"
)


OUTER VALIDATION — LONG-GAP HGB
Top-160 candidate coverage: 82.05%
Overall accuracy: 41.35%
Seen-AnimalId accuracy: 44.85%
Conditional ranking accuracy: 50.40%
Correct predictions: 827 / 2000

------------------------------------------------------------------------
OLD frozen champion overall: 38.50%
OLD frozen champion conditional: 46.92%


In [86]:
# ============================================================
# COMPARE TWO MODELS BY REFERENCE AGE
#
# Model A:
#   Frozen fresh champion
#
# Model B:
#   Long-gap HGB
#
# Same queries
# Same Top-160 candidates
# ============================================================

FRESHNESS_BINS = [
    0,
    30,
    60,
    90,
    120,
    160,
    9999,
]

FRESHNESS_LABELS = [
    "1-30 days",
    "31-60 days",
    "61-90 days",
    "91-120 days",
    "121-160 days",
    ">160 days",
]


def summarize_by_freshness(
    eval_df,
    model_name,
):

    temp = eval_df.copy()

    temp[
        "DaysSinceReferenceCutoff"
    ] = (
        pd.to_datetime(
            temp["EventDate"]
        )
        -
        OUTER_VAL_CUTOFF
    ).dt.days


    temp[
        "FreshnessWindow"
    ] = pd.cut(
        temp[
            "DaysSinceReferenceCutoff"
        ],
        bins=FRESHNESS_BINS,
        labels=FRESHNESS_LABELS,
        include_lowest=True,
        right=True,
    )


    rows = []


    for window in FRESHNESS_LABELS:

        subset = temp.loc[
            temp[
                "FreshnessWindow"
            ]
            ==
            window
        ]


        if len(subset) == 0:
            continue


        present = subset.loc[
            subset[
                "CorrectCandidatePresent"
            ]
        ]


        overall = (
            subset[
                "PredictionCorrect"
            ].mean()
        )


        conditional = (
            present[
                "PredictionCorrect"
            ].mean()
            if len(present) > 0
            else np.nan
        )


        rows.append({

            "FreshnessWindow":
                window,

            f"{model_name}_Overall":
                overall * 100,

            f"{model_name}_Conditional":
                conditional * 100,

            "Queries":
                len(subset),
        })


    return pd.DataFrame(
        rows
    )


# ============================================================
# Frozen fresh champion
# ============================================================

fresh_by_gap = summarize_by_freshness(
    outer_val_final_eval,
    "FreshChampion",
)


# ============================================================
# Long-gap model
# ============================================================

longgap_by_gap = summarize_by_freshness(
    longgap_outer_eval,
    "LongGap",
)


# ============================================================
# Merge comparison
# ============================================================

model_gap_comparison = (
    fresh_by_gap
    .merge(
        longgap_by_gap.drop(
            columns=["Queries"]
        ),
        on="FreshnessWindow",
        how="inner",
    )
)


model_gap_comparison[
    "OverallGain"
] = (
    model_gap_comparison[
        "LongGap_Overall"
    ]
    -
    model_gap_comparison[
        "FreshChampion_Overall"
    ]
)


model_gap_comparison[
    "ConditionalGain"
] = (
    model_gap_comparison[
        "LongGap_Conditional"
    ]
    -
    model_gap_comparison[
        "FreshChampion_Conditional"
    ]
)


display(
    model_gap_comparison
)

,FreshnessWindow,FreshChampion_Overall,FreshChampion_Conditional,Queries,LongGap_Overall,LongGap_Conditional,OverallGain,ConditionalGain
0,1-30 days,71.827411,74.670185,394,69.796954,72.559367,-2.030457,-2.110818
1,31-60 days,50.666667,56.547619,375,50.133333,55.952381,-0.533333,-0.595238
2,61-90 days,31.770833,37.654321,384,34.635417,41.049383,2.864583,3.395062
3,91-120 days,24.099723,31.182796,361,29.085873,37.634409,4.986150,6.451613
4,121-160 days,18.106996,27.244582,486,25.925926,39.009288,7.818930,11.764706


In [88]:
# ============================================================
# HYBRID TRAINING SET
#
# Combine:
#   Fresh multi-snapshot training
#   +
#   Long-gap multi-snapshot training
#
# Same:
#   Top-160
#   52 features
#   HGB parameters
# ============================================================


# ============================================================
# 1. Safety check: QueryIds must not overlap
# ============================================================

fresh_query_ids = set(
    fresh_train160_model[
        "QueryId"
    ].unique()
)

longgap_query_ids = set(
    longgap_train160_model[
        "QueryId"
    ].unique()
)


query_overlap = (
    fresh_query_ids
    &
    longgap_query_ids
)


print(
    "Fresh training queries:",
    len(fresh_query_ids)
)

print(
    "Long-gap training queries:",
    len(longgap_query_ids)
)

print(
    "QueryId overlap:",
    len(query_overlap)
)


assert len(query_overlap) == 0


# ============================================================
# 2. Exact feature compatibility
# ============================================================

assert all(
    feature in fresh_train160_model.columns
    for feature in frozen_feature_order
)


assert all(
    feature in longgap_train160_model.columns
    for feature in frozen_feature_order
)


# ============================================================
# 3. Combine model-ready candidate rows
# ============================================================

HYBRID_COLUMNS = (
    [
        "QueryId",
        "CandidateClassId",
        "TrueClassId",
        "IsCorrectCandidate",
    ]
    +
    frozen_feature_order
)


hybrid_train160_model = (
    pd.concat(
        [
            fresh_train160_model[
                HYBRID_COLUMNS
            ],

            longgap_train160_model[
                HYBRID_COLUMNS
            ],
        ],
        ignore_index=True,
    )
)


# ============================================================
# 4. Diagnostics
# ============================================================

hybrid_query_count = (
    hybrid_train160_model[
        "QueryId"
    ].nunique()
)


hybrid_positive_rows = int(
    hybrid_train160_model[
        "IsCorrectCandidate"
    ].sum()
)


print("\n" + "=" * 70)

print(
    "Hybrid training queries:",
    hybrid_query_count
)

print(
    "Expected:",
    len(fresh_query_ids)
    +
    len(longgap_query_ids)
)

print(
    "Hybrid candidate rows:",
    f"{len(hybrid_train160_model):,}"
)

print(
    "Positive rows:",
    hybrid_positive_rows
)

print(
    "Features:",
    len(frozen_feature_order)
)

print(
    "Memory:",
    f"{hybrid_train160_model.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

Fresh training queries: 1777
Long-gap training queries: 1600
QueryId overlap: 0

Hybrid training queries: 3377
Expected: 3377
Hybrid candidate rows: 539,751
Positive rows: 3377
Features: 52
Memory: 214.69 MB


In [89]:
# ============================================================
# HYBRID HGB
# SAME PARAMETERS AS BOTH PREVIOUS MODELS
# ============================================================

from sklearn.ensemble import (
    HistGradientBoostingClassifier
)


X_hybrid_train = (
    hybrid_train160_model[
        frozen_feature_order
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .astype(np.float32)
)


y_hybrid_train = (
    hybrid_train160_model[
        "IsCorrectCandidate"
    ]
    .astype(np.int8)
)


print(
    "Hybrid training matrix:",
    X_hybrid_train.shape
)


hybrid_hgb160 = (
    HistGradientBoostingClassifier(

        learning_rate=0.05,

        max_iter=400,

        max_leaf_nodes=7,

        min_samples_leaf=10,

        l2_regularization=1.0,

        early_stopping=True,

        validation_fraction=0.10,

        n_iter_no_change=20,

        random_state=RANDOM_STATE,
    )
)


hybrid_hgb160.fit(
    X_hybrid_train,
    y_hybrid_train,
)


print(
    "Hybrid HGB trained."
)

print(
    "Iterations used:",
    hybrid_hgb160.n_iter_
)

Hybrid training matrix: (539751, 52)
Hybrid HGB trained.
Iterations used: 400


In [90]:
# ============================================================
# HYBRID HGB — OUTER VALIDATION
#
# SAME 2,000 outer-validation queries
# SAME candidate set
# ============================================================


# ============================================================
# 1. Use already prepared outer-validation matrix
# ============================================================

X_outer_hybrid = (
    outer_val160_longgap_ranked[
        frozen_feature_order
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .astype(np.float32)
)


# ============================================================
# 2. Predict
# ============================================================

hybrid_outer_prob = (
    hybrid_hgb160.predict_proba(
        X_outer_hybrid
    )[:, 1]
)


hybrid_outer_scored = (
    outer_val160_longgap_ranked[
        [
            "QueryId",
            "CandidateClassId",
            "TrueClassId",
            "IsCorrectCandidate",
        ]
    ]
    .copy()
)


hybrid_outer_scored[
    "Probability"
] = hybrid_outer_prob


# ============================================================
# 3. Highest probability candidate per query
# ============================================================

hybrid_outer_best = (
    hybrid_outer_scored
    .sort_values(
        [
            "QueryId",
            "Probability",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .groupby(
        "QueryId",
        as_index=False,
    )
    .first()
)


hybrid_outer_best[
    "PredictionCorrect"
] = (
    hybrid_outer_best[
        "CandidateClassId"
    ]
    ==
    hybrid_outer_best[
        "TrueClassId"
    ]
)


# ============================================================
# 4. Query-level evaluation
# ============================================================

hybrid_outer_eval = (
    outer_val_eval
    .merge(
        hybrid_outer_best[
            [
                "QueryId",
                "PredictionCorrect",
            ]
        ],
        on="QueryId",
        how="left",
    )
)


hybrid_outer_eval[
    "PredictionCorrect"
] = (
    hybrid_outer_eval[
        "PredictionCorrect"
    ]
    .fillna(False)
    .astype(bool)
)


# ============================================================
# 5. Metrics
# ============================================================

hybrid_outer_overall = (
    hybrid_outer_eval[
        "PredictionCorrect"
    ].mean()
)


hybrid_outer_seen = (
    hybrid_outer_eval.loc[
        hybrid_outer_eval[
            "AnimalSeenInTrain"
        ],
        "PredictionCorrect",
    ].mean()
)


hybrid_outer_conditional = (
    hybrid_outer_eval.loc[
        hybrid_outer_eval[
            "CorrectCandidatePresent"
        ],
        "PredictionCorrect",
    ].mean()
)


hybrid_outer_correct = int(
    hybrid_outer_eval[
        "PredictionCorrect"
    ].sum()
)


print("\n" + "=" * 72)

print(
    "OUTER VALIDATION — HYBRID HGB"
)

print("=" * 72)

print(
    "Top-160 candidate coverage:",
    f"{hybrid_outer_eval['CorrectCandidatePresent'].mean():.2%}"
)

print(
    "Overall accuracy:",
    f"{hybrid_outer_overall:.2%}"
)

print(
    "Seen-AnimalId accuracy:",
    f"{hybrid_outer_seen:.2%}"
)

print(
    "Conditional ranking accuracy:",
    f"{hybrid_outer_conditional:.2%}"
)

print(
    "Correct predictions:",
    hybrid_outer_correct,
    "/",
    len(hybrid_outer_eval)
)


print("\nComparison:")

print(
    "Fresh champion: 38.50%"
)

print(
    "Long-gap HGB:   41.35%"
)

print(
    "Hybrid HGB:    ",
    f"{hybrid_outer_overall:.2%}"
)


OUTER VALIDATION — HYBRID HGB
Top-160 candidate coverage: 82.05%
Overall accuracy: 41.65%
Seen-AnimalId accuracy: 45.17%
Conditional ranking accuracy: 50.76%
Correct predictions: 833 / 2000

Comparison:
Fresh champion: 38.50%
Long-gap HGB:   41.35%
Hybrid HGB:     41.65%


In [91]:
# ============================================================
# ADD REFERENCE AGE FEATURE
#
# New feature:
#   ReferenceAgeDays
#
# Definition:
#   query EventDate - reference snapshot cutoff
#
# No model training yet.
# ============================================================

REFERENCE_AGE_FEATURE = "ReferenceAgeDays"


# ============================================================
# 1. FRESH TRAINING: QueryId -> ReferenceAgeDays
# ============================================================

fresh_snapshot_prefixes = {
    pd.Timestamp("2020-07-10"): "fresh0710_query",
    pd.Timestamp("2020-07-20"): "fresh0720_query",
    pd.Timestamp("2020-07-31"): "fresh0731_query",
    pd.Timestamp("2020-08-10"): "fresh0810_query",
    pd.Timestamp("2020-08-20"): "fresh0820_query",
}


fresh_reference_age_map = {}


for snapshot_date, prefix in fresh_snapshot_prefixes.items():

    query_df = snapshot_query_groups[
        snapshot_date
    ]

    query_dates = pd.to_datetime(
        query_df["EventDate"]
    )

    ages = (
        query_dates
        -
        snapshot_date
    ).dt.days


    for idx, age in ages.items():

        query_id = (
            f"{prefix}_{idx}"
        )

        fresh_reference_age_map[
            query_id
        ] = int(age)


fresh_train160_model[
    REFERENCE_AGE_FEATURE
] = (
    fresh_train160_model[
        "QueryId"
    ]
    .map(
        fresh_reference_age_map
    )
)


# ============================================================
# 2. LONG-GAP TRAINING: QueryId -> ReferenceAgeDays
# ============================================================

longgap_snapshot_prefixes = {
    pd.Timestamp("2020-04-01"): "longgap0401_query",
    pd.Timestamp("2020-05-01"): "longgap0501_query",
    pd.Timestamp("2020-06-01"): "longgap0601_query",
    pd.Timestamp("2020-07-01"): "longgap0701_query",
}


longgap_reference_age_map = {}


for snapshot_date, prefix in longgap_snapshot_prefixes.items():

    query_df = (
        longgap_training_queries.loc[
            longgap_training_queries[
                "SnapshotDate"
            ]
            ==
            snapshot_date
        ]
    )

    for idx, row in query_df.iterrows():

        age = (
            pd.Timestamp(
                row["EventDate"]
            )
            -
            snapshot_date
        ).days

        query_id = (
            f"{prefix}_{idx}"
        )

        longgap_reference_age_map[
            query_id
        ] = int(age)


longgap_train160_model[
    REFERENCE_AGE_FEATURE
] = (
    longgap_train160_model[
        "QueryId"
    ]
    .map(
        longgap_reference_age_map
    )
)


# ============================================================
# 3. OUTER VALIDATION:
#    fixed reference cutoff = 2020-12-13
# ============================================================

outer_reference_age_map = {}


for idx, row in outer_validation_queries.iterrows():

    age = (
        pd.Timestamp(
            row["EventDate"]
        )
        -
        OUTER_VAL_CUTOFF
    ).days

    query_id = (
        f"{OUTER_VAL_PREFIX}_{idx}"
    )

    outer_reference_age_map[
        query_id
    ] = int(age)


outer_val160_longgap_ranked[
    REFERENCE_AGE_FEATURE
] = (
    outer_val160_longgap_ranked[
        "QueryId"
    ]
    .map(
        outer_reference_age_map
    )
)


# ============================================================
# 4. Safety checks
# ============================================================

assert (
    fresh_train160_model[
        REFERENCE_AGE_FEATURE
    ].notna().all()
)

assert (
    longgap_train160_model[
        REFERENCE_AGE_FEATURE
    ].notna().all()
)

assert (
    outer_val160_longgap_ranked[
        REFERENCE_AGE_FEATURE
    ].notna().all()
)


assert (
    fresh_train160_model[
        REFERENCE_AGE_FEATURE
    ].ge(0).all()
)

assert (
    longgap_train160_model[
        REFERENCE_AGE_FEATURE
    ].ge(0).all()
)

assert (
    outer_val160_longgap_ranked[
        REFERENCE_AGE_FEATURE
    ].ge(0).all()
)


# ============================================================
# 5. Query-level age diagnostics
# ============================================================

def query_age_summary(df, name):

    query_age = (
        df[
            [
                "QueryId",
                REFERENCE_AGE_FEATURE,
            ]
        ]
        .drop_duplicates(
            "QueryId"
        )
    )

    return {
        "Dataset": name,

        "Queries":
            len(query_age),

        "MinAge":
            query_age[
                REFERENCE_AGE_FEATURE
            ].min(),

        "MedianAge":
            query_age[
                REFERENCE_AGE_FEATURE
            ].median(),

        "MeanAge":
            query_age[
                REFERENCE_AGE_FEATURE
            ].mean(),

        "MaxAge":
            query_age[
                REFERENCE_AGE_FEATURE
            ].max(),
    }


reference_age_summary_df = pd.DataFrame(
    [
        query_age_summary(
            fresh_train160_model,
            "Fresh training",
        ),

        query_age_summary(
            longgap_train160_model,
            "Long-gap training",
        ),

        query_age_summary(
            outer_val160_longgap_ranked,
            "Outer validation",
        ),
    ]
)


display(
    reference_age_summary_df
)


print(
    "Fresh missing ages:",
    fresh_train160_model[
        REFERENCE_AGE_FEATURE
    ].isna().sum()
)

print(
    "Long-gap missing ages:",
    longgap_train160_model[
        REFERENCE_AGE_FEATURE
    ].isna().sum()
)

print(
    "Outer validation missing ages:",
    outer_val160_longgap_ranked[
        REFERENCE_AGE_FEATURE
    ].isna().sum()
)

,Dataset,Queries,MinAge,MedianAge,MeanAge,MaxAge
0,Fresh training,1777,8,46.0,43.371412,72
1,Long-gap training,1600,1,70.0,73.933750,160
2,Outer validation,2000,1,78.0,79.112000,158


Fresh missing ages: 0
Long-gap missing ages: 0
Outer validation missing ages: 0


In [92]:
# ============================================================
# HYBRID HGB + REFERENCE AGE
#
# 53 features:
#   original 52
#   +
#   ReferenceAgeDays
#
# Everything else remains unchanged.
# ============================================================

AGE_HYBRID_FEATURES = (
    frozen_feature_order
    +
    [REFERENCE_AGE_FEATURE]
)


assert (
    len(AGE_HYBRID_FEATURES)
    ==
    53
)


# ============================================================
# 1. Build model-ready Fresh training rows
# ============================================================

fresh_age_train = (
    fresh_train160_model[
        [
            "QueryId",
            "CandidateClassId",
            "TrueClassId",
            "IsCorrectCandidate",
        ]
        +
        AGE_HYBRID_FEATURES
    ]
    .copy()
)


# ============================================================
# 2. Build model-ready Long-gap training rows
# ============================================================

longgap_age_train = (
    longgap_train160_model[
        [
            "QueryId",
            "CandidateClassId",
            "TrueClassId",
            "IsCorrectCandidate",
        ]
        +
        AGE_HYBRID_FEATURES
    ]
    .copy()
)


# ============================================================
# 3. Combine
# ============================================================

hybrid_age_train = (
    pd.concat(
        [
            fresh_age_train,
            longgap_age_train,
        ],
        ignore_index=True,
    )
)


# ============================================================
# 4. Diagnostics
# ============================================================

print(
    "Training queries:",
    hybrid_age_train[
        "QueryId"
    ].nunique()
)

print(
    "Training candidate rows:",
    f"{len(hybrid_age_train):,}"
)

print(
    "Positive rows:",
    int(
        hybrid_age_train[
            "IsCorrectCandidate"
        ].sum()
    )
)

print(
    "Features:",
    len(
        AGE_HYBRID_FEATURES
    )
)

print(
    "ReferenceAge range:",
    hybrid_age_train[
        REFERENCE_AGE_FEATURE
    ].min(),
    "→",
    hybrid_age_train[
        REFERENCE_AGE_FEATURE
    ].max()
)

Training queries: 3377
Training candidate rows: 539,751
Positive rows: 3377
Features: 53
ReferenceAge range: 1 → 160


In [94]:
# ============================================================
# 5. TRAIN 53-FEATURE HYBRID HGB
# ============================================================

from sklearn.ensemble import HistGradientBoostingClassifier


X_hybrid_age_train = (
    hybrid_age_train[
        AGE_HYBRID_FEATURES
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .astype(np.float32)
)


y_hybrid_age_train = (
    hybrid_age_train[
        "IsCorrectCandidate"
    ]
    .astype(np.int8)
)


hybrid_age_hgb160 = (
    HistGradientBoostingClassifier(

        learning_rate=0.05,

        max_iter=400,

        max_leaf_nodes=7,

        min_samples_leaf=10,

        l2_regularization=1.0,

        early_stopping=True,

        validation_fraction=0.10,

        n_iter_no_change=20,

        random_state=RANDOM_STATE,
    )
)


hybrid_age_hgb160.fit(
    X_hybrid_age_train,
    y_hybrid_age_train,
)


print(
    "53-feature Hybrid HGB trained."
)

print(
    "Iterations used:",
    hybrid_age_hgb160.n_iter_
)

53-feature Hybrid HGB trained.
Iterations used: 400


In [95]:
# ============================================================
# 6. OUTER VALIDATION — 53 FEATURE MODEL
# ============================================================

X_outer_age = (
    outer_val160_longgap_ranked[
        AGE_HYBRID_FEATURES
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .astype(np.float32)
)


# ============================================================
# 7. Predict candidate probabilities
# ============================================================

outer_age_prob = (
    hybrid_age_hgb160.predict_proba(
        X_outer_age
    )[:, 1]
)


outer_age_scored = (
    outer_val160_longgap_ranked[
        [
            "QueryId",
            "CandidateClassId",
            "TrueClassId",
            "IsCorrectCandidate",
        ]
    ]
    .copy()
)


outer_age_scored[
    "Probability"
] = outer_age_prob


# ============================================================
# 8. Select best candidate per query
# ============================================================

outer_age_best = (
    outer_age_scored
    .sort_values(
        [
            "QueryId",
            "Probability",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .groupby(
        "QueryId",
        as_index=False,
    )
    .first()
)


outer_age_best[
    "PredictionCorrect"
] = (
    outer_age_best[
        "CandidateClassId"
    ]
    ==
    outer_age_best[
        "TrueClassId"
    ]
)


# ============================================================
# 9. Query-level evaluation
# ============================================================

outer_age_eval = (
    outer_val_eval
    .merge(
        outer_age_best[
            [
                "QueryId",
                "PredictionCorrect",
            ]
        ],
        on="QueryId",
        how="left",
    )
)


outer_age_eval[
    "PredictionCorrect"
] = (
    outer_age_eval[
        "PredictionCorrect"
    ]
    .fillna(False)
    .astype(bool)
)


# ============================================================
# 10. Metrics
# ============================================================

age_overall = (
    outer_age_eval[
        "PredictionCorrect"
    ].mean()
)


age_seen = (
    outer_age_eval.loc[
        outer_age_eval[
            "AnimalSeenInTrain"
        ],
        "PredictionCorrect",
    ].mean()
)


age_conditional = (
    outer_age_eval.loc[
        outer_age_eval[
            "CorrectCandidatePresent"
        ],
        "PredictionCorrect",
    ].mean()
)


age_correct = int(
    outer_age_eval[
        "PredictionCorrect"
    ].sum()
)


print("\n" + "=" * 72)

print(
    "OUTER VALIDATION — HYBRID + REFERENCE AGE"
)

print("=" * 72)

print(
    "Top-160 candidate coverage:",
    f"{outer_age_eval['CorrectCandidatePresent'].mean():.2%}"
)

print(
    "Overall accuracy:",
    f"{age_overall:.2%}"
)

print(
    "Seen-AnimalId accuracy:",
    f"{age_seen:.2%}"
)

print(
    "Conditional ranking accuracy:",
    f"{age_conditional:.2%}"
)

print(
    "Correct predictions:",
    age_correct,
    "/",
    len(outer_age_eval)
)


print("\nComparison:")

print(
    "Fresh champion:       38.50%"
)

print(
    "Long-gap HGB:         41.35%"
)

print(
    "Hybrid 52-feature:    41.65%"
)

print(
    "Hybrid + RefAge (53):",
    f"{age_overall:.2%}"
)


OUTER VALIDATION — HYBRID + REFERENCE AGE
Top-160 candidate coverage: 82.05%
Overall accuracy: 41.30%
Seen-AnimalId accuracy: 44.79%
Conditional ranking accuracy: 50.34%
Correct predictions: 826 / 2000

Comparison:
Fresh champion:       38.50%
Long-gap HGB:         41.35%
Hybrid 52-feature:    41.65%
Hybrid + RefAge (53): 41.30%


In [96]:
# ============================================================
# MODEL COMPLEMENTARITY DIAGNOSTIC
#
# Compare:
#   1. Frozen Fresh Champion
#   2. Long-gap HGB
#
# Same 2,000 outer-validation queries
# ============================================================


# ============================================================
# 1. Get prediction correctness from both models
# ============================================================

fresh_compare = (
    outer_val_final_eval[
        [
            "QueryId",
            "EventDate",
            "CorrectCandidatePresent",
            "PredictionCorrect",
        ]
    ]
    .copy()
    .rename(
        columns={
            "PredictionCorrect":
                "FreshCorrect"
        }
    )
)


longgap_compare = (
    longgap_outer_eval[
        [
            "QueryId",
            "PredictionCorrect",
        ]
    ]
    .copy()
    .rename(
        columns={
            "PredictionCorrect":
                "LongGapCorrect"
        }
    )
)


# ============================================================
# 2. Merge
# ============================================================

model_complementarity = (
    fresh_compare
    .merge(
        longgap_compare,
        on="QueryId",
        how="inner",
        validate="one_to_one",
    )
)


assert len(model_complementarity) == 2000


# ============================================================
# 3. Reference age
# ============================================================

model_complementarity[
    "ReferenceAgeDays"
] = (
    pd.to_datetime(
        model_complementarity[
            "EventDate"
        ]
    )
    -
    OUTER_VAL_CUTOFF
).dt.days


# ============================================================
# 4. Four possible outcomes
# ============================================================

both_correct = (
    model_complementarity[
        "FreshCorrect"
    ]
    &
    model_complementarity[
        "LongGapCorrect"
    ]
)


fresh_only = (
    model_complementarity[
        "FreshCorrect"
    ]
    &
    ~model_complementarity[
        "LongGapCorrect"
    ]
)


longgap_only = (
    ~model_complementarity[
        "FreshCorrect"
    ]
    &
    model_complementarity[
        "LongGapCorrect"
    ]
)


both_wrong = (
    ~model_complementarity[
        "FreshCorrect"
    ]
    &
    ~model_complementarity[
        "LongGapCorrect"
    ]
)


# ============================================================
# 5. Overall complementarity
# ============================================================

both_correct_n = int(
    both_correct.sum()
)

fresh_only_n = int(
    fresh_only.sum()
)

longgap_only_n = int(
    longgap_only.sum()
)

both_wrong_n = int(
    both_wrong.sum()
)


oracle_correct = (
    both_correct_n
    +
    fresh_only_n
    +
    longgap_only_n
)


oracle_accuracy = (
    oracle_correct
    /
    len(model_complementarity)
)


print("=" * 72)

print(
    "Fresh correct:",
    int(
        model_complementarity[
            "FreshCorrect"
        ].sum()
    )
)

print(
    "Long-gap correct:",
    int(
        model_complementarity[
            "LongGapCorrect"
        ].sum()
    )
)

print()

print(
    "Both correct:",
    both_correct_n
)

print(
    "Fresh only correct:",
    fresh_only_n
)

print(
    "Long-gap only correct:",
    longgap_only_n
)

print(
    "Both wrong:",
    both_wrong_n
)

print()

print(
    "Oracle correct:",
    oracle_correct,
    "/",
    len(model_complementarity)
)

print(
    "Oracle accuracy:",
    f"{oracle_accuracy:.2%}"
)

Fresh correct: 770
Long-gap correct: 827

Both correct: 640
Fresh only correct: 130
Long-gap only correct: 187
Both wrong: 1043

Oracle correct: 957 / 2000
Oracle accuracy: 47.85%


In [97]:
# ============================================================
# 6. Complementarity by reference-age window
# ============================================================

model_complementarity[
    "FreshnessWindow"
] = pd.cut(
    model_complementarity[
        "ReferenceAgeDays"
    ],
    bins=[
        0,
        30,
        60,
        90,
        120,
        160,
    ],
    labels=[
        "1-30 days",
        "31-60 days",
        "61-90 days",
        "91-120 days",
        "121-160 days",
    ],
    include_lowest=True,
)


complementarity_by_age = []


for window, subset in (
    model_complementarity
    .groupby(
        "FreshnessWindow",
        observed=True,
    )
):

    both = (
        subset[
            "FreshCorrect"
        ]
        &
        subset[
            "LongGapCorrect"
        ]
    )

    fresh_only_here = (
        subset[
            "FreshCorrect"
        ]
        &
        ~subset[
            "LongGapCorrect"
        ]
    )

    longgap_only_here = (
        ~subset[
            "FreshCorrect"
        ]
        &
        subset[
            "LongGapCorrect"
        ]
    )

    oracle_here = (
        subset[
            "FreshCorrect"
        ]
        |
        subset[
            "LongGapCorrect"
        ]
    )


    complementarity_by_age.append({

        "FreshnessWindow":
            str(window),

        "Queries":
            len(subset),

        "BothCorrect":
            int(
                both.sum()
            ),

        "FreshOnly":
            int(
                fresh_only_here.sum()
            ),

        "LongGapOnly":
            int(
                longgap_only_here.sum()
            ),

        "FreshAccuracy":
            subset[
                "FreshCorrect"
            ].mean() * 100,

        "LongGapAccuracy":
            subset[
                "LongGapCorrect"
            ].mean() * 100,

        "OracleAccuracy":
            oracle_here.mean() * 100,
    })


complementarity_by_age_df = (
    pd.DataFrame(
        complementarity_by_age
    )
)


display(
    complementarity_by_age_df
)

,FreshnessWindow,Queries,BothCorrect,FreshOnly,LongGapOnly,FreshAccuracy,LongGapAccuracy,OracleAccuracy
0,1-30 days,394,261,22,14,71.827411,69.796954,75.380711
1,31-60 days,375,169,21,19,50.666667,50.133333,55.733333
2,61-90 days,384,93,29,40,31.770833,34.635417,42.187500
3,91-120 days,361,52,35,53,24.099723,29.085873,38.781163
4,121-160 days,486,65,23,61,18.106996,25.925926,30.658436


## What to do after this notebook runs

First compare the XGBRanker metrics with the saved HGB reference:

- HGB overall accuracy: **28.40%**
- HGB conditional ranking accuracy: **42.14%**

If XGBRanker is clearly better, keep the same candidate-generation system and tune the ranking model next. If it is similar or worse, the next bottleneck is more likely candidate retrieval / candidate features rather than the final model alone.

The outer validation and test sets remain untouched in this notebook. They should only be used after the model-development choice is settled.